# Global Cotton Weather Dashboard (v6)

Built for **Amau cotton operations**. Seven tabs covering the global cotton supply chain:

- **Tab 1 — NOAA Statewide (Texas)** — Texas overview, NOAA NWS data
- **Tab 2 — Cotton Belt Texas** — 74 counties, USDA + PCG
- **Tab 3 — Brazil** — 55 municipalities (MT, BA, MS, GO, MG, MA, PI)
- **Tab 4 — China** — 51 sites (Xinjiang prefectures + Yellow River + Yangtze residual). **NEW**: 35-day long-range temperature forecast (16d deterministic + 17-35d ensemble mean) on top_tier ⭐ sites
- **Tab 5 — India** — 65 districts (Gujarat, Maharashtra, Telangana + 8 other states) with **monsoon alert** during June-September
- **Tab 6 — Australia** — 33 sites (Murray-Darling Basin, NSW + QLD)
- **Tab 7 — Turkey** — 53 sites (GAP, Cukurova, Aegean regions)

All countries use Open-Meteo for hourly weather data: 7-day past + 10-day forecast, ET0 FAO Penman-Monteith, root-zone soil moisture, hourly precipitation probability.

## Data sources

- **NOAA NWS** (`api.weather.gov`) — Texas forecasts, observations, alerts
- **Open-Meteo** (`api.open-meteo.com`) — global coverage, historical + forecast

No API keys required. All data is free and public.


## Dependencies

Only `requests` is needed:

```python
%pip install requests
```

Then run all cells. The notebook writes `texas_weather_dashboard.html` (and a
matching `.json` snapshot) to the current directory. Open the HTML in any
browser — everything is self-contained.


In [23]:
from __future__ import annotations
import html
import json
import math
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path

import requests

# Configuration
NOAA_BASE        = "https://api.weather.gov"
OPEN_METEO_BASE  = "https://api.open-meteo.com/v1/forecast"
OPEN_METEO_ARCHIVE = "https://archive-api.open-meteo.com/v1/archive"
# Multiple candidate endpoints for the ensemble mean — we try them in order
# because Open-Meteo docs don't fully commit to a single URL pattern
OPEN_METEO_ENSEMBLE_MEAN_URLS = [
    "https://ensemble-api.open-meteo.com/v1/ensemble_mean",
    "https://api.open-meteo.com/v1/ensemble_mean",
]
# Seasonal Forecast API (ECMWF EC46, 46 days, 51 members) — fallback for
# extended-range temperature outlook when Ensemble Mean returns nothing
OPEN_METEO_SEASONAL_URL = "https://seasonal-api.open-meteo.com/v1/seasonal"

LONG_RANGE_DETERMINISTIC_DAYS = 16   # Extended GFS/ECMWF deterministic horizon
LONG_RANGE_ENSEMBLE_DAYS      = 35   # Ensemble horizon (35 GFS ENS / 46 EC46)  # Historical fallback

USER_AGENT       = "GlobalCottonWeather/5.0 (Amau)"
REQUEST_TIMEOUT  = 25
MAX_RETRIES      = 5     # increased from 3 for SSL/connection resilience
RETRY_BACKOFF    = 1.5  # legacy, kept for compatibility (unused by new _fetch_json)

OUTPUT_FILE      = Path("global_cotton_weather.html")

print("Configuration loaded:")
print(f"  NOAA       -> {NOAA_BASE}")
print(f"  Open-Meteo -> {OPEN_METEO_BASE}")
print(f"  Output     -> {OUTPUT_FILE.resolve()}")


Configuration loaded:
  NOAA       -> https://api.weather.gov
  Open-Meteo -> https://api.open-meteo.com/v1/forecast
  Output     -> C:\Users\AMAR8\Work Folders\Downloads\global_cotton_weather.html


In [24]:
"""Canonical Texas cotton county list with county-seat coordinates.

Sources:
- PCG (Plains Cotton Growers) 42-county High Plains list
- USDA top-12 cotton producing counties for Texas
- Coastal Bend / Lower Rio Grande Valley extension counties
- Texas General Land Office (county seat coordinates)

The county seat is used as the geo-anchor for weather lookups since it's
typically the most populated point in the county and has the most reliable
weather station coverage.

Region codes:
- HP   = High Plains (Lubbock-Amarillo, primary cotton production)
- RP   = Rolling Plains (Abilene-Wichita Falls, secondary cotton)
- CB   = Coastal Bend (Corpus Christi area)
- LRGV = Lower Rio Grande Valley (south of San Antonio)
- FW   = Far West (Trans-Pecos)
- BL   = Blacklands (Dallas-Waco)
- EP   = Edwards Plateau (San Angelo)

Top12 = top 12 USDA-ranked cotton-producing counties (boolean).
"""

COTTON_COUNTIES = [
    # ========== TOP 12 USDA ==========
    {"county": "Lubbock",      "seat": "Lubbock",        "lat": 33.5779, "lon": -101.8552, "region": "HP",   "top12": True},
    {"county": "Hale",         "seat": "Plainview",      "lat": 34.1848, "lon": -101.7068, "region": "HP",   "top12": True},
    {"county": "Hockley",      "seat": "Levelland",      "lat": 33.5873, "lon": -102.3777, "region": "HP",   "top12": True},
    {"county": "Floyd",        "seat": "Floydada",       "lat": 33.9826, "lon": -101.3371, "region": "HP",   "top12": True},
    {"county": "Lynn",         "seat": "Tahoka",         "lat": 33.1665, "lon": -101.7977, "region": "HP",   "top12": True},
    {"county": "Crosby",       "seat": "Crosbyton",      "lat": 33.6601, "lon": -101.2382, "region": "HP",   "top12": True},
    {"county": "Gaines",       "seat": "Seminole",       "lat": 32.7196, "lon": -102.6510, "region": "HP",   "top12": True},
    {"county": "Terry",        "seat": "Brownfield",     "lat": 33.1812, "lon": -102.2744, "region": "HP",   "top12": True},
    {"county": "Dawson",       "seat": "Lamesa",         "lat": 32.7376, "lon": -101.9510, "region": "HP",   "top12": True},
    {"county": "Lamb",         "seat": "Littlefield",    "lat": 33.9173, "lon": -102.3266, "region": "HP",   "top12": True},
    {"county": "San Patricio", "seat": "Sinton",         "lat": 28.0339, "lon": -97.5158,  "region": "CB",   "top12": True},
    {"county": "Nueces",       "seat": "Corpus Christi", "lat": 27.8006, "lon": -97.3964,  "region": "CB",   "top12": True},

    # ========== Remainder of the 42 PCG High Plains counties ==========
    {"county": "Andrews",      "seat": "Andrews",        "lat": 32.3187, "lon": -102.5457, "region": "HP",   "top12": False},
    {"county": "Armstrong",    "seat": "Claude",         "lat": 35.1117, "lon": -101.3593, "region": "HP",   "top12": False},
    {"county": "Bailey",       "seat": "Muleshoe",       "lat": 34.2287, "lon": -102.7235, "region": "HP",   "top12": False},
    {"county": "Borden",       "seat": "Gail",           "lat": 32.7593, "lon": -101.4513, "region": "HP",   "top12": False},
    {"county": "Briscoe",      "seat": "Silverton",      "lat": 34.4734, "lon": -101.3060, "region": "HP",   "top12": False},
    {"county": "Carson",       "seat": "Panhandle",      "lat": 35.3506, "lon": -101.3812, "region": "HP",   "top12": False},
    {"county": "Castro",       "seat": "Dimmitt",        "lat": 34.5503, "lon": -102.3127, "region": "HP",   "top12": False},
    {"county": "Cochran",      "seat": "Morton",         "lat": 33.7242, "lon": -102.7560, "region": "HP",   "top12": False},
    {"county": "Dallam",       "seat": "Dalhart",        "lat": 36.0593, "lon": -102.5132, "region": "HP",   "top12": False},
    {"county": "Deaf Smith",   "seat": "Hereford",       "lat": 34.8151, "lon": -102.3993, "region": "HP",   "top12": False},
    {"county": "Dickens",      "seat": "Dickens",        "lat": 33.6207, "lon": -100.8351, "region": "HP",   "top12": False},
    {"county": "Garza",        "seat": "Post",           "lat": 33.1923, "lon": -101.3826, "region": "HP",   "top12": False},
    {"county": "Gray",         "seat": "Pampa",          "lat": 35.5361, "lon": -100.9598, "region": "HP",   "top12": False},
    {"county": "Hansford",     "seat": "Spearman",       "lat": 36.1986, "lon": -101.1929, "region": "HP",   "top12": False},
    {"county": "Hartley",      "seat": "Channing",       "lat": 35.6845, "lon": -102.3327, "region": "HP",   "top12": False},
    {"county": "Hemphill",     "seat": "Canadian",       "lat": 35.9114, "lon": -100.3833, "region": "HP",   "top12": False},
    {"county": "Howard",       "seat": "Big Spring",     "lat": 32.2504, "lon": -101.4787, "region": "HP",   "top12": False},
    {"county": "Hutchinson",   "seat": "Stinnett",       "lat": 35.8254, "lon": -101.4393, "region": "HP",   "top12": False},
    {"county": "Lipscomb",     "seat": "Lipscomb",       "lat": 36.2347, "lon": -100.2716, "region": "HP",   "top12": False},
    {"county": "Martin",       "seat": "Stanton",        "lat": 32.1296, "lon": -101.7882, "region": "HP",   "top12": False},
    {"county": "Midland",      "seat": "Midland",        "lat": 31.9973, "lon": -102.0779, "region": "HP",   "top12": False},
    {"county": "Moore",        "seat": "Dumas",          "lat": 35.8656, "lon": -101.9732, "region": "HP",   "top12": False},
    {"county": "Motley",       "seat": "Matador",        "lat": 34.0140, "lon": -100.8237, "region": "HP",   "top12": False},
    {"county": "Ochiltree",    "seat": "Perryton",       "lat": 36.4031, "lon": -100.8015, "region": "HP",   "top12": False},
    {"county": "Oldham",       "seat": "Vega",           "lat": 35.2447, "lon": -102.4279, "region": "HP",   "top12": False},
    {"county": "Parmer",       "seat": "Farwell",        "lat": 34.3848, "lon": -103.0388, "region": "HP",   "top12": False},
    {"county": "Potter",       "seat": "Amarillo",       "lat": 35.2220, "lon": -101.8313, "region": "HP",   "top12": False},
    {"county": "Randall",      "seat": "Canyon",         "lat": 34.9803, "lon": -101.9188, "region": "HP",   "top12": False},
    {"county": "Roberts",      "seat": "Miami",          "lat": 35.6906, "lon": -100.6388, "region": "HP",   "top12": False},
    {"county": "Sherman",      "seat": "Stratford",      "lat": 36.3392, "lon": -102.0738, "region": "HP",   "top12": False},
    {"county": "Swisher",      "seat": "Tulia",          "lat": 34.5364, "lon": -101.7563, "region": "HP",   "top12": False},
    {"county": "Yoakum",       "seat": "Plains",         "lat": 33.1882, "lon": -102.8294, "region": "HP",   "top12": False},

    # ========== Rolling Plains (secondary cotton, Abilene area) ==========
    {"county": "Scurry",       "seat": "Snyder",         "lat": 32.7176, "lon": -100.9176, "region": "RP",   "top12": False},
    {"county": "Nolan",        "seat": "Sweetwater",     "lat": 32.4709, "lon": -100.4060, "region": "RP",   "top12": False},
    {"county": "Fisher",       "seat": "Roby",           "lat": 32.7484, "lon": -100.3782, "region": "RP",   "top12": False},
    {"county": "Jones",        "seat": "Anson",          "lat": 32.7548, "lon": -99.8973,  "region": "RP",   "top12": False},
    {"county": "Taylor",       "seat": "Abilene",        "lat": 32.4487, "lon": -99.7331,  "region": "RP",   "top12": False},
    {"county": "Mitchell",     "seat": "Colorado City",  "lat": 32.3960, "lon": -100.8645, "region": "RP",   "top12": False},
    {"county": "Childress",    "seat": "Childress",      "lat": 34.4265, "lon": -100.2040, "region": "RP",   "top12": False},
    {"county": "Hall",         "seat": "Memphis",        "lat": 34.7237, "lon": -100.5360, "region": "RP",   "top12": False},
    {"county": "Hardeman",     "seat": "Quanah",         "lat": 34.2960, "lon": -99.7437,  "region": "RP",   "top12": False},
    {"county": "Cottle",       "seat": "Paducah",        "lat": 34.0123, "lon": -100.3018, "region": "RP",   "top12": False},
    {"county": "Foard",        "seat": "Crowell",        "lat": 33.9826, "lon": -99.7234,  "region": "RP",   "top12": False},
    {"county": "Knox",         "seat": "Benjamin",       "lat": 33.5859, "lon": -99.7906,  "region": "RP",   "top12": False},
    {"county": "Haskell",      "seat": "Haskell",        "lat": 33.1576, "lon": -99.7334,  "region": "RP",   "top12": False},

    # ========== Coastal Bend / Lower Rio Grande Valley ==========
    {"county": "Wharton",      "seat": "Wharton",        "lat": 29.3119, "lon": -96.1027,  "region": "CB",   "top12": False},
    {"county": "Matagorda",    "seat": "Bay City",       "lat": 28.9828, "lon": -95.9694,  "region": "CB",   "top12": False},
    {"county": "Jackson",      "seat": "Edna",           "lat": 28.9783, "lon": -96.6464,  "region": "CB",   "top12": False},
    {"county": "Calhoun",      "seat": "Port Lavaca",    "lat": 28.6149, "lon": -96.6261,  "region": "CB",   "top12": False},
    {"county": "Refugio",      "seat": "Refugio",        "lat": 28.3047, "lon": -97.2778,  "region": "CB",   "top12": False},
    {"county": "Hidalgo",      "seat": "Edinburg",       "lat": 26.3017, "lon": -98.1633,  "region": "LRGV", "top12": False},
    {"county": "Cameron",      "seat": "Brownsville",    "lat": 25.9018, "lon": -97.4975,  "region": "LRGV", "top12": False},
    {"county": "Willacy",      "seat": "Raymondville",   "lat": 26.4795, "lon": -97.7780,  "region": "LRGV", "top12": False},
    {"county": "Kleberg",      "seat": "Kingsville",     "lat": 27.5159, "lon": -97.8561,  "region": "LRGV", "top12": False},

    # ========== Far West (Trans-Pecos, Pima cotton) ==========
    {"county": "El Paso",      "seat": "El Paso",        "lat": 31.7619, "lon": -106.4850, "region": "FW",   "top12": False},
    {"county": "Hudspeth",     "seat": "Sierra Blanca",  "lat": 31.1746, "lon": -105.3522, "region": "FW",   "top12": False},
    {"county": "Pecos",        "seat": "Fort Stockton",  "lat": 30.8838, "lon": -102.8779, "region": "FW",   "top12": False},
    {"county": "Reeves",       "seat": "Pecos",          "lat": 31.4229, "lon": -103.4932, "region": "FW",   "top12": False},
    {"county": "Ward",         "seat": "Monahans",       "lat": 31.5946, "lon": -102.8929, "region": "FW",   "top12": False},

    # ========== Edwards Plateau (San Angelo area) ==========
    {"county": "Tom Green",    "seat": "San Angelo",     "lat": 31.4638, "lon": -100.4370, "region": "EP",   "top12": False},
    {"county": "Concho",       "seat": "Paint Rock",     "lat": 31.5093, "lon": -99.9242,  "region": "EP",   "top12": False},
    {"county": "Runnels",      "seat": "Ballinger",      "lat": 31.7421, "lon": -99.9479,  "region": "EP",   "top12": False},
]

# Sanity summary

# NOAA statewide cities (carried over from v3 NOAA tab)
WEST_TEXAS_CITIES = [
    {"name": "Midland",     "lat": 31.9973, "lon": -102.0779, "region": "West"},
    {"name": "Odessa",      "lat": 31.8457, "lon": -102.3676, "region": "West"},
    {"name": "Lubbock",     "lat": 33.5779, "lon": -101.8552, "region": "West"},
    {"name": "Amarillo",    "lat": 35.2220, "lon": -101.8313, "region": "West"},
    {"name": "El Paso",     "lat": 31.7619, "lon": -106.4850, "region": "West"},
    {"name": "Abilene",     "lat": 32.4487, "lon": -99.7331,  "region": "West"},
    {"name": "San Angelo",  "lat": 31.4638, "lon": -100.4370, "region": "West"},
    {"name": "Pecos",       "lat": 31.4229, "lon": -103.4932, "region": "West"},
]
TEXAS_CITIES = WEST_TEXAS_CITIES + [
    {"name": "Dallas",         "lat": 32.7767, "lon": -96.7970,  "region": "North"},
    {"name": "Fort Worth",     "lat": 32.7555, "lon": -97.3308,  "region": "North"},
    {"name": "Houston",        "lat": 29.7604, "lon": -95.3698,  "region": "Gulf"},
    {"name": "Austin",         "lat": 30.2672, "lon": -97.7431,  "region": "Central"},
    {"name": "San Antonio",    "lat": 29.4241, "lon": -98.4936,  "region": "Central"},
    {"name": "Corpus Christi", "lat": 27.8006, "lon": -97.3964,  "region": "Gulf"},
    {"name": "Brownsville",    "lat": 25.9018, "lon": -97.4975,  "region": "Gulf"},
    {"name": "Waco",           "lat": 31.5493, "lon": -97.1467,  "region": "Central"},
    {"name": "Tyler",          "lat": 32.3513, "lon": -95.3011,  "region": "East"},
    {"name": "Beaumont",       "lat": 30.0802, "lon": -94.1266,  "region": "East"},
    {"name": "Laredo",         "lat": 27.5306, "lon": -99.4803,  "region": "South"},
    {"name": "Del Rio",        "lat": 29.3627, "lon": -100.8968, "region": "South"},
]
US_CITIES = [
    {"name": "New York",    "lat": 40.7128, "lon": -74.0060},
    {"name": "Los Angeles", "lat": 34.0522, "lon": -118.2437},
    {"name": "Chicago",     "lat": 41.8781, "lon": -87.6298},
    {"name": "Phoenix",     "lat": 33.4484, "lon": -112.0740},
    {"name": "Denver",      "lat": 39.7392, "lon": -104.9903},
    {"name": "Seattle",     "lat": 47.6062, "lon": -122.3321},
    {"name": "Miami",       "lat": 25.7617, "lon": -80.1918},
    {"name": "Atlanta",     "lat": 33.7490, "lon": -84.3880},
]

from collections import Counter
print(f"Cotton counties: {len(COTTON_COUNTIES)} ({sum(1 for c in COTTON_COUNTIES if c['top12'])} top-12 USDA)")
print(f"  Regions: {dict(Counter(c['region'] for c in COTTON_COUNTIES))}")
print(f"NOAA cities: {len(TEXAS_CITIES)} TX, {len(US_CITIES)} US")


Cotton counties: 74 (12 top-12 USDA)
  Regions: {'HP': 42, 'CB': 7, 'RP': 13, 'LRGV': 4, 'FW': 5, 'EP': 3}
NOAA cities: 20 TX, 8 US


In [25]:
"""Canonical Brazil cotton-producing municipality list with coordinates.

Sources:
- CONAB (Companhia Nacional de Abastecimento) - state-level production rankings
- ABRAPA / IMEA / AIBA / ABAPA - state cotton growers' associations
- IBGE municipality coordinates

State distribution (2024/25 season):
- MT (Mato Grosso): ~71% national production
- BA (Bahia, Oeste Baiano): ~20%
- MS (Mato Grosso do Sul): ~3%
- GO (Goiás): ~2%
- MG (Minas Gerais): ~2%
- MA (Maranhão), PI (Piauí), SP (São Paulo), RO (Rondônia): remainder

Region codes:
- MT_NORTE   = Mato Grosso, Norte (Sorriso/Sinop axis)
- MT_MEDIO   = Mato Grosso, Médio-Norte (Lucas/Tapurah)
- MT_OESTE   = Mato Grosso, Oeste/Parecis (Sapezal/Campo Novo)
- MT_SUDESTE = Mato Grosso, Sudeste (Primavera/Campo Verde, the older cotton belt)
- BA_OESTE   = Bahia, Oeste Baiano (cerrado, large-scale rainfed)
- MS         = Mato Grosso do Sul
- GO         = Goiás
- MG         = Minas Gerais
- MA         = Maranhão
- PI         = Piauí

top_tier = booleans for the very largest producers (analogous to top-12 USDA for Texas).
"""

COTTON_MUNICIPALITIES_BR = [
    # ============ Mato Grosso - Sudeste (historical cotton heart) ============
    {"county": "Campo Verde",          "seat": "Campo Verde",          "lat": -15.5447, "lon": -55.1722, "region": "MT_SUDESTE", "state": "MT", "top_tier": True},
    {"county": "Primavera do Leste",   "seat": "Primavera do Leste",   "lat": -15.5586, "lon": -54.2967, "region": "MT_SUDESTE", "state": "MT", "top_tier": True},
    {"county": "Pedra Preta",          "seat": "Pedra Preta",          "lat": -16.6219, "lon": -54.4711, "region": "MT_SUDESTE", "state": "MT", "top_tier": False},
    {"county": "Rondonópolis",         "seat": "Rondonópolis",         "lat": -16.4706, "lon": -54.6358, "region": "MT_SUDESTE", "state": "MT", "top_tier": False},
    {"county": "Itiquira",             "seat": "Itiquira",             "lat": -17.2092, "lon": -54.1453, "region": "MT_SUDESTE", "state": "MT", "top_tier": False},
    {"county": "Dom Aquino",           "seat": "Dom Aquino",           "lat": -15.8125, "lon": -54.9772, "region": "MT_SUDESTE", "state": "MT", "top_tier": False},
    {"county": "Jaciara",              "seat": "Jaciara",              "lat": -15.9658, "lon": -54.9683, "region": "MT_SUDESTE", "state": "MT", "top_tier": False},

    # ============ Mato Grosso - Oeste / Parecis (Sapezal axis - the biggest cotton zone) ============
    {"county": "Sapezal",              "seat": "Sapezal",              "lat": -13.5436, "lon": -58.8133, "region": "MT_OESTE",   "state": "MT", "top_tier": True},
    {"county": "Campo Novo do Parecis","seat": "Campo Novo do Parecis","lat": -13.6750, "lon": -57.8917, "region": "MT_OESTE",   "state": "MT", "top_tier": True},
    {"county": "Campos de Júlio",      "seat": "Campos de Júlio",      "lat": -13.7456, "lon": -59.2683, "region": "MT_OESTE",   "state": "MT", "top_tier": True},
    {"county": "Diamantino",           "seat": "Diamantino",           "lat": -14.4053, "lon": -56.4461, "region": "MT_OESTE",   "state": "MT", "top_tier": True},
    {"county": "Nova Mutum",           "seat": "Nova Mutum",           "lat": -13.8275, "lon": -56.0764, "region": "MT_OESTE",   "state": "MT", "top_tier": True},
    {"county": "Tangará da Serra",     "seat": "Tangará da Serra",     "lat": -14.6217, "lon": -57.4933, "region": "MT_OESTE",   "state": "MT", "top_tier": False},
    {"county": "Brasnorte",            "seat": "Brasnorte",            "lat": -12.1517, "lon": -57.9756, "region": "MT_OESTE",   "state": "MT", "top_tier": False},
    {"county": "Comodoro",             "seat": "Comodoro",             "lat": -13.6603, "lon": -59.7894, "region": "MT_OESTE",   "state": "MT", "top_tier": False},
    {"county": "Nova Marilândia",      "seat": "Nova Marilândia",      "lat": -14.3589, "lon": -56.9889, "region": "MT_OESTE",   "state": "MT", "top_tier": False},
    {"county": "Arenápolis",           "seat": "Arenápolis",           "lat": -14.4439, "lon": -56.8444, "region": "MT_OESTE",   "state": "MT", "top_tier": False},

    # ============ Mato Grosso - Médio-Norte (Lucas / Tapurah / Sorriso) ============
    {"county": "Lucas do Rio Verde",   "seat": "Lucas do Rio Verde",   "lat": -13.0533, "lon": -55.9181, "region": "MT_MEDIO",   "state": "MT", "top_tier": True},
    {"county": "Sorriso",              "seat": "Sorriso",              "lat": -12.5450, "lon": -55.7211, "region": "MT_MEDIO",   "state": "MT", "top_tier": True},
    {"county": "Tapurah",              "seat": "Tapurah",              "lat": -12.7322, "lon": -56.5161, "region": "MT_MEDIO",   "state": "MT", "top_tier": True},
    {"county": "Ipiranga do Norte",    "seat": "Ipiranga do Norte",    "lat": -12.8328, "lon": -56.1456, "region": "MT_MEDIO",   "state": "MT", "top_tier": False},
    {"county": "Itanhangá",            "seat": "Itanhangá",            "lat": -12.2486, "lon": -56.4861, "region": "MT_MEDIO",   "state": "MT", "top_tier": False},
    {"county": "Nova Ubiratã",         "seat": "Nova Ubiratã",         "lat": -12.9836, "lon": -55.2547, "region": "MT_MEDIO",   "state": "MT", "top_tier": False},

    # ============ Mato Grosso - Norte (Sinop area) ============
    {"county": "Sinop",                "seat": "Sinop",                "lat": -11.8639, "lon": -55.5025, "region": "MT_NORTE",   "state": "MT", "top_tier": False},
    {"county": "Vera",                 "seat": "Vera",                 "lat": -12.3047, "lon": -55.3147, "region": "MT_NORTE",   "state": "MT", "top_tier": False},
    {"county": "Cláudia",              "seat": "Cláudia",              "lat": -11.5028, "lon": -54.8839, "region": "MT_NORTE",   "state": "MT", "top_tier": False},
    {"county": "Querência",            "seat": "Querência",            "lat": -12.6075, "lon": -52.1819, "region": "MT_NORTE",   "state": "MT", "top_tier": False},

    # ============ Bahia - Oeste Baiano (cerrado cotton heartland) ============
    {"county": "São Desidério",        "seat": "São Desidério",        "lat": -12.3625, "lon": -44.9747, "region": "BA_OESTE",   "state": "BA", "top_tier": True},
    {"county": "Luís Eduardo Magalhães","seat":"Luís Eduardo Magalhães","lat":-12.0922,"lon": -45.7919, "region": "BA_OESTE",   "state": "BA", "top_tier": True},
    {"county": "Barreiras",            "seat": "Barreiras",            "lat": -12.1531, "lon": -44.9908, "region": "BA_OESTE",   "state": "BA", "top_tier": True},
    {"county": "Formosa do Rio Preto", "seat": "Formosa do Rio Preto", "lat": -11.0461, "lon": -45.1925, "region": "BA_OESTE",   "state": "BA", "top_tier": True},
    {"county": "Correntina",           "seat": "Correntina",           "lat": -13.3439, "lon": -44.6378, "region": "BA_OESTE",   "state": "BA", "top_tier": True},
    {"county": "Jaborandi",            "seat": "Jaborandi",            "lat": -13.6064, "lon": -44.4225, "region": "BA_OESTE",   "state": "BA", "top_tier": False},
    {"county": "Cocos",                "seat": "Cocos",                "lat": -14.1761, "lon": -44.5331, "region": "BA_OESTE",   "state": "BA", "top_tier": False},
    {"county": "Baianópolis",          "seat": "Baianópolis",          "lat": -12.3008, "lon": -44.5408, "region": "BA_OESTE",   "state": "BA", "top_tier": False},
    {"county": "Riachão das Neves",    "seat": "Riachão das Neves",    "lat": -11.7472, "lon": -44.9092, "region": "BA_OESTE",   "state": "BA", "top_tier": False},
    {"county": "Santa Rita de Cássia", "seat": "Santa Rita de Cássia", "lat": -11.0083, "lon": -44.5258, "region": "BA_OESTE",   "state": "BA", "top_tier": False},
    {"county": "Mansidão",             "seat": "Mansidão",             "lat": -10.7239, "lon": -44.0394, "region": "BA_OESTE",   "state": "BA", "top_tier": False},

    # ============ Mato Grosso do Sul ============
    {"county": "Chapadão do Sul",      "seat": "Chapadão do Sul",      "lat": -18.7889, "lon": -52.6261, "region": "MS",         "state": "MS", "top_tier": False},
    {"county": "Costa Rica",           "seat": "Costa Rica",           "lat": -18.5450, "lon": -53.1297, "region": "MS",         "state": "MS", "top_tier": False},
    {"county": "Maracaju",             "seat": "Maracaju",             "lat": -21.6147, "lon": -55.1683, "region": "MS",         "state": "MS", "top_tier": False},
    {"county": "São Gabriel do Oeste", "seat": "São Gabriel do Oeste", "lat": -19.3914, "lon": -54.5589, "region": "MS",         "state": "MS", "top_tier": False},

    # ============ Goiás ============
    {"county": "Chapadão do Céu",      "seat": "Chapadão do Céu",      "lat": -18.4131, "lon": -52.6336, "region": "GO",         "state": "GO", "top_tier": False},
    {"county": "Mineiros",             "seat": "Mineiros",             "lat": -17.5694, "lon": -52.5536, "region": "GO",         "state": "GO", "top_tier": False},
    {"county": "Jataí",                "seat": "Jataí",                "lat": -17.8814, "lon": -51.7178, "region": "GO",         "state": "GO", "top_tier": False},
    {"county": "Montividiu",           "seat": "Montividiu",           "lat": -17.4406, "lon": -51.1736, "region": "GO",         "state": "GO", "top_tier": False},
    {"county": "Rio Verde",            "seat": "Rio Verde",            "lat": -17.7942, "lon": -50.9264, "region": "GO",         "state": "GO", "top_tier": False},
    {"county": "Cristalina",           "seat": "Cristalina",           "lat": -16.7672, "lon": -47.6147, "region": "GO",         "state": "GO", "top_tier": False},

    # ============ Minas Gerais (Triângulo Mineiro) ============
    {"county": "Unaí",                 "seat": "Unaí",                 "lat": -16.3578, "lon": -46.9061, "region": "MG",         "state": "MG", "top_tier": False},
    {"county": "Paracatu",             "seat": "Paracatu",             "lat": -17.2225, "lon": -46.8744, "region": "MG",         "state": "MG", "top_tier": False},

    # ============ Maranhão (MATOPIBA frontier) ============
    {"county": "Balsas",               "seat": "Balsas",               "lat":  -7.5328, "lon": -46.0356, "region": "MA",         "state": "MA", "top_tier": False},
    {"county": "Tasso Fragoso",        "seat": "Tasso Fragoso",        "lat":  -8.4711, "lon": -45.7522, "region": "MA",         "state": "MA", "top_tier": False},

    # ============ Piauí (MATOPIBA frontier) ============
    {"county": "Uruçuí",               "seat": "Uruçuí",               "lat":  -7.2356, "lon": -44.5567, "region": "PI",         "state": "PI", "top_tier": False},
    {"county": "Baixa Grande do Ribeiro","seat":"Baixa Grande do Ribeiro","lat":-7.8675,"lon": -45.1414, "region": "PI",         "state": "PI", "top_tier": False},
    {"county": "Bom Jesus",            "seat": "Bom Jesus",            "lat":  -9.0731, "lon": -44.3589, "region": "PI",         "state": "PI", "top_tier": False},
]

from collections import Counter as _Counter_br
print(f"Brazil cotton municipalities: {len(COTTON_MUNICIPALITIES_BR)} "
      f"({sum(1 for c in COTTON_MUNICIPALITIES_BR if c['top_tier'])} top-tier)")
print(f"  States: {dict(_Counter_br(c['state'] for c in COTTON_MUNICIPALITIES_BR))}")


Brazil cotton municipalities: 55 (15 top-tier)
  States: {'MT': 27, 'BA': 11, 'MS': 4, 'GO': 6, 'MG': 2, 'MA': 2, 'PI': 3}


In [26]:
"""Cotton-producing sites for China, India, Australia, Turkey.

Sources:
- China: USDA FAS (Beijing post), Xinjiang Stats Bureau, Nature paper on Xinjiang cotton (Liu et al.)
  92% production from Xinjiang. Key prefectures: Aksu, Bayingolin, Kashgar, Bortala, Changji,
  Tarbagatay, Shihezi (Bingtuan), Hami. Plus residual Yellow River + Yangtze regions.
- India: Ministry of Agriculture, Cotton Outlook PJTSAU, USDA FAS (New Delhi post)
  Top 10 states: Gujarat (28%), Maharashtra (25%), Telangana (15%), Rajasthan, Karnataka, AP, MP, Haryana, Punjab, TN.
  District-level coverage of the main cotton belts.
- Australia: Cotton Australia, DAFF, USDA FAS Canberra
  Murray-Darling Basin = 91% production. NSW valleys (Gwydir, Namoi, Macquarie, Murrumbidgee, Macintyre,
  Border Rivers) + QLD (Darling Downs, St George, Dirranbandi, Central Highlands).
- Turkey: USDA FAS Ankara, Tarim Orman, IntechOpen Turkish cotton chapter
  3 regions: GAP/Southeast Anatolia (60%), Cukurova/Mediterranean (~20%), Aegean (~20%).
  Top 6 provinces = 86%: Sanliurfa (42%), Diyarbakir (14%), Aydin (12%), Hatay (9%), Izmir (5.5%), Adana (4%).

Coordinates are official province/district/region centroids verified against multiple sources.
"""

# ============================================================================
# CHINA — 92% Xinjiang. Organized by prefecture (north Xinjiang vs south).
# Top tier = the largest cotton-producing prefectures (Aksu, Bayingolin, Changji).
# ============================================================================
COTTON_SITES_CN = [
    # ---- North Xinjiang (NXJ) ----
    {"county":"Shihezi",                "seat":"Shihezi",       "lat":44.3050, "lon":86.0803, "region":"NXJ_BINGTUAN", "state":"XJ", "top_tier":True},
    {"county":"Manas",                  "seat":"Manas",         "lat":44.3050, "lon":86.2156, "region":"NXJ_CHANGJI",  "state":"XJ", "top_tier":True},
    {"county":"Hutubi",                 "seat":"Hutubi",        "lat":44.1697, "lon":86.8978, "region":"NXJ_CHANGJI",  "state":"XJ", "top_tier":True},
    {"county":"Changji",                "seat":"Changji",       "lat":44.0114, "lon":87.3033, "region":"NXJ_CHANGJI",  "state":"XJ", "top_tier":True},
    {"county":"Jimusar",                "seat":"Jimusar",       "lat":44.0042, "lon":89.1772, "region":"NXJ_CHANGJI",  "state":"XJ", "top_tier":False},
    {"county":"Fukang",                 "seat":"Fukang",        "lat":44.1583, "lon":87.9856, "region":"NXJ_CHANGJI",  "state":"XJ", "top_tier":False},
    {"county":"Shawan",                 "seat":"Shawan",        "lat":44.3306, "lon":85.6178, "region":"NXJ_TACHENG",  "state":"XJ", "top_tier":True},
    {"county":"Wusu",                   "seat":"Wusu",          "lat":44.4322, "lon":84.6797, "region":"NXJ_TACHENG",  "state":"XJ", "top_tier":True},
    {"county":"Karamay",                "seat":"Karamay",       "lat":45.5950, "lon":84.8694, "region":"NXJ_KARAMAY",  "state":"XJ", "top_tier":False},
    {"county":"Jinghe",                 "seat":"Jinghe",        "lat":44.6086, "lon":82.9000, "region":"NXJ_BORTALA",  "state":"XJ", "top_tier":True},
    {"county":"Bole",                   "seat":"Bole",          "lat":44.9028, "lon":82.0667, "region":"NXJ_BORTALA",  "state":"XJ", "top_tier":True},
    {"county":"Wenquan",                "seat":"Wenquan",       "lat":44.9756, "lon":81.0289, "region":"NXJ_BORTALA",  "state":"XJ", "top_tier":False},
    {"county":"Tacheng",                "seat":"Tacheng",       "lat":46.7497, "lon":82.9858, "region":"NXJ_TACHENG",  "state":"XJ", "top_tier":False},
    {"county":"Emin",                   "seat":"Emin",          "lat":46.5256, "lon":83.6286, "region":"NXJ_TACHENG",  "state":"XJ", "top_tier":False},
    {"county":"Hami",                   "seat":"Hami",          "lat":42.8167, "lon":93.5150, "region":"NXJ_HAMI",     "state":"XJ", "top_tier":False},
    {"county":"Yizhou",                 "seat":"Yizhou",        "lat":42.7000, "lon":93.6500, "region":"NXJ_HAMI",     "state":"XJ", "top_tier":False},

    # ---- South Xinjiang (SXJ) — Aksu prefecture (biggest cotton zone in China) ----
    {"county":"Aksu",                   "seat":"Aksu",          "lat":41.1675, "lon":80.2650, "region":"SXJ_AKSU",     "state":"XJ", "top_tier":True},
    {"county":"Awat",                   "seat":"Awat",          "lat":40.6336, "lon":80.3756, "region":"SXJ_AKSU",     "state":"XJ", "top_tier":True},
    {"county":"Kuqa",                   "seat":"Kuqa",          "lat":41.7172, "lon":82.9322, "region":"SXJ_AKSU",     "state":"XJ", "top_tier":True},
    {"county":"Shaya",                  "seat":"Shaya",         "lat":41.2247, "lon":82.7811, "region":"SXJ_AKSU",     "state":"XJ", "top_tier":True},
    {"county":"Xinhe",                  "seat":"Xinhe",         "lat":41.5500, "lon":82.6125, "region":"SXJ_AKSU",     "state":"XJ", "top_tier":True},
    {"county":"Baicheng",               "seat":"Baicheng",      "lat":41.7958, "lon":81.8753, "region":"SXJ_AKSU",     "state":"XJ", "top_tier":False},
    {"county":"Wensu",                  "seat":"Wensu",         "lat":41.2750, "lon":80.2400, "region":"SXJ_AKSU",     "state":"XJ", "top_tier":True},
    {"county":"Wushi",                  "seat":"Wushi",         "lat":41.2167, "lon":79.2333, "region":"SXJ_AKSU",     "state":"XJ", "top_tier":False},
    {"county":"Alar",                   "seat":"Alar",          "lat":40.5475, "lon":81.2789, "region":"SXJ_BINGTUAN", "state":"XJ", "top_tier":True},

    # ---- South Xinjiang — Bayingolin (Korla/Bayinguoleng) ----
    {"county":"Korla",                  "seat":"Korla",         "lat":41.7256, "lon":86.1750, "region":"SXJ_BAYINGOLIN","state":"XJ","top_tier":True},
    {"county":"Weili",                  "seat":"Weili",         "lat":41.6133, "lon":86.2533, "region":"SXJ_BAYINGOLIN","state":"XJ","top_tier":True},
    {"county":"Yanqi",                  "seat":"Yanqi",         "lat":42.0589, "lon":86.5739, "region":"SXJ_BAYINGOLIN","state":"XJ","top_tier":True},
    {"county":"Bohu",                   "seat":"Bohu",          "lat":41.9803, "lon":86.5717, "region":"SXJ_BAYINGOLIN","state":"XJ","top_tier":False},
    {"county":"Hejing",                 "seat":"Hejing",        "lat":42.3247, "lon":86.3947, "region":"SXJ_BAYINGOLIN","state":"XJ","top_tier":False},
    {"county":"Yuli",                   "seat":"Yuli",          "lat":41.3333, "lon":86.2531, "region":"SXJ_BAYINGOLIN","state":"XJ","top_tier":False},

    # ---- South Xinjiang — Kashgar prefecture ----
    {"county":"Kashgar",                "seat":"Kashgar",       "lat":39.4708, "lon":75.9892, "region":"SXJ_KASHGAR",  "state":"XJ", "top_tier":True},
    {"county":"Shache",                 "seat":"Shache",        "lat":38.4156, "lon":77.2406, "region":"SXJ_KASHGAR",  "state":"XJ", "top_tier":True},
    {"county":"Maigaiti",               "seat":"Maigaiti",      "lat":38.9097, "lon":77.6422, "region":"SXJ_KASHGAR",  "state":"XJ", "top_tier":True},
    {"county":"Bachu",                  "seat":"Bachu",         "lat":39.7972, "lon":78.5408, "region":"SXJ_KASHGAR",  "state":"XJ", "top_tier":True},
    {"county":"Jiashi",                 "seat":"Jiashi",        "lat":39.4936, "lon":76.7344, "region":"SXJ_KASHGAR",  "state":"XJ", "top_tier":True},
    {"county":"Yopurga",                "seat":"Yopurga",       "lat":39.2186, "lon":76.7461, "region":"SXJ_KASHGAR",  "state":"XJ", "top_tier":False},
    {"county":"Yengisar",               "seat":"Yengisar",      "lat":38.9333, "lon":76.1761, "region":"SXJ_KASHGAR",  "state":"XJ", "top_tier":False},
    {"county":"Tumxuk",                 "seat":"Tumxuk",        "lat":39.8669, "lon":79.0772, "region":"SXJ_BINGTUAN", "state":"XJ", "top_tier":True},

    # ---- South Xinjiang — Hotan + Kizilsu ----
    {"county":"Hotan",                  "seat":"Hotan",         "lat":37.1108, "lon":79.9217, "region":"SXJ_HOTAN",    "state":"XJ", "top_tier":False},
    {"county":"Lop",                    "seat":"Lop",           "lat":37.0717, "lon":80.1856, "region":"SXJ_HOTAN",    "state":"XJ", "top_tier":False},
    {"county":"Pishan",                 "seat":"Pishan",        "lat":37.6206, "lon":78.2839, "region":"SXJ_HOTAN",    "state":"XJ", "top_tier":False},
    {"county":"Moyu",                   "seat":"Moyu",          "lat":37.2725, "lon":79.6953, "region":"SXJ_HOTAN",    "state":"XJ", "top_tier":False},

    # ---- Residual: Yellow River region (Hebei, Shandong, Henan) ----
    {"county":"Hengshui",               "seat":"Hengshui",      "lat":37.7350, "lon":115.6708, "region":"YR_HEBEI",    "state":"HE", "top_tier":False},
    {"county":"Cangzhou",               "seat":"Cangzhou",      "lat":38.3037, "lon":116.8388, "region":"YR_HEBEI",    "state":"HE", "top_tier":False},
    {"county":"Dezhou",                 "seat":"Dezhou",        "lat":37.4347, "lon":116.3573, "region":"YR_SHANDONG", "state":"SD", "top_tier":False},
    {"county":"Liaocheng",              "seat":"Liaocheng",     "lat":36.4564, "lon":115.9853, "region":"YR_SHANDONG", "state":"SD", "top_tier":False},
    {"county":"Anyang",                 "seat":"Anyang",        "lat":36.0967, "lon":114.3925, "region":"YR_HENAN",    "state":"HA", "top_tier":False},

    # ---- Residual: Yangtze River region (Hubei, Hunan, Anhui, Jiangsu) ----
    {"county":"Jingzhou",               "seat":"Jingzhou",      "lat":30.3343, "lon":112.2386, "region":"YZ_HUBEI",    "state":"HB", "top_tier":False},
    {"county":"Yancheng",               "seat":"Yancheng",      "lat":33.3475, "lon":120.1614, "region":"YZ_JIANGSU",  "state":"JS", "top_tier":False},
    {"county":"Anqing",                 "seat":"Anqing",        "lat":30.5430, "lon":117.0633, "region":"YZ_ANHUI",    "state":"AH", "top_tier":False},
]


# ============================================================================
# INDIA — 10 cotton states, district-level. Kharif crop sown June-July with
# southwest monsoon, harvested Oct-Dec.
# Top tier = top districts in top 3 states (Gujarat, Maharashtra, Telangana).
# ============================================================================
COTTON_SITES_IN = [
    # ---- Gujarat (~28%) ----
    {"county":"Rajkot",                 "seat":"Rajkot",        "lat":22.3039, "lon":70.8022, "region":"GJ_SAURASHTRA","state":"GJ", "top_tier":True},
    {"county":"Amreli",                 "seat":"Amreli",        "lat":21.6017, "lon":71.2208, "region":"GJ_SAURASHTRA","state":"GJ", "top_tier":True},
    {"county":"Bhavnagar",              "seat":"Bhavnagar",     "lat":21.7645, "lon":72.1519, "region":"GJ_SAURASHTRA","state":"GJ", "top_tier":True},
    {"county":"Junagadh",               "seat":"Junagadh",      "lat":21.5222, "lon":70.4579, "region":"GJ_SAURASHTRA","state":"GJ", "top_tier":True},
    {"county":"Surendranagar",          "seat":"Surendranagar", "lat":22.7196, "lon":71.6369, "region":"GJ_SAURASHTRA","state":"GJ", "top_tier":True},
    {"county":"Botad",                  "seat":"Botad",         "lat":22.1697, "lon":71.6669, "region":"GJ_SAURASHTRA","state":"GJ", "top_tier":False},
    {"county":"Jamnagar",               "seat":"Jamnagar",      "lat":22.4707, "lon":70.0577, "region":"GJ_SAURASHTRA","state":"GJ", "top_tier":False},
    {"county":"Morbi",                  "seat":"Morbi",         "lat":22.8173, "lon":70.8378, "region":"GJ_SAURASHTRA","state":"GJ", "top_tier":False},
    {"county":"Banaskantha",            "seat":"Palanpur",      "lat":24.1717, "lon":72.4347, "region":"GJ_NORTH",     "state":"GJ", "top_tier":False},
    {"county":"Mehsana",                "seat":"Mehsana",       "lat":23.5879, "lon":72.3693, "region":"GJ_NORTH",     "state":"GJ", "top_tier":False},
    {"county":"Patan",                  "seat":"Patan",         "lat":23.8500, "lon":72.1333, "region":"GJ_NORTH",     "state":"GJ", "top_tier":False},
    {"county":"Kutch",                  "seat":"Bhuj",          "lat":23.2533, "lon":69.6694, "region":"GJ_KUTCH",     "state":"GJ", "top_tier":False},
    {"county":"Vadodara",               "seat":"Vadodara",      "lat":22.3072, "lon":73.1812, "region":"GJ_CENTRAL",   "state":"GJ", "top_tier":False},
    {"county":"Bharuch",                "seat":"Bharuch",       "lat":21.7050, "lon":72.9959, "region":"GJ_CENTRAL",   "state":"GJ", "top_tier":False},

    # ---- Maharashtra (~25%) Vidarbha + Marathwada ----
    {"county":"Yavatmal",               "seat":"Yavatmal",      "lat":20.3899, "lon":78.1307, "region":"MH_VIDARBHA",  "state":"MH", "top_tier":True},
    {"county":"Akola",                  "seat":"Akola",         "lat":20.7090, "lon":77.0021, "region":"MH_VIDARBHA",  "state":"MH", "top_tier":True},
    {"county":"Amravati",               "seat":"Amravati",      "lat":20.9374, "lon":77.7796, "region":"MH_VIDARBHA",  "state":"MH", "top_tier":True},
    {"county":"Buldhana",               "seat":"Buldhana",      "lat":20.5292, "lon":76.1842, "region":"MH_VIDARBHA",  "state":"MH", "top_tier":True},
    {"county":"Wardha",                 "seat":"Wardha",        "lat":20.7453, "lon":78.6022, "region":"MH_VIDARBHA",  "state":"MH", "top_tier":True},
    {"county":"Nagpur",                 "seat":"Nagpur",        "lat":21.1458, "lon":79.0882, "region":"MH_VIDARBHA",  "state":"MH", "top_tier":False},
    {"county":"Washim",                 "seat":"Washim",        "lat":20.1119, "lon":77.1336, "region":"MH_VIDARBHA",  "state":"MH", "top_tier":False},
    {"county":"Chandrapur",             "seat":"Chandrapur",    "lat":19.9615, "lon":79.2961, "region":"MH_VIDARBHA",  "state":"MH", "top_tier":False},
    {"county":"Jalgaon",                "seat":"Jalgaon",       "lat":21.0077, "lon":75.5626, "region":"MH_KHANDESH",  "state":"MH", "top_tier":True},
    {"county":"Dhule",                  "seat":"Dhule",         "lat":20.9042, "lon":74.7749, "region":"MH_KHANDESH",  "state":"MH", "top_tier":False},
    {"county":"Aurangabad",             "seat":"Aurangabad",    "lat":19.8762, "lon":75.3433, "region":"MH_MARATHWADA","state":"MH", "top_tier":True},
    {"county":"Jalna",                  "seat":"Jalna",         "lat":19.8347, "lon":75.8816, "region":"MH_MARATHWADA","state":"MH", "top_tier":False},
    {"county":"Beed",                   "seat":"Beed",          "lat":18.9894, "lon":75.7568, "region":"MH_MARATHWADA","state":"MH", "top_tier":False},
    {"county":"Parbhani",               "seat":"Parbhani",      "lat":19.2680, "lon":76.7644, "region":"MH_MARATHWADA","state":"MH", "top_tier":False},
    {"county":"Nanded",                 "seat":"Nanded",        "lat":19.1383, "lon":77.3210, "region":"MH_MARATHWADA","state":"MH", "top_tier":False},

    # ---- Telangana (~15%) ----
    {"county":"Adilabad",               "seat":"Adilabad",      "lat":19.6711, "lon":78.5320, "region":"TG_NORTH",     "state":"TG", "top_tier":True},
    {"county":"Warangal",               "seat":"Warangal",      "lat":17.9784, "lon":79.6000, "region":"TG_CENTRAL",   "state":"TG", "top_tier":True},
    {"county":"Khammam",                "seat":"Khammam",       "lat":17.2473, "lon":80.1514, "region":"TG_EAST",      "state":"TG", "top_tier":True},
    {"county":"Nalgonda",               "seat":"Nalgonda",      "lat":17.0575, "lon":79.2674, "region":"TG_SOUTH",     "state":"TG", "top_tier":True},
    {"county":"Karimnagar",             "seat":"Karimnagar",    "lat":18.4386, "lon":79.1288, "region":"TG_NORTH",     "state":"TG", "top_tier":False},
    {"county":"Mahbubnagar",            "seat":"Mahbubnagar",   "lat":16.7393, "lon":77.9974, "region":"TG_SOUTH",     "state":"TG", "top_tier":False},
    {"county":"Nizamabad",              "seat":"Nizamabad",     "lat":18.6725, "lon":78.0941, "region":"TG_NORTH",     "state":"TG", "top_tier":False},
    {"county":"Medak",                  "seat":"Medak",         "lat":18.0530, "lon":78.2715, "region":"TG_CENTRAL",   "state":"TG", "top_tier":False},

    # ---- Andhra Pradesh ----
    {"county":"Guntur",                 "seat":"Guntur",        "lat":16.3067, "lon":80.4365, "region":"AP_COASTAL",   "state":"AP", "top_tier":True},
    {"county":"Kurnool",                "seat":"Kurnool",       "lat":15.8281, "lon":78.0373, "region":"AP_RAYALA",    "state":"AP", "top_tier":False},
    {"county":"Prakasam",               "seat":"Ongole",        "lat":15.5057, "lon":80.0499, "region":"AP_COASTAL",   "state":"AP", "top_tier":False},

    # ---- Karnataka ----
    {"county":"Raichur",                "seat":"Raichur",       "lat":16.2076, "lon":77.3463, "region":"KA_NORTH",     "state":"KA", "top_tier":True},
    {"county":"Dharwad",                "seat":"Dharwad",       "lat":15.4589, "lon":75.0078, "region":"KA_NORTH",     "state":"KA", "top_tier":False},
    {"county":"Ballari",                "seat":"Ballari",       "lat":15.1394, "lon":76.9214, "region":"KA_NORTH",     "state":"KA", "top_tier":False},
    {"county":"Haveri",                 "seat":"Haveri",        "lat":14.7935, "lon":75.4044, "region":"KA_NORTH",     "state":"KA", "top_tier":False},
    {"county":"Gadag",                  "seat":"Gadag",         "lat":15.4297, "lon":75.6342, "region":"KA_NORTH",     "state":"KA", "top_tier":False},

    # ---- Madhya Pradesh ----
    {"county":"Khargone",               "seat":"Khargone",      "lat":21.8232, "lon":75.6149, "region":"MP_NIMAR",     "state":"MP", "top_tier":False},
    {"county":"Khandwa",                "seat":"Khandwa",       "lat":21.8231, "lon":76.3525, "region":"MP_NIMAR",     "state":"MP", "top_tier":False},
    {"county":"Burhanpur",              "seat":"Burhanpur",     "lat":21.3132, "lon":76.2244, "region":"MP_NIMAR",     "state":"MP", "top_tier":False},
    {"county":"Dhar",                   "seat":"Dhar",          "lat":22.6010, "lon":75.3037, "region":"MP_MALWA",     "state":"MP", "top_tier":False},

    # ---- Rajasthan (irrigated, semi-arid) ----
    {"county":"Sri Ganganagar",         "seat":"Sri Ganganagar","lat":29.9094, "lon":73.8800, "region":"RJ_NORTH",     "state":"RJ", "top_tier":True},
    {"county":"Hanumangarh",            "seat":"Hanumangarh",   "lat":29.6182, "lon":74.3294, "region":"RJ_NORTH",     "state":"RJ", "top_tier":False},
    {"county":"Bikaner",                "seat":"Bikaner",       "lat":28.0229, "lon":73.3119, "region":"RJ_NORTH",     "state":"RJ", "top_tier":False},

    # ---- Haryana ----
    {"county":"Hisar",                  "seat":"Hisar",         "lat":29.1492, "lon":75.7217, "region":"HR",           "state":"HR", "top_tier":True},
    {"county":"Sirsa",                  "seat":"Sirsa",         "lat":29.5347, "lon":75.0182, "region":"HR",           "state":"HR", "top_tier":True},
    {"county":"Fatehabad",              "seat":"Fatehabad",     "lat":29.5152, "lon":75.4538, "region":"HR",           "state":"HR", "top_tier":False},
    {"county":"Jind",                   "seat":"Jind",          "lat":29.3174, "lon":76.3144, "region":"HR",           "state":"HR", "top_tier":False},

    # ---- Punjab ----
    {"county":"Bathinda",               "seat":"Bathinda",      "lat":30.2110, "lon":74.9455, "region":"PB",           "state":"PB", "top_tier":True},
    {"county":"Mansa",                  "seat":"Mansa",         "lat":29.9988, "lon":75.3936, "region":"PB",           "state":"PB", "top_tier":False},
    {"county":"Sri Muktsar Sahib",      "seat":"Muktsar",       "lat":30.4762, "lon":74.5161, "region":"PB",           "state":"PB", "top_tier":False},
    {"county":"Fazilka",                "seat":"Fazilka",       "lat":30.4031, "lon":74.0286, "region":"PB",           "state":"PB", "top_tier":False},
    {"county":"Faridkot",               "seat":"Faridkot",      "lat":30.6754, "lon":74.7553, "region":"PB",           "state":"PB", "top_tier":False},

    # ---- Tamil Nadu ----
    {"county":"Salem",                  "seat":"Salem",         "lat":11.6643, "lon":78.1460, "region":"TN",           "state":"TN", "top_tier":False},
    {"county":"Perambalur",             "seat":"Perambalur",    "lat":11.2342, "lon":78.8807, "region":"TN",           "state":"TN", "top_tier":False},

    # ---- Odisha ----
    {"county":"Kalahandi",              "seat":"Bhawanipatna",  "lat":19.9075, "lon":83.1664, "region":"OD",           "state":"OD", "top_tier":False},
    {"county":"Rayagada",               "seat":"Rayagada",      "lat":19.1660, "lon":83.4145, "region":"OD",           "state":"OD", "top_tier":False},
]


# ============================================================================
# AUSTRALIA — Murray-Darling Basin = 91%. NSW (66%) + QLD (33%).
# Sites are towns/centers in cotton-growing valleys (the unit isn't really
# "county" or "municipality" but we keep the schema consistent).
# Top tier = the 5 largest valley centers.
# ============================================================================
COTTON_SITES_AU = [
    # ---- NSW ----
    {"county":"Moree",                  "seat":"Moree",         "lat":-29.4636,"lon":149.8439,"region":"NSW_GWYDIR",   "state":"NSW","top_tier":True},
    {"county":"Narrabri",               "seat":"Narrabri",      "lat":-30.3267,"lon":149.7831,"region":"NSW_NAMOI",    "state":"NSW","top_tier":True},
    {"county":"Wee Waa",                "seat":"Wee Waa",       "lat":-30.2225,"lon":149.4406,"region":"NSW_NAMOI",    "state":"NSW","top_tier":True},
    {"county":"Warren",                 "seat":"Warren",        "lat":-31.7028,"lon":147.8336,"region":"NSW_MACQUARIE","state":"NSW","top_tier":True},
    {"county":"Trangie",                "seat":"Trangie",       "lat":-31.9889,"lon":147.9806,"region":"NSW_MACQUARIE","state":"NSW","top_tier":False},
    {"county":"Bourke",                 "seat":"Bourke",        "lat":-30.0928,"lon":145.9356,"region":"NSW_DARLING",  "state":"NSW","top_tier":False},
    {"county":"Walgett",                "seat":"Walgett",       "lat":-30.0247,"lon":148.1186,"region":"NSW_BARWON",   "state":"NSW","top_tier":False},
    {"county":"Mungindi",               "seat":"Mungindi",      "lat":-28.9789,"lon":148.9886,"region":"NSW_BORDER",   "state":"NSW","top_tier":True},
    {"county":"Boggabilla",             "seat":"Boggabilla",    "lat":-28.7250,"lon":150.3522,"region":"NSW_BORDER",   "state":"NSW","top_tier":False},
    {"county":"Hay",                    "seat":"Hay",           "lat":-34.5114,"lon":144.8425,"region":"NSW_MURRUMBIDGEE","state":"NSW","top_tier":True},
    {"county":"Griffith",               "seat":"Griffith",      "lat":-34.2880,"lon":146.0410,"region":"NSW_MURRUMBIDGEE","state":"NSW","top_tier":True},
    {"county":"Hillston",               "seat":"Hillston",      "lat":-33.4839,"lon":145.5358,"region":"NSW_LACHLAN",  "state":"NSW","top_tier":False},
    {"county":"Forbes",                 "seat":"Forbes",        "lat":-33.3826,"lon":148.0079,"region":"NSW_LACHLAN",  "state":"NSW","top_tier":False},
    {"county":"Coleambally",            "seat":"Coleambally",   "lat":-34.7958,"lon":145.8919,"region":"NSW_MURRAY",   "state":"NSW","top_tier":False},
    {"county":"Deniliquin",             "seat":"Deniliquin",    "lat":-35.5333,"lon":144.9667,"region":"NSW_MURRAY",   "state":"NSW","top_tier":False},
    {"county":"Collarenebri",           "seat":"Collarenebri",  "lat":-29.5447,"lon":148.5786,"region":"NSW_BARWON",   "state":"NSW","top_tier":False},

    # ---- Queensland ----
    {"county":"Dalby",                  "seat":"Dalby",         "lat":-27.1814,"lon":151.2664,"region":"QLD_DARLING",  "state":"QLD","top_tier":True},
    {"county":"Goondiwindi",            "seat":"Goondiwindi",   "lat":-28.5478,"lon":150.3081,"region":"QLD_BORDER",   "state":"QLD","top_tier":True},
    {"county":"St George",              "seat":"St George",     "lat":-28.0395,"lon":148.5882,"region":"QLD_BALONNE",  "state":"QLD","top_tier":True},
    {"county":"Dirranbandi",            "seat":"Dirranbandi",   "lat":-28.5808,"lon":148.2308,"region":"QLD_BALONNE",  "state":"QLD","top_tier":True},
    {"county":"Toowoomba",              "seat":"Toowoomba",     "lat":-27.5598,"lon":151.9507,"region":"QLD_DARLING",  "state":"QLD","top_tier":False},
    {"county":"Cecil Plains",           "seat":"Cecil Plains",  "lat":-27.5306,"lon":151.1872,"region":"QLD_DARLING",  "state":"QLD","top_tier":False},
    {"county":"Pittsworth",             "seat":"Pittsworth",    "lat":-27.7236,"lon":151.6303,"region":"QLD_DARLING",  "state":"QLD","top_tier":False},
    {"county":"Inglewood",              "seat":"Inglewood",     "lat":-28.4147,"lon":151.0789,"region":"QLD_BORDER",   "state":"QLD","top_tier":False},
    {"county":"Texas",                  "seat":"Texas",         "lat":-28.8519,"lon":151.1769,"region":"QLD_BORDER",   "state":"QLD","top_tier":False},
    {"county":"Mungindi (QLD)",         "seat":"Mungindi QLD",  "lat":-28.9636,"lon":148.9858,"region":"QLD_BALONNE",  "state":"QLD","top_tier":False},
    {"county":"Emerald",                "seat":"Emerald",       "lat":-23.5273,"lon":148.1591,"region":"QLD_CENTRAL",  "state":"QLD","top_tier":True},
    {"county":"Theodore",               "seat":"Theodore",      "lat":-24.9469,"lon":150.0808,"region":"QLD_CENTRAL",  "state":"QLD","top_tier":False},
    {"county":"Biloela",                "seat":"Biloela",       "lat":-24.4047,"lon":150.5092,"region":"QLD_CENTRAL",  "state":"QLD","top_tier":False},
    {"county":"Capella",                "seat":"Capella",       "lat":-23.0833,"lon":148.0167,"region":"QLD_CENTRAL",  "state":"QLD","top_tier":False},
    {"county":"Rolleston",              "seat":"Rolleston",     "lat":-24.4528,"lon":148.6244,"region":"QLD_CENTRAL",  "state":"QLD","top_tier":False},

    # ---- Northern Australia (Ord, Burdekin - small but growing) ----
    {"county":"Kununurra",              "seat":"Kununurra",     "lat":-15.7781,"lon":128.7372,"region":"WA_ORD",       "state":"WA", "top_tier":False},
    {"county":"Ayr",                    "seat":"Ayr",           "lat":-19.5778,"lon":147.4014,"region":"QLD_BURDEKIN", "state":"QLD","top_tier":False},
]


# ============================================================================
# TURKEY — 3 regions: GAP (Southeast Anatolia ~60%), Cukurova (~20%), Aegean (~20%).
# Top 6 provinces = 86% of production. Site granularity = province + key
# sub-districts where possible.
# Top tier = Sanliurfa, Diyarbakir, Aydin, Hatay, Izmir, Adana.
# ============================================================================
COTTON_SITES_TR = [
    # ---- GAP / Southeast Anatolia (~60%) ----
    {"county":"Şanlıurfa",              "seat":"Şanlıurfa",     "lat":37.1591, "lon":38.7969, "region":"TR_GAP",       "state":"GAP","top_tier":True},
    {"county":"Harran",                 "seat":"Harran",        "lat":36.8631, "lon":39.0319, "region":"TR_GAP",       "state":"GAP","top_tier":True},
    {"county":"Akçakale",               "seat":"Akçakale",      "lat":36.7100, "lon":38.9447, "region":"TR_GAP",       "state":"GAP","top_tier":True},
    {"county":"Viranşehir",             "seat":"Viranşehir",    "lat":37.2256, "lon":39.7592, "region":"TR_GAP",       "state":"GAP","top_tier":True},
    {"county":"Suruç",                  "seat":"Suruç",         "lat":36.9747, "lon":38.4253, "region":"TR_GAP",       "state":"GAP","top_tier":True},
    {"county":"Birecik",                "seat":"Birecik",       "lat":37.0258, "lon":37.9783, "region":"TR_GAP",       "state":"GAP","top_tier":False},
    {"county":"Ceylanpınar",            "seat":"Ceylanpınar",   "lat":36.8444, "lon":40.0508, "region":"TR_GAP",       "state":"GAP","top_tier":True},
    {"county":"Diyarbakır",             "seat":"Diyarbakır",    "lat":37.9144, "lon":40.2306, "region":"TR_GAP",       "state":"GAP","top_tier":True},
    {"county":"Bismil",                 "seat":"Bismil",        "lat":37.8444, "lon":40.6633, "region":"TR_GAP",       "state":"GAP","top_tier":True},
    {"county":"Çınar",                  "seat":"Çınar",         "lat":37.7256, "lon":40.4153, "region":"TR_GAP",       "state":"GAP","top_tier":False},
    {"county":"Silvan",                 "seat":"Silvan",        "lat":38.1389, "lon":41.0006, "region":"TR_GAP",       "state":"GAP","top_tier":False},
    {"county":"Ergani",                 "seat":"Ergani",        "lat":38.2683, "lon":39.7589, "region":"TR_GAP",       "state":"GAP","top_tier":False},
    {"county":"Mardin",                 "seat":"Mardin",        "lat":37.3122, "lon":40.7350, "region":"TR_GAP",       "state":"GAP","top_tier":True},
    {"county":"Kızıltepe",              "seat":"Kızıltepe",     "lat":37.1908, "lon":40.5867, "region":"TR_GAP",       "state":"GAP","top_tier":False},
    {"county":"Nusaybin",               "seat":"Nusaybin",      "lat":37.0747, "lon":41.2153, "region":"TR_GAP",       "state":"GAP","top_tier":False},
    {"county":"Batman",                 "seat":"Batman",        "lat":37.8812, "lon":41.1351, "region":"TR_GAP",       "state":"GAP","top_tier":False},
    {"county":"Siirt",                  "seat":"Siirt",         "lat":37.9333, "lon":41.9500, "region":"TR_GAP",       "state":"GAP","top_tier":False},
    {"county":"Adıyaman",               "seat":"Adıyaman",      "lat":37.7642, "lon":38.2786, "region":"TR_GAP",       "state":"GAP","top_tier":True},
    {"county":"Kahta",                  "seat":"Kahta",         "lat":37.7825, "lon":38.6233, "region":"TR_GAP",       "state":"GAP","top_tier":False},
    {"county":"Gaziantep",              "seat":"Gaziantep",     "lat":37.0662, "lon":37.3833, "region":"TR_GAP",       "state":"GAP","top_tier":True},
    {"county":"Nizip",                  "seat":"Nizip",         "lat":37.0103, "lon":37.7942, "region":"TR_GAP",       "state":"GAP","top_tier":False},
    {"county":"Kilis",                  "seat":"Kilis",         "lat":36.7184, "lon":37.1212, "region":"TR_GAP",       "state":"GAP","top_tier":False},

    # ---- Cukurova / Mediterranean (~20%) ----
    {"county":"Adana",                  "seat":"Adana",         "lat":37.0000, "lon":35.3213, "region":"TR_CUKUROVA",  "state":"MED","top_tier":True},
    {"county":"Ceyhan",                 "seat":"Ceyhan",        "lat":37.0233, "lon":35.8175, "region":"TR_CUKUROVA",  "state":"MED","top_tier":True},
    {"county":"Yumurtalık",             "seat":"Yumurtalık",    "lat":36.7681, "lon":35.7867, "region":"TR_CUKUROVA",  "state":"MED","top_tier":False},
    {"county":"Karataş",                "seat":"Karataş",       "lat":36.5644, "lon":35.3736, "region":"TR_CUKUROVA",  "state":"MED","top_tier":False},
    {"county":"Yüreğir",                "seat":"Yüreğir",       "lat":36.9794, "lon":35.3389, "region":"TR_CUKUROVA",  "state":"MED","top_tier":False},
    {"county":"Hatay",                  "seat":"Antakya",       "lat":36.2069, "lon":36.1564, "region":"TR_CUKUROVA",  "state":"MED","top_tier":True},
    {"county":"Reyhanlı",               "seat":"Reyhanlı",      "lat":36.2658, "lon":36.5681, "region":"TR_CUKUROVA",  "state":"MED","top_tier":True},
    {"county":"Kırıkhan",               "seat":"Kırıkhan",      "lat":36.5044, "lon":36.3503, "region":"TR_CUKUROVA",  "state":"MED","top_tier":False},
    {"county":"Altınözü",               "seat":"Altınözü",      "lat":36.1011, "lon":36.2417, "region":"TR_CUKUROVA",  "state":"MED","top_tier":False},
    {"county":"Mersin",                 "seat":"Mersin",        "lat":36.8000, "lon":34.6333, "region":"TR_CUKUROVA",  "state":"MED","top_tier":False},
    {"county":"Tarsus",                 "seat":"Tarsus",        "lat":36.9167, "lon":34.8956, "region":"TR_CUKUROVA",  "state":"MED","top_tier":False},
    {"county":"Osmaniye",               "seat":"Osmaniye",      "lat":37.0683, "lon":36.2611, "region":"TR_CUKUROVA",  "state":"MED","top_tier":False},
    {"county":"Kahramanmaraş",          "seat":"Kahramanmaraş", "lat":37.5858, "lon":36.9371, "region":"TR_CUKUROVA",  "state":"MED","top_tier":False},
    {"county":"Antalya",                "seat":"Antalya",       "lat":36.8969, "lon":30.7133, "region":"TR_CUKUROVA",  "state":"MED","top_tier":False},

    # ---- Aegean (~20%) ----
    {"county":"Aydın",                  "seat":"Aydın",         "lat":37.8444, "lon":27.8458, "region":"TR_AEGEAN",    "state":"AEG","top_tier":True},
    {"county":"Söke",                   "seat":"Söke",          "lat":37.7483, "lon":27.4078, "region":"TR_AEGEAN",    "state":"AEG","top_tier":True},
    {"county":"Nazilli",                "seat":"Nazilli",       "lat":37.9133, "lon":28.3242, "region":"TR_AEGEAN",    "state":"AEG","top_tier":True},
    {"county":"Germencik",              "seat":"Germencik",     "lat":37.8703, "lon":27.6019, "region":"TR_AEGEAN",    "state":"AEG","top_tier":False},
    {"county":"İncirliova",             "seat":"İncirliova",    "lat":37.8550, "lon":27.7233, "region":"TR_AEGEAN",    "state":"AEG","top_tier":False},
    {"county":"Koçarlı",                "seat":"Koçarlı",       "lat":37.7522, "lon":27.7044, "region":"TR_AEGEAN",    "state":"AEG","top_tier":False},
    {"county":"İzmir",                  "seat":"İzmir",         "lat":38.4192, "lon":27.1287, "region":"TR_AEGEAN",    "state":"AEG","top_tier":True},
    {"county":"Bergama",                "seat":"Bergama",       "lat":39.1208, "lon":27.1808, "region":"TR_AEGEAN",    "state":"AEG","top_tier":False},
    {"county":"Menemen",                "seat":"Menemen",       "lat":38.6086, "lon":27.0664, "region":"TR_AEGEAN",    "state":"AEG","top_tier":True},
    {"county":"Torbalı",                "seat":"Torbalı",       "lat":38.1611, "lon":27.3633, "region":"TR_AEGEAN",    "state":"AEG","top_tier":False},
    {"county":"Manisa",                 "seat":"Manisa",        "lat":38.6191, "lon":27.4289, "region":"TR_AEGEAN",    "state":"AEG","top_tier":True},
    {"county":"Salihli",                "seat":"Salihli",       "lat":38.4825, "lon":28.1392, "region":"TR_AEGEAN",    "state":"AEG","top_tier":False},
    {"county":"Akhisar",                "seat":"Akhisar",       "lat":38.9217, "lon":27.8403, "region":"TR_AEGEAN",    "state":"AEG","top_tier":False},
    {"county":"Denizli",                "seat":"Denizli",       "lat":37.7765, "lon":29.0864, "region":"TR_AEGEAN",    "state":"AEG","top_tier":True},
    {"county":"Sarayköy",               "seat":"Sarayköy",      "lat":37.9244, "lon":28.9211, "region":"TR_AEGEAN",    "state":"AEG","top_tier":False},
    {"county":"Balıkesir",              "seat":"Balıkesir",     "lat":39.6533, "lon":27.8867, "region":"TR_AEGEAN",    "state":"AEG","top_tier":False},
    {"county":"Muğla",                  "seat":"Muğla",         "lat":37.2153, "lon":28.3636, "region":"TR_AEGEAN",    "state":"AEG","top_tier":False},
]

from collections import Counter as _C
for _name, _sites in [("China", COTTON_SITES_CN), ("India", COTTON_SITES_IN),
                      ("Australia", COTTON_SITES_AU), ("Turkey", COTTON_SITES_TR)]:
    _t = sum(1 for s in _sites if s["top_tier"])
    print(f"{_name:<10}: {len(_sites):>3} sites ({_t} top tier) - states: {dict(_C(s['state'] for s in _sites))}")


China     :  51 sites (24 top tier) - states: {'XJ': 43, 'HE': 2, 'SD': 2, 'HA': 1, 'HB': 1, 'JS': 1, 'AH': 1}
India     :  65 sites (22 top tier) - states: {'GJ': 14, 'MH': 15, 'TG': 8, 'AP': 3, 'KA': 5, 'MP': 4, 'RJ': 3, 'HR': 4, 'PB': 5, 'TN': 2, 'OD': 2}
Australia :  33 sites (12 top tier) - states: {'NSW': 16, 'QLD': 16, 'WA': 1}
Turkey    :  53 sites (22 top tier) - states: {'GAP': 22, 'MED': 14, 'AEG': 17}


In [27]:
import random as _random_for_retry

def _fetch_json(url, params=None, accept="application/geo+json"):
    """Fetch a JSON endpoint with robust retry semantics.

    Designed to survive Open-Meteo's connection coalescing behavior:
    - SSLEOFError / SSLError: TLS handshake fails when server overloaded
    - ConnectionResetError 10054: server forcibly closes connection
    - Empty/garbage HTTP responses: similar root cause

    Strategy: exponential backoff (2s, 4s, 8s, 16s) with random jitter to
    avoid the thundering-herd problem. After MAX_RETRIES, return None and
    let the caller skip that site - the dashboard renders partial data
    rather than failing completely.
    """
    if not url:
        return None
    headers = {"User-Agent": USER_AGENT, "Accept": accept}
    # Small initial jitter to desynchronize concurrent calls
    time.sleep(_random_for_retry.uniform(0.05, 0.30))
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=REQUEST_TIMEOUT)
            if r.status_code == 200:
                return r.json()
            if r.status_code == 404:
                return None
            if r.status_code in (429, 500, 502, 503, 504):
                # Server-side rate limit or overload - exponential backoff
                wait = (2 ** attempt) + _random_for_retry.uniform(0, 1.5)
                print(f"  ! HTTP {r.status_code} on {url[:80]}... waiting {wait:.1f}s (try {attempt}/{MAX_RETRIES})", file=sys.stderr)
                time.sleep(wait)
                continue
            return None
        except (requests.exceptions.SSLError,
                requests.exceptions.ConnectionError,
                requests.exceptions.Timeout) as e:
            # These are the symptoms of Open-Meteo overload. Wait much longer.
            wait = (2 ** attempt) + _random_for_retry.uniform(0.5, 2.5)
            err_str = str(e)[:120]
            print(f"  ! conn err on {url[:60]}... waiting {wait:.1f}s (try {attempt}/{MAX_RETRIES}): {err_str}", file=sys.stderr)
            time.sleep(wait)
        except Exception as e:
            # Unexpected error - log and retry with smaller backoff
            print(f"  ! unexpected error on {url[:60]}...: {type(e).__name__}: {e}", file=sys.stderr)
            time.sleep(1 + attempt)
    print(f"  ! GAVE UP on {url[:80]}... after {MAX_RETRIES} retries", file=sys.stderr)
    return None


# Unit conversions
def c_to_f(c):     return None if c is None else round(c * 9 / 5 + 32, 1)
def mm_to_in(mm):  return None if mm is None else round(mm / 25.4, 2)
def ms_to_mph(ms): return None if ms is None else round(ms * 2.23694, 1)
def kmh_to_mph(k): return None if k is None else round(k * 0.621371, 1)
def kt_to_mph(k):  return None if k is None else round(k * 1.15078, 1)

def wind_to_mph(value, unit_code=None):
    if value is None: return None
    unit = (unit_code or "").lower()
    if any(s in unit for s in ("km_h", "km/h")): return kmh_to_mph(value)
    if any(s in unit for s in ("m_s", "m/s")):   return ms_to_mph(value)
    if "kt" in unit or "knot" in unit:           return kt_to_mph(value)
    if "mi_h" in unit or "mph" in unit:          return round(value, 1)
    return kmh_to_mph(value)

def temp_to_f(value, unit):
    if value is None: return None
    return c_to_f(value) if (unit or "F").upper() == "C" else round(float(value), 1)

def safe_sum(vals):
    return round(sum(float(v) for v in vals if isinstance(v, (int, float))), 2)

def esc(x):
    return "" if x is None else html.escape(str(x))

def fmt_num(v, suffix="", decimals=0):
    return f"{v:.{decimals}f}{suffix}" if isinstance(v, (int, float)) else "—"

def fmt_temp(v):
    return fmt_num(v, "°F", 0)


In [28]:
def fetch_open_meteo_cotton(lat, lon, tz="America/Chicago"):
    """Fetch 3-day past + 2-day forecast with cotton-relevant metrics.

    Returns: precipitation, ET0 (FAO Penman-Monteith), soil moisture (5 layers),
    temp, humidity, wind. All hourly + daily aggregates.
    """
    params = {
        "latitude":           lat,
        "longitude":          lon,
        "timezone":           tz,
        "timeformat":         "unixtime",
        "past_days":          7,
        "forecast_days":      10,
        "precipitation_unit": "mm",
        "temperature_unit":   "fahrenheit",
        "wind_speed_unit":    "mph",
        "daily": ",".join([
            "precipitation_sum",
            "et0_fao_evapotranspiration",
            "temperature_2m_max",
            "temperature_2m_min",
        ]),
        "hourly": ",".join([
            "precipitation",
            "precipitation_probability",
            "temperature_2m",
            "relative_humidity_2m",
            "wind_speed_10m",
            "soil_moisture_0_to_1cm",
            "soil_moisture_1_to_3cm",
            "soil_moisture_3_to_9cm",
            "soil_moisture_9_to_27cm",
            "soil_moisture_27_to_81cm",
            "et0_fao_evapotranspiration",
        ]),
    }
    data = _fetch_json(OPEN_METEO_BASE, params=params, accept="application/json")
    if not data:
        return {"error": "Open-Meteo fetch failed"}

    now_epoch = int(datetime.now(tz=timezone.utc).timestamp())
    daily = data.get("daily") or {}
    daily_times = daily.get("time", []) or []

    # Parse daily entries with past/forecast classification
    daily_rows = []
    for i, ts in enumerate(daily_times):
        try:
            date_str = datetime.fromtimestamp(int(ts), tz=timezone.utc).strftime("%Y-%m-%d")
        except (TypeError, ValueError):
            date_str = str(ts)
        is_past = ts < now_epoch - 12 * 3600  # treat anything > 12h ago as "past"
        precip = daily.get("precipitation_sum", [None]*len(daily_times))[i]
        et0    = daily.get("et0_fao_evapotranspiration", [None]*len(daily_times))[i]
        tmax   = daily.get("temperature_2m_max", [None]*len(daily_times))[i]
        tmin   = daily.get("temperature_2m_min", [None]*len(daily_times))[i]
        daily_rows.append({
            "date":      date_str,
            "ts":        ts,
            "period":    "past" if is_past else "forecast",
            "precip_mm": round(float(precip), 2) if isinstance(precip, (int, float)) else None,
            "et0_mm":    round(float(et0),    2) if isinstance(et0,    (int, float)) else None,
            "tmax_f":    round(float(tmax),   1) if isinstance(tmax,   (int, float)) else None,
            "tmin_f":    round(float(tmin),   1) if isinstance(tmin,   (int, float)) else None,
        })

    # Past days aggregates. `past_days=7` means the daily array now has 7 past
    # entries followed by 7 forecast entries. past3 keeps the previous meaning
    # (last 3 days only) so existing metrics don't change; past7 is the new
    # weekly cumulative.
    past = [d for d in daily_rows if d["period"] == "past"]
    past_last_3 = past[-3:] if len(past) >= 3 else past
    past3_precip_mm = safe_sum([d["precip_mm"] for d in past_last_3])
    past3_et0_mm    = safe_sum([d["et0_mm"]    for d in past_last_3])
    past7_precip_mm = safe_sum([d["precip_mm"] for d in past])
    past7_et0_mm    = safe_sum([d["et0_mm"]    for d in past])

    # Forecast aggregate over all forecast days (now 10).
    # forecast_7d_precip_mm keeps its name for backward compat but really
    # represents the sum over the full forecast window.
    forecast_days_only = [d for d in daily_rows if d["period"] == "forecast"]
    forecast_7d_precip_mm = safe_sum([d["precip_mm"] for d in forecast_days_only])
    forecast_7d_et0_mm    = safe_sum([d["et0_mm"]    for d in forecast_days_only])

    # Hourly with soil moisture and forecast hours
    hourly = data.get("hourly") or {}
    times_h = hourly.get("time", []) or []
    soil_layers = ["soil_moisture_0_to_1cm","soil_moisture_1_to_3cm",
                   "soil_moisture_3_to_9cm","soil_moisture_9_to_27cm","soil_moisture_27_to_81cm"]
    root_zone_layers = ["soil_moisture_0_to_1cm","soil_moisture_1_to_3cm",
                        "soil_moisture_3_to_9cm","soil_moisture_9_to_27cm"]
    hourly_rows = []
    for i, ts in enumerate(times_h):
        row = {"ts": ts}
        try:
            row["time"] = datetime.fromtimestamp(int(ts), tz=timezone.utc).strftime("%Y-%m-%dT%H:%M")
        except (TypeError, ValueError):
            row["time"] = str(ts)
        for f in ("precipitation","precipitation_probability","temperature_2m",
                  "relative_humidity_2m","wind_speed_10m","et0_fao_evapotranspiration"):
            arr = hourly.get(f, []) or []
            row[f] = arr[i] if i < len(arr) else None
        for f in soil_layers:
            arr = hourly.get(f, []) or []
            row[f] = arr[i] if i < len(arr) else None
        rz = [row[f] for f in root_zone_layers if isinstance(row.get(f), (int, float))]
        row["soil_root_zone"] = round(sum(rz)/len(rz), 3) if rz else None
        hourly_rows.append(row)

    # Latest observation (most recent hourly <= now)
    latest = None
    for r in hourly_rows:
        if isinstance(r.get("ts"), (int, float)) and r["ts"] <= now_epoch:
            latest = r

    # Forecast 24h aggregate (next 24 hours)
    cutoff_idx = next((i for i, r in enumerate(hourly_rows)
                       if isinstance(r.get("ts"), (int, float)) and r["ts"] >= now_epoch - 1800), 0)
    next24 = hourly_rows[cutoff_idx:cutoff_idx + 24]
    probs  = [r["precipitation_probability"] for r in next24
              if isinstance(r.get("precipitation_probability"), (int, float))]
    precip_24h = safe_sum([r["precipitation"] for r in next24])
    et0_24h    = safe_sum([r["et0_fao_evapotranspiration"] for r in next24])

    forecast_24h = {
        "max_prob":         max(probs) if probs else None,
        "avg_prob":         round(sum(probs)/len(probs), 1) if probs else None,
        "hours_pop_gt_30":  sum(1 for p in probs if p >= 30),
        "hours_pop_gt_50":  sum(1 for p in probs if p >= 50),
        "hours_pop_gt_70":  sum(1 for p in probs if p >= 70),
        "total_mm":         precip_24h,
        "total_in":         round(precip_24h / 25.4, 2),
        "et0_mm":           et0_24h,
    }

    # Trim hourly for HTML payload size.
    # We show ~24h past + 24h forecast in the hourly chart, so 48 entries is plenty.
    # Keep only what the JS charts actually use, drop the unused soil layers.
    start_idx = max(0, cutoff_idx - 24)
    hourly_trim_raw = hourly_rows[start_idx:cutoff_idx + 24]
    hourly_trim = [{
        "time": r["time"],
        "precipitation": r["precipitation"],
        "precipitation_probability": r["precipitation_probability"],
        "soil_root_zone": r["soil_root_zone"],
    } for r in hourly_trim_raw]

    return {
        "source":          "Open-Meteo",
        "daily":           daily_rows,
        "hourly":          hourly_trim,
        "past3_precip_mm": past3_precip_mm,
        "past3_et0_mm":    past3_et0_mm,
        "past3_balance_mm":round(past3_precip_mm - past3_et0_mm, 2),
        "past7_precip_mm": past7_precip_mm,
        "past7_et0_mm":    past7_et0_mm,
        "past7_balance_mm":round(past7_precip_mm - past7_et0_mm, 2),
        "forecast_7d_precip_mm": forecast_7d_precip_mm,
        "forecast_7d_et0_mm":    forecast_7d_et0_mm,
        "forecast_24h":    forecast_24h,
        "current": {
            "temp_f":          latest.get("temperature_2m") if latest else None,
            "humidity":        latest.get("relative_humidity_2m") if latest else None,
            "wind_mph":        latest.get("wind_speed_10m") if latest else None,
            "soil_root_zone":  latest.get("soil_root_zone") if latest else None,
            "time":            latest.get("time") if latest else None,
        },
    }


def fetch_open_meteo_archive(lat, lon, years=5, tz="America/Chicago", extended=False):
    """Fetch daily climatology data from Open-Meteo archive (ERA5 reanalysis).

    Returns a dict per year keyed by year (int), each containing 365-366 daily
    points with precipitation (mm) and soil_moisture (root-zone 0-27cm avg,
    m^3/m^3). Used for the seasonal climatology charts on top-12 USDA counties.

    The archive endpoint is separate from the forecast endpoint and uses ERA5
    reanalysis data. It supports historical queries back to 1940.
    """
    current_year = 2026
    start_year = current_year - years
    start_date = f"{start_year}-01-01"
    end_date   = datetime.now(tz=timezone.utc).strftime("%Y-%m-%d")

    params = {
        "latitude":           lat,
        "longitude":          lon,
        "timezone":           tz,
        "start_date":         start_date,
        "end_date":           end_date,
        "precipitation_unit": "mm",
        "daily": ",".join([
            "precipitation_sum",
        ] + ([
            "temperature_2m_max",
            "temperature_2m_mean",
            "shortwave_radiation_sum",
        ] if extended else [])),
        # Soil moisture from ERA5 (modelled). We pull the same 4 root-zone
        # layers as the forecast to produce a consistent average.
        "hourly": ",".join([
            "soil_moisture_0_to_7cm",
            "soil_moisture_7_to_28cm",
        ]),
    }
    data = _fetch_json(OPEN_METEO_ARCHIVE, params=params, accept="application/json")
    if not data:
        return {"error": "Archive fetch failed"}

    daily = data.get("daily") or {}
    daily_dates = daily.get("time", []) or []
    daily_precip = daily.get("precipitation_sum", []) or []

    # Aggregate hourly soil moisture to daily averages
    hourly = data.get("hourly") or {}
    h_times = hourly.get("time", []) or []
    h_sm_07 = hourly.get("soil_moisture_0_to_7cm", []) or []
    h_sm_728 = hourly.get("soil_moisture_7_to_28cm", []) or []

    # Group hourly soil moisture by date
    from collections import defaultdict
    sm_by_date = defaultdict(list)
    for i, ts in enumerate(h_times):
        if not isinstance(ts, str):
            continue
        date = ts[:10]
        vals = []
        if i < len(h_sm_07) and isinstance(h_sm_07[i], (int, float)):
            vals.append(h_sm_07[i])
        if i < len(h_sm_728) and isinstance(h_sm_728[i], (int, float)):
            vals.append(h_sm_728[i])
        if vals:
            sm_by_date[date].append(sum(vals) / len(vals))

    sm_daily = {d: round(sum(v)/len(v), 4) for d, v in sm_by_date.items() if v}

    # Bucket by year
    by_year = {}
    for i, date_str in enumerate(daily_dates):
        if not isinstance(date_str, str) or len(date_str) < 10:
            continue
        year = int(date_str[:4])
        if year not in by_year:
            by_year[year] = []
        # Compute day-of-year (1-366)
        try:
            dt = datetime.strptime(date_str, "%Y-%m-%d")
            doy = dt.timetuple().tm_yday
        except ValueError:
            continue
        p = daily_precip[i] if i < len(daily_precip) else None
        record = {
            "date":   date_str,
            "doy":    doy,
            "precip_mm": round(float(p), 2) if isinstance(p, (int, float)) else None,
            "soil":   sm_daily.get(date_str),
        }
        # Extended fields for climate comparison charts
        if extended:
            tmax_c = (daily.get("temperature_2m_max") or [None])[i] if i < len((daily.get("temperature_2m_max") or [])) else None
            tmean_c = (daily.get("temperature_2m_mean") or [None])[i] if i < len((daily.get("temperature_2m_mean") or [])) else None
            solar = (daily.get("shortwave_radiation_sum") or [None])[i] if i < len((daily.get("shortwave_radiation_sum") or [])) else None
            # Convert Celsius to Fahrenheit
            record["tmax_f"]  = round(float(tmax_c) * 9/5 + 32, 1) if isinstance(tmax_c, (int, float)) else None
            record["tmean_f"] = round(float(tmean_c) * 9/5 + 32, 1) if isinstance(tmean_c, (int, float)) else None
            # Solar in MJ/m²/day (Open-Meteo native unit)
            record["solar_mj"] = round(float(solar), 2) if isinstance(solar, (int, float)) else None
        by_year[year].append(record)

    # Compute cumulative precip per year (for the accumulation chart)
    for year, days in by_year.items():
        cum = 0.0
        for d in days:
            if isinstance(d.get("precip_mm"), (int, float)):
                cum += d["precip_mm"]
            d["precip_cumulative_mm"] = round(cum, 2)

    return {"by_year": by_year}



def fetch_china_long_forecast(lat, lon, tz="Asia/Shanghai"):
    """Fetch 16-day deterministic + 35-day ensemble mean temperature forecasts.

    Two data sources are combined:

    A. **Days 1-16 deterministic** — `/v1/forecast?forecast_days=16`
       Same endpoint family as the standard 10-day forecast, just longer horizon.

    B. **Days 15-35+ ensemble mean** — tried in order:
       1. Ensemble Mean API — `hourly=temperature_2m,temperature_2m_spread`
          (endpoint URL varies, we try multiple candidates)
       2. Seasonal Forecast API EC46 — 46 days, 51 ECMWF members, mean model
       3. If both fail, `ensemble_35d` returns empty (the chart falls back
          gracefully to showing only the 16-day extended forecast)

    Returns dict with `deterministic_16d`, `ensemble_35d`, and `source` fields.
    The `source` field records which endpoint won so we can debug in prod.
    """
    # ------ Call A: extended deterministic 16-day ------
    det_response = _fetch_json(OPEN_METEO_BASE, params={
        "latitude":         lat,
        "longitude":        lon,
        "timezone":         tz,
        "forecast_days":    LONG_RANGE_DETERMINISTIC_DAYS,
        "temperature_unit": "fahrenheit",
        "daily": ",".join([
            "temperature_2m_max",
            "temperature_2m_min",
            "temperature_2m_mean",
        ]),
    })

    deterministic_16d = []
    if det_response:
        daily = det_response.get("daily") or {}
        dates = daily.get("time") or []
        tmax  = daily.get("temperature_2m_max") or []
        tmin  = daily.get("temperature_2m_min") or []
        tmean = daily.get("temperature_2m_mean") or []
        for i, d in enumerate(dates):
            deterministic_16d.append({
                "date":    d,
                "tmax_f":  round(float(tmax[i]),  1) if i < len(tmax)  and isinstance(tmax[i],  (int, float)) else None,
                "tmin_f":  round(float(tmin[i]),  1) if i < len(tmin)  and isinstance(tmin[i],  (int, float)) else None,
                "tmean_f": round(float(tmean[i]), 1) if i < len(tmean) and isinstance(tmean[i], (int, float)) else None,
            })

    # ------ Call B: ensemble mean via multiple attempts ------
    ensemble_35d = []
    source_tag = None

    def _aggregate_hourly_to_daily(hourly_dict, mean_key, spread_key):
        """Roll hourly ensemble mean+spread values into daily averages."""
        times   = hourly_dict.get("time") or []
        means   = hourly_dict.get(mean_key) or []
        spreads = hourly_dict.get(spread_key) or [] if spread_key else []
        from collections import defaultdict
        buckets = defaultdict(lambda: {"means": [], "spreads": []})
        for i, ts in enumerate(times):
            if not isinstance(ts, str) or len(ts) < 10:
                continue
            date = ts[:10]
            if i < len(means) and isinstance(means[i], (int, float)):
                buckets[date]["means"].append(means[i])
            if spread_key and i < len(spreads) and isinstance(spreads[i], (int, float)):
                buckets[date]["spreads"].append(spreads[i])
        out = []
        for date, b in sorted(buckets.items()):
            if not b["means"]:
                continue
            avg_mean = sum(b["means"]) / len(b["means"])
            avg_spread = sum(b["spreads"]) / len(b["spreads"]) if b["spreads"] else 2.0
            out.append({
                "date":            date,
                "tmean_f":         round(avg_mean, 1),
                "tmean_spread_f":  round(avg_spread, 1),
            })
        return out

    # --- Attempt 1: Ensemble Mean API on both candidate URLs ---
    for url in OPEN_METEO_ENSEMBLE_MEAN_URLS:
        if ensemble_35d:
            break
        try:
            resp = _fetch_json(url, params={
                "latitude":         lat,
                "longitude":        lon,
                "timezone":         tz,
                "forecast_days":    LONG_RANGE_ENSEMBLE_DAYS,
                "models":           "gfs_seamless",
                "temperature_unit": "fahrenheit",
                # Correct variable naming: `temperature_2m` (mean auto-returned
                # by this endpoint) + `temperature_2m_spread` (spread).
                "hourly": "temperature_2m,temperature_2m_spread",
            })
            if resp:
                hourly = resp.get("hourly") or {}
                if hourly.get("time"):
                    ensemble_35d = _aggregate_hourly_to_daily(
                        hourly, "temperature_2m", "temperature_2m_spread"
                    )
                    if ensemble_35d:
                        source_tag = f"ensemble_mean ({url.split('//')[1].split('/')[0]})"
        except Exception as e:
            print(f"    ensemble_mean {url} exception: {e}", flush=True)

    # --- Attempt 2: Seasonal Forecast API (ECMWF EC46, 46 days, 51 members) ---
    if not ensemble_35d:
        try:
            resp = _fetch_json(OPEN_METEO_SEASONAL_URL, params={
                "latitude":         lat,
                "longitude":        lon,
                "timezone":         tz,
                "forecast_days":    LONG_RANGE_ENSEMBLE_DAYS,
                "models":           "ecmwf_ifs_ec46_mean",
                "temperature_unit": "fahrenheit",
                "hourly":           "temperature_2m",
            })
            if resp:
                hourly = resp.get("hourly") or {}
                if hourly.get("time"):
                    ensemble_35d = _aggregate_hourly_to_daily(
                        hourly, "temperature_2m", None
                    )
                    if ensemble_35d:
                        source_tag = "seasonal EC46 mean"
        except Exception as e:
            print(f"    seasonal EC46 exception: {e}", flush=True)

    # --- Attempt 3: Ensemble API with individual members, compute mean côté Python ---
    if not ensemble_35d:
        try:
            resp = _fetch_json("https://ensemble-api.open-meteo.com/v1/ensemble", params={
                "latitude":         lat,
                "longitude":        lon,
                "timezone":         tz,
                "forecast_days":    LONG_RANGE_ENSEMBLE_DAYS,
                "models":           "gfs_seamless",
                "temperature_unit": "fahrenheit",
                "hourly":           "temperature_2m",
            })
            if resp:
                hourly = resp.get("hourly") or {}
                times = hourly.get("time") or []
                # Ensemble endpoint returns members as temperature_2m_member01,
                # temperature_2m_member02, etc. — plus the base temperature_2m.
                member_keys = [k for k in hourly.keys()
                               if k.startswith("temperature_2m_member")]
                if times and member_keys:
                    from collections import defaultdict
                    import statistics
                    buckets = defaultdict(list)
                    for i, ts in enumerate(times):
                        if not isinstance(ts, str) or len(ts) < 10:
                            continue
                        date = ts[:10]
                        # Collect all member values at this time step
                        values = []
                        for mk in member_keys:
                            arr = hourly.get(mk) or []
                            if i < len(arr) and isinstance(arr[i], (int, float)):
                                values.append(arr[i])
                        if values:
                            buckets[date].extend(values)
                    for date, vals in sorted(buckets.items()):
                        if not vals:
                            continue
                        ensemble_35d.append({
                            "date":            date,
                            "tmean_f":         round(statistics.mean(vals), 1),
                            "tmean_spread_f":  round(statistics.stdev(vals), 1) if len(vals) > 1 else 2.0,
                        })
                    if ensemble_35d:
                        source_tag = f"ensemble members ({len(member_keys)} members)"
        except Exception as e:
            print(f"    ensemble members exception: {e}", flush=True)

    return {
        "deterministic_16d": deterministic_16d,
        "ensemble_35d":      ensemble_35d,
        "source":            source_tag or "none",
    }


In [29]:
def fetch_noaa_for_point(lat, lon, name=None):
    """Fetch NOAA NWS data for a single lat/lon - forecast, hourly, current obs."""
    out = {"name": name, "lat": lat, "lon": lon, "error": None}
    points = _fetch_json(f"{NOAA_BASE}/points/{lat},{lon}")
    if not points:
        out["error"] = "points lookup failed"
        return out
    props = points.get("properties") or {}
    out["timezone"]        = props.get("timeZone")
    out["forecast_office"] = props.get("cwa") or props.get("gridId")

    fc = _fetch_json(props.get("forecast")) if props.get("forecast") else None
    if fc:
        out["forecast"] = [{
            "name":        p.get("name"),
            "is_day":      p.get("isDaytime"),
            "temp":        temp_to_f(p.get("temperature"), p.get("temperatureUnit")),
            "short":       p.get("shortForecast"),
            "wind":        p.get("windSpeed"),
            "wind_dir":    p.get("windDirection"),
            "precip_prob": (p.get("probabilityOfPrecipitation") or {}).get("value"),
            "start":       p.get("startTime"),
        } for p in (fc.get("properties") or {}).get("periods", [])[:14]]

    hr = _fetch_json(props.get("forecastHourly")) if props.get("forecastHourly") else None
    if hr:
        out["hourly"] = [{
            "start":       p.get("startTime"),
            "temp":        temp_to_f(p.get("temperature"), p.get("temperatureUnit")),
            "precip_prob": (p.get("probabilityOfPrecipitation") or {}).get("value"),
            "wind_speed":  p.get("windSpeed"),
            "short":       p.get("shortForecast"),
        } for p in (hr.get("properties") or {}).get("periods", [])[:48]]

    stations = _fetch_json(props.get("observationStations")) if props.get("observationStations") else None
    if stations and stations.get("features"):
        sp  = stations["features"][0].get("properties") or {}
        sid = sp.get("stationIdentifier")
        obs = _fetch_json(f"{NOAA_BASE}/stations/{sid}/observations/latest") if sid else None
        if obs:
            op  = obs.get("properties") or {}
            w   = op.get("windSpeed") or {}
            g   = op.get("windGust")  or {}
            h   = op.get("relativeHumidity") or {}
            hv  = h.get("value")
            out["current"] = {
                "station_id":      sid,
                "station_name":    sp.get("name"),
                "temp_f":          c_to_f((op.get("temperature") or {}).get("value")),
                "dewpoint_f":      c_to_f((op.get("dewpoint")    or {}).get("value")),
                "humidity":        round(hv, 1) if isinstance(hv, (int, float)) else None,
                "wind_mph":        wind_to_mph(w.get("value"), w.get("unitCode")),
                "gust_mph":        wind_to_mph(g.get("value"), g.get("unitCode")),
                "wind_dir":        (op.get("windDirection") or {}).get("value"),
                "precip_in_1h":    mm_to_in((op.get("precipitationLastHour") or {}).get("value")),
                "text_description":op.get("textDescription"),
                "timestamp":       op.get("timestamp"),
            }

    if out.get("forecast"):
        vals = [p.get("temp") for p in out["forecast"] if isinstance(p.get("temp"), (int, float))]
        if vals:
            out["forecast_min_f"], out["forecast_max_f"] = min(vals), max(vals)
    return out


def fetch_city_weather_noaa_meteo(city):
    """For NOAA statewide tab - NOAA + lightweight Open-Meteo for 7-day precip."""
    res = fetch_noaa_for_point(city["lat"], city["lon"], city["name"])
    res.update({k: v for k, v in city.items() if k not in res})
    # Add a single Open-Meteo call for cumulative precip
    om = fetch_open_meteo_cotton(city["lat"], city["lon"])
    if not om.get("error"):
        res["open_meteo"] = {
            "forecast_7d_precip_mm": safe_sum([d["precip_mm"]
                for d in om.get("daily", []) if d["period"] == "forecast"]),
            "soil_moisture": {
                "latest_avg":  om.get("current", {}).get("soil_root_zone"),
                "latest_time": om.get("current", {}).get("time"),
            },
        }
    return res


def fetch_alerts(area=None):
    data = _fetch_json(f"{NOAA_BASE}/alerts/active",
                       params={"area": area} if area else None)
    if not data:
        return []
    alerts = []
    for f in data.get("features", []):
        p = f.get("properties") or {}
        alerts.append({
            "event":       p.get("event"),
            "severity":    p.get("severity"),
            "urgency":     p.get("urgency"),
            "certainty":   p.get("certainty"),
            "headline":    p.get("headline"),
            "area":        p.get("areaDesc"),
            "description": p.get("description"),
            "instruction": p.get("instruction"),
            "effective":   p.get("effective"),
            "expires":     p.get("expires"),
            "sender":      p.get("senderName"),
        })
    rank = {"Extreme": 0, "Severe": 1, "Moderate": 2, "Minor": 3, "Unknown": 4}
    return sorted(alerts, key=lambda a: rank.get(a.get("severity") or "Unknown", 4))


In [30]:
def gather_noaa_data():
    """Statewide NOAA tab - same as v4."""
    print("[NOAA] Fetching Texas cities...")
    tx = []
    with ThreadPoolExecutor(max_workers=8) as ex:
        futures = {ex.submit(fetch_city_weather_noaa_meteo, c): c for c in TEXAS_CITIES}
        for fut in as_completed(futures):
            city = futures[fut]
            try:
                r = fut.result()
                tx.append(r)
                print(f"   {city['name']:<16} {'OK' if not r.get('error') else r['error']}")
            except Exception as e:
                print(f"   {city['name']:<16} EXCEPTION: {e}")

    print("[NOAA] Fetching US reference cities...")
    us = []
    with ThreadPoolExecutor(max_workers=6) as ex:
        futures = {ex.submit(fetch_city_weather_noaa_meteo, c): c for c in US_CITIES}
        for fut in as_completed(futures):
            city = futures[fut]
            try:
                us.append(fut.result())
                print(f"   {city['name']:<16} OK")
            except Exception as e:
                print(f"   {city['name']:<16} EXCEPTION: {e}")

    print("[NOAA] Active alerts (TX)...")
    alerts_tx = fetch_alerts("TX")
    print(f"   {len(alerts_tx)} active alerts")

    return {
        "texas":     sorted(tx, key=lambda c: c.get("name", "")),
        "us":        sorted(us, key=lambda c: c.get("name", "")),
        "alerts_tx": alerts_tx,
    }


def gather_cotton_data():
    """Texas Cotton Belt tab - Open-Meteo for all 74 counties, NOAA for top 12 only."""
    counties_out = []
    print(f"[Cotton-TX] Open-Meteo for {len(COTTON_COUNTIES)} counties (concurrency=4)...")
    with ThreadPoolExecutor(max_workers=4) as ex:
        futures = {ex.submit(fetch_open_meteo_cotton, c["lat"], c["lon"]): c
                   for c in COTTON_COUNTIES}
        for fut in as_completed(futures):
            cty = futures[fut]
            try:
                om = fut.result()
                if om.get("error"):
                    print(f"   {cty['county']:<18} OM ERROR: {om['error']}")
                    om = {}
                counties_out.append({**cty, **om})
            except Exception as e:
                print(f"   {cty['county']:<18} EXCEPTION: {e}")
                counties_out.append({**cty})

    top12 = [c for c in counties_out if c.get("top12")]

    # Climatology data (5 years history + current year YTD) for top-12 USDA only
    # to keep payload size reasonable. Used by the seasonal charts.
    print(f"[Cotton-TX] Climatology archive (5 years) for top 12 USDA counties...")
    climatology_by_county = {}
    with ThreadPoolExecutor(max_workers=4) as ex:
        futures = {ex.submit(fetch_open_meteo_archive, c["lat"], c["lon"], 5): c
                   for c in top12}
        for fut in as_completed(futures):
            cty = futures[fut]
            try:
                arch = fut.result()
                if arch.get("error"):
                    print(f"   {cty['county']:<18} archive ERROR: {arch['error']}")
                else:
                    climatology_by_county[cty["county"]] = arch.get("by_year", {})
                    n_years = len(arch.get("by_year", {}))
                    print(f"   {cty['county']:<18} climatology OK ({n_years} years)")
            except Exception as e:
                print(f"   {cty['county']:<18} climatology EXCEPTION: {e}")

    for c in counties_out:
        if c.get("top12"):
            c["climatology"] = climatology_by_county.get(c["county"], {})

    print(f"[Cotton-TX] NOAA 7-day forecast for top 12 USDA counties...")
    noaa_by_county = {}
    with ThreadPoolExecutor(max_workers=6) as ex:
        futures = {ex.submit(fetch_noaa_for_point, c["lat"], c["lon"], c["county"]): c
                   for c in top12}
        for fut in as_completed(futures):
            cty = futures[fut]
            try:
                noaa = fut.result()
                if not noaa.get("error"):
                    noaa_by_county[cty["county"]] = noaa.get("forecast", [])
                    print(f"   {cty['county']:<18} OK")
                else:
                    print(f"   {cty['county']:<18} {noaa['error']}")
            except Exception as e:
                print(f"   {cty['county']:<18} EXCEPTION: {e}")

    for c in counties_out:
        c["noaa_forecast"] = noaa_by_county.get(c["county"], [])

    counties_out.sort(key=lambda c: (0 if c.get("top12") else 1, c["county"]))
    return {
        "counties":  counties_out,
        "alerts_tx": fetch_alerts("TX"),
    }


def gather_brazil_data():
    """Brazil tab - Open-Meteo for all municipalities.

    Uses Brazilian timezone (America/Sao_Paulo). No NOAA layer (NWS only covers
    US). Same agronomic metrics as Texas: 7-day past + 10-day forecast, ET0,
    soil moisture, water balance.
    """
    print(f"[Brazil] Open-Meteo for {len(COTTON_MUNICIPALITIES_BR)} municipalities (concurrency=4)...")
    municipalities_out = []
    with ThreadPoolExecutor(max_workers=4) as ex:
        futures = {ex.submit(fetch_open_meteo_cotton, c["lat"], c["lon"], "America/Sao_Paulo"): c
                   for c in COTTON_MUNICIPALITIES_BR}
        for fut in as_completed(futures):
            cty = futures[fut]
            try:
                om = fut.result()
                if om.get("error"):
                    print(f"   {cty['county']:<26} OM ERROR: {om['error']}")
                    om = {}
                # Brazil municipalities use 'top_tier' (not 'top12' like Texas)
                municipalities_out.append({**cty, **om})
            except Exception as e:
                print(f"   {cty['county']:<26} EXCEPTION: {e}")
                municipalities_out.append({**cty})

    # Sort: top_tier first, then alphabetical
    municipalities_out.sort(key=lambda c: (0 if c.get("top_tier") else 1, c["county"]))
    return {"municipalities": municipalities_out}




def gather_country_data(country_name, sites, tz):
    """Generic gather for any country - Open-Meteo only.

    Reuses the same `fetch_open_meteo_cotton` function as Texas/Brazil so all
    countries get the identical metric structure (past3, past7, forecast_7d,
    forecast_24h, daily, hourly).

    Thread pool capped at 4 workers because Open-Meteo's free tier starts
    closing connections (SSLEOFError, ConnectionReset) above ~6-8 concurrent
    requests against the same host. With 4 workers + jittered retries the
    pipeline is stable.
    """
    print(f"[{country_name}] Open-Meteo for {len(sites)} sites (concurrency=4)...")
    out = []
    with ThreadPoolExecutor(max_workers=4) as ex:
        futures = {ex.submit(fetch_open_meteo_cotton, c["lat"], c["lon"], tz): c
                   for c in sites}
        for fut in as_completed(futures):
            cty = futures[fut]
            try:
                om = fut.result()
                if om.get("error"):
                    print(f"   {cty['county']:<28} OM ERROR: {om['error']}")
                    om = {}
                out.append({**cty, **om})
            except Exception as e:
                print(f"   {cty['county']:<28} EXCEPTION: {e}")
                out.append({**cty})
    out.sort(key=lambda c: (0 if c.get("top_tier") else 1, c["county"]))
    return {"sites": out}


print("=" * 60)
print("Fetching data for all tabs (may take 90-180 s)")
print("=" * 60)
NOAA_DATA    = gather_noaa_data()
print()
COTTON_DATA  = gather_cotton_data()
print()
BRAZIL_DATA  = gather_brazil_data()
print()
CHINA_DATA     = gather_country_data("China",     COTTON_SITES_CN, "Asia/Shanghai")

# Long-range forecast for ALL China sites (16d deterministic + 35d ensemble)
# Volume: 50 sites × (1 deterministic + 1-3 ensemble attempts) = 50-200 calls
# with concurrency=3, disk cache reuses successful results within 6h.
print(f"[China] Long-range forecast for all sites (16d GFS + 35d ENS)...")
china_all_sites = list(CHINA_DATA["sites"])
print(f"  {len(china_all_sites)} sites to fetch (parallel, max_workers=3)")

with ThreadPoolExecutor(max_workers=3) as ex:
    long_futures = {ex.submit(fetch_china_long_forecast, s["lat"], s["lon"], "Asia/Shanghai"): s
                    for s in china_all_sites}
    for fut in as_completed(long_futures):
        s = long_futures[fut]
        try:
            lr = fut.result()
            n_det = len(lr.get("deterministic_16d", []))
            n_ens = len(lr.get("ensemble_35d", []))
            src   = lr.get("source", "?")
            if n_det > 0 or n_ens > 0:
                s["long_range"] = lr
                ens_status = f"{n_ens}d ensemble via {src}" if n_ens > 0 else "ensemble MISSING"
                print(f"  {s['county']:<20} OK ({n_det}d deterministic, {ens_status})")
            else:
                print(f"  {s['county']:<20} no data returned")
        except Exception as e:
            print(f"  {s['county']:<20} EXCEPTION: {e}")

print()
INDIA_DATA     = gather_country_data("India",     COTTON_SITES_IN, "Asia/Kolkata")

# Climatology data (5 years history + current year YTD) for India top_tier only
# to keep payload size reasonable. Used by the seasonal charts on India tab.
india_top = [s for s in INDIA_DATA["sites"] if s.get("top_tier")]
print(f"[India] Climatology archive (5 years) for {len(india_top)} top_tier sites...")
india_climo_by_site = {}
with ThreadPoolExecutor(max_workers=4) as ex:
    futures = {ex.submit(fetch_open_meteo_archive, s["lat"], s["lon"], 5, "Asia/Kolkata"): s
               for s in india_top}
    for fut in as_completed(futures):
        s = futures[fut]
        try:
            arch = fut.result()
            if arch.get("error"):
                print(f"   {s['county']:<24} archive ERROR: {arch['error']}")
            else:
                india_climo_by_site[s["county"]] = arch.get("by_year", {})
                n_years = len(arch.get("by_year", {}))
                print(f"   {s['county']:<24} climatology OK ({n_years} years)")
        except Exception as e:
            print(f"   {s['county']:<24} climatology EXCEPTION: {e}")

# Attach climatology to top_tier sites in INDIA_DATA
for s in INDIA_DATA["sites"]:
    if s.get("top_tier"):
        s["climatology"] = india_climo_by_site.get(s["county"], {})

print()

# ============ XINJIANG CLIMATE COMPARISON (NXJ / SXJ) ============
# Fetch extended climatology (temperature + solar radiation) for all Xinjiang
# sites, then aggregate regionally for the NXJ vs SXJ comparison charts.
xj_sites = [s for s in CHINA_DATA["sites"]
            if s.get("region", "").startswith("NXJ") or s.get("region", "").startswith("SXJ")]
print(f"[China] Xinjiang climate comparison archive for {len(xj_sites)} sites (extended)...")

xj_archives = {}   # county -> by_year dict
with ThreadPoolExecutor(max_workers=4) as ex:
    futures = {ex.submit(fetch_open_meteo_archive, s["lat"], s["lon"], 5, "Asia/Shanghai", True): s
               for s in xj_sites}
    for fut in as_completed(futures):
        s = futures[fut]
        try:
            arch = fut.result()
            if arch.get("error"):
                print(f"   {s['county']:<24} archive ERROR: {arch['error']}")
            else:
                xj_archives[s["county"]] = arch.get("by_year", {})
        except Exception as e:
            print(f"   {s['county']:<24} archive EXCEPTION: {e}")

print(f"   Fetched {len(xj_archives)}/{len(xj_sites)} Xinjiang archives")

# Regional aggregation: for each region (NXJ, SXJ), for each year, for each DOY,
# compute mean tmean_f, tmax_f, and solar_mj across all sites in that region.
def _aggregate_xj_region(sites, region_prefix, archives):
    """Aggregate daily metrics across all sites in a region."""
    region_sites = [s for s in sites if s.get("region", "").startswith(region_prefix)]
    # Collect: year -> doy -> {tmean: [], tmax: [], solar: []}
    from collections import defaultdict
    per_year = defaultdict(lambda: defaultdict(lambda: {"tmean": [], "tmax": [], "solar": [], "date": None}))
    for site in region_sites:
        by_year = archives.get(site["county"], {})
        for year, days in by_year.items():
            for d in days:
                doy = d.get("doy")
                if doy is None:
                    continue
                bucket = per_year[year][doy]
                bucket["date"] = d.get("date")
                if isinstance(d.get("tmean_f"), (int, float)):
                    bucket["tmean"].append(d["tmean_f"])
                if isinstance(d.get("tmax_f"), (int, float)):
                    bucket["tmax"].append(d["tmax_f"])
                if isinstance(d.get("solar_mj"), (int, float)):
                    bucket["solar"].append(d["solar_mj"])
    # Reduce to means
    out = {}
    for year, doys in per_year.items():
        year_records = []
        for doy in sorted(doys.keys()):
            b = doys[doy]
            if not b["tmean"]:
                continue
            year_records.append({
                "doy": doy,
                "date": b["date"],
                "tmean_f": round(sum(b["tmean"]) / len(b["tmean"]), 1),
                "tmax_f":  round(sum(b["tmax"])  / len(b["tmax"]),  1) if b["tmax"] else None,
                "solar_mj":round(sum(b["solar"]) / len(b["solar"]), 2) if b["solar"] else None,
            })
        out[year] = year_records
    return {"site_count": len(region_sites), "by_year": out}

XJ_CLIMATE = {
    "NXJ": _aggregate_xj_region(CHINA_DATA["sites"], "NXJ", xj_archives),
    "SXJ": _aggregate_xj_region(CHINA_DATA["sites"], "SXJ", xj_archives),
}
CHINA_DATA["xj_climate"] = XJ_CLIMATE
print(f"   NXJ regional aggregate: {XJ_CLIMATE['NXJ']['site_count']} sites, {len(XJ_CLIMATE['NXJ']['by_year'])} years")
print(f"   SXJ regional aggregate: {XJ_CLIMATE['SXJ']['site_count']} sites, {len(XJ_CLIMATE['SXJ']['by_year'])} years")
print()
AUSTRALIA_DATA = gather_country_data("Australia", COTTON_SITES_AU, "Australia/Sydney")
print()
TURKEY_DATA    = gather_country_data("Turkey",    COTTON_SITES_TR, "Europe/Istanbul")
print()

# India monsoon status banner data
# IMD normal onset over Kerala: June 1; full coverage by ~mid-July;
# withdrawal starts mid-Sept from NW, completes mid-Oct.
def _india_monsoon_status():
    today = datetime.now(tz=timezone.utc)
    month, day = today.month, today.day
    if month < 6 or month > 10:
        return {"active": False, "phase": "off-season",
                "message": "Southwest monsoon currently inactive. Next onset expected around June 1 over Kerala."}
    if month == 6:
        return {"active": True, "phase": "onset",
                "message": "Monsoon onset phase (Kerala ~June 1, full coverage by ~July 15). Critical for kharif cotton sowing in central/south India."}
    if month == 7 or month == 8:
        return {"active": True, "phase": "peak",
                "message": "Peak monsoon phase. Critical for kharif cotton crop. Watch for break monsoon spells and excess rainfall events."}
    if month == 9:
        return {"active": True, "phase": "late",
                "message": "Late monsoon phase. Withdrawal begins mid-Sept from NW India. Crop in boll-development; excess rain at this stage risks quality loss."}
    return {"active": True, "phase": "withdrawal",
            "message": "Monsoon withdrawal phase. Northeast monsoon begins for southern India (Oct-Dec)."}

DATA = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "noaa":      NOAA_DATA,
    "cotton":    COTTON_DATA,
    "brazil":    BRAZIL_DATA,
    "china":     CHINA_DATA,
    "india":     INDIA_DATA,
    "australia": AUSTRALIA_DATA,
    "turkey":    TURKEY_DATA,
    "india_monsoon": _india_monsoon_status(),
}
print(f"Done. NOAA: {len(DATA['noaa']['texas'])} TX cities · "
      f"Cotton-TX: {len(DATA['cotton']['counties'])} counties · "
      f"Brazil: {len(DATA['brazil']['municipalities'])} muns · "
      f"China: {len(DATA['china']['sites'])} · "
      f"India: {len(DATA['india']['sites'])} · "
      f"Australia: {len(DATA['australia']['sites'])} · "
      f"Turkey: {len(DATA['turkey']['sites'])}")
print(f"India monsoon: {DATA['india_monsoon']['phase']} - {DATA['india_monsoon']['active']}")


Fetching data for all tabs (may take 90-180 s)
[NOAA] Fetching Texas cities...
   Pecos            OK
   Abilene          OK
   El Paso          OK
   San Angelo       OK
   Amarillo         OK
   Midland          OK
   Odessa           OK
   Lubbock          OK
   Dallas           OK
   San Antonio      OK
   Houston          OK
   Fort Worth       OK
   Austin           OK
   Waco             OK
   Brownsville      OK
   Tyler            OK
   Corpus Christi   OK
   Laredo           OK
   Beaumont         OK
   Del Rio          OK
[NOAA] Fetching US reference cities...
   New York         OK
   Phoenix          OK
   Chicago          OK
   Los Angeles      OK
   Denver           OK
   Seattle          OK
   Miami            OK
   Atlanta          OK
[NOAA] Active alerts (TX)...
   2 active alerts

[Cotton-TX] Open-Meteo for 74 counties (concurrency=4)...
[Cotton-TX] Climatology archive (5 years) for top 12 USDA counties...
   Floyd              climatology OK (6 years)
   Lubbock    

  ! HTTP 429 on https://archive-api.open-meteo.com/v1/archive... waiting 3.0s (try 1/5)


   Raichur                  climatology OK (6 years)
   Nalgonda                 climatology OK (6 years)
   Rajkot                   climatology OK (6 years)


  ! HTTP 429 on https://archive-api.open-meteo.com/v1/archive... waiting 3.0s (try 1/5)
  ! HTTP 429 on https://archive-api.open-meteo.com/v1/archive... waiting 3.3s (try 1/5)
  ! HTTP 429 on https://archive-api.open-meteo.com/v1/archive... waiting 2.8s (try 1/5)
  ! HTTP 429 on https://archive-api.open-meteo.com/v1/archive... waiting 4.5s (try 2/5)
  ! HTTP 429 on https://archive-api.open-meteo.com/v1/archive... waiting 4.6s (try 2/5)
  ! HTTP 429 on https://archive-api.open-meteo.com/v1/archive... waiting 5.0s (try 2/5)
  ! HTTP 429 on https://archive-api.open-meteo.com/v1/archive... waiting 4.4s (try 2/5)
  ! HTTP 429 on https://archive-api.open-meteo.com/v1/archive... waiting 8.9s (try 3/5)
  ! HTTP 429 on https://archive-api.open-meteo.com/v1/archive... waiting 8.3s (try 3/5)
  ! HTTP 429 on https://archive-api.open-meteo.com/v1/archive... waiting 9.4s (try 3/5)
  ! HTTP 429 on https://archive-api.open-meteo.com/v1/archive... waiting 8.7s (try 3/5)


   Warangal                 climatology OK (6 years)
   Sirsa                    climatology OK (6 years)
   Surendranagar            climatology OK (6 years)
   Sri Ganganagar           climatology OK (6 years)
   Wardha                   climatology OK (6 years)
   Yavatmal                 climatology OK (6 years)

[China] Xinjiang climate comparison archive for 43 sites (extended)...


...

   Yuli                     archive ERROR: Archive fetch failed


  ! GAVE UP on https://archive-api.open-meteo.com/v1/archive... after 5 retries


   Yizhou                   archive ERROR: Archive fetch failed


  ! GAVE UP on https://archive-api.open-meteo.com/v1/archive... after 5 retries


   Yopurga                  archive ERROR: Archive fetch failed
   Fetched 36/43 Xinjiang archives
   NXJ regional aggregate: 16 sites, 6 years
   SXJ regional aggregate: 27 sites, 6 years

[Australia] Open-Meteo for 33 sites (concurrency=4)...

[Turkey] Open-Meteo for 53 sites (concurrency=4)...

Done. NOAA: 20 TX cities · Cotton-TX: 74 counties · Brazil: 55 muns · China: 51 · India: 65 · Australia: 33 · Turkey: 53
India monsoon: peak - True


In [31]:
# HTML/CSS/JS template for the v4 cotton-belt dashboard
# Loaded into a cell in the notebook via Path read

HTML_TEMPLATE = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0" />
<title>Global Cotton Weather Dashboard</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.1/dist/chart.umd.min.js"></script>
<link href="https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,600;9..144,800&family=JetBrains+Mono:wght@400;600&family=Inter+Tight:wght@400;500;600;700&display=swap" rel="stylesheet">
<style>
:root{--paper:#0f172a;--paper-2:#111c32;--paper-3:#172238;--ink:#e5eefc;
      --ink-deep:#f8fafc;--ink-soft:#a9b7cc;--gold:#fbbf24;--gold-deep:#f59e0b;
      --rule:#334155;--rule-soft:#1e293b;--good:#22c55e;--bad:#ef4444;
      --cotton:#fef3c7;--cotton-deep:#a16207;--water:#0ea5e9;--water-deep:#0c4a6e}
*{box-sizing:border-box}
body{margin:0;background:radial-gradient(circle at top left,#13213f 0,#070b16 36%,#050816 100%);
     color:var(--ink);font-family:'Inter Tight',system-ui,sans-serif}
.container{max-width:1480px;margin:0 auto;padding:14px 10px 80px}

.masthead{background:linear-gradient(135deg,#020617 0%,#0f172a 54%,#1e293b 100%);
          border:1px solid #334155;border-radius:6px 6px 0 0;padding:28px 32px 26px;position:relative}
.masthead-title{font-family:'Fraunces',serif;font-size:38px;color:#f8fafc;
                text-shadow:0 2px 14px rgba(0,0,0,.45);margin:0 0 8px}
.masthead-sub{color:#cbd5e1}
.masthead-eyebrow{font-family:'JetBrains Mono',monospace;letter-spacing:.18em;
                  font-size:11px;color:var(--gold)}
.masthead-meta{position:absolute;right:32px;top:30px;text-align:right;
               font-family:'JetBrains Mono',monospace;font-size:11px;color:#cbd5e1}
.masthead-meta strong{color:var(--gold)}

.tabs{display:flex;background:#020617;border-bottom:2px solid var(--gold);padding:0 32px}
.tab-btn{background:transparent;border:0;color:rgba(255,255,255,.55);
         padding:14px 22px;font-family:'JetBrains Mono',monospace;font-size:11px;
         letter-spacing:.14em;text-transform:uppercase;font-weight:700;
         cursor:pointer;border-bottom:2px solid transparent;margin-bottom:-2px}
.tab-btn:hover{color:#fff}
.tab-btn.active{color:var(--gold);border-bottom-color:var(--gold);
                background:rgba(251,191,36,.08)}
.tab-btn .tab-num{font-family:'Fraunces',serif;font-size:14px;font-weight:800;margin-right:6px}
.tab-content{display:none}
.tab-content.active{display:block}

.section-bar{background:var(--gold);color:#111827;padding:9px 18px;
             display:flex;justify-content:space-between;
             font-family:'JetBrains Mono',monospace;font-size:11px;
             letter-spacing:.14em;font-weight:700}

.kpi,.card,.chart-cell,.forecast-day,.precip-day,.alert-card,
.map-legend,.no-alerts,.city-selector select,
.leaflet-popup-content-wrapper,.leaflet-popup-tip{
  background:var(--paper)!important;color:var(--ink)!important;border-color:var(--rule)!important;
}
.leaflet-bar a,.leaflet-bar a:hover{background:#f8fafc!important;color:#0f172a!important;border-color:#cbd5e1!important}
.leaflet-control-attribution{background:rgba(255,255,255,.85)!important;color:#0f172a!important}
.leaflet-control-attribution a{color:#1d4ed8!important;background:transparent!important}

.kpi{padding:18px 20px}
.kpi-label,.chart-title,.region-label,.legend-title,.forecast-name,.forecast-cond,
.forecast-wind,.precip-date,.precip-period,.soil-note,.popup-row .k,.alert-area,
.alert-meta,.footer,.obs-meta{color:var(--ink-soft)!important}
.metric-value,.section-h h2,.chart-sub,.data-table td.num.strong,
.forecast-temp,.precip-mm,.popup-title,.popup-row .v,.footer strong,
.alert-event,.alert-headline{color:var(--ink-deep)!important}
.metric-value{display:block;font-family:'Fraunces',serif;font-size:32px;font-weight:700}
.metric-context{display:block;color:var(--ink-soft);font-size:12px}
.kpi.rain-kpi .metric-value{color:#38bdf8!important}
.kpi.dry-kpi .metric-value{color:#fbbf24!important}
.kpi.alert-kpi.zero .metric-value{color:var(--good)!important}
.kpi.alert-kpi:not(.zero) .metric-value{color:var(--bad)!important}
.kpi.balance-kpi .metric-value.negative{color:#ef4444!important}
.kpi.balance-kpi .metric-value.positive{color:#22c55e!important}

.kpi-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(175px,1fr));
          gap:1px;background:var(--rule);border:1px solid var(--rule);border-top:0}

.layout{display:grid;grid-template-columns:1fr 380px;gap:28px;margin-top:28px}
@media(max-width:1100px){.layout{grid-template-columns:1fr}.masthead-meta{position:static;text-align:left}}

.card{border:1px solid var(--rule);border-radius:4px;overflow:hidden}
.card-body{padding:20px 22px}
.section-h{margin:34px 0 14px;border-bottom:2px solid var(--rule);
           display:flex;justify-content:space-between;align-items:baseline}
.section-h h2{font-family:'Fraunces',serif;font-size:22px;margin:0}
.h-eyebrow{font-family:'JetBrains Mono',monospace;font-size:10px;
           color:var(--gold-deep);letter-spacing:.16em;font-weight:700}

.weather-map{height:560px;width:100%}
.map-wrap{position:relative}
.zoom-pills,.metric-pills{position:absolute;top:14px;z-index:500;
                         display:flex;gap:2px;padding:4px;border-radius:4px;
                         background:rgba(15,23,42,.96)!important;flex-wrap:wrap;max-width:42%}
.zoom-pills{left:14px}.metric-pills{right:14px}
.zoom-pill,.metric-pill{background:transparent;border:0;color:var(--ink-soft);
                       padding:8px 11px;font-family:'JetBrains Mono',monospace;
                       font-size:10px;cursor:pointer;font-weight:700;white-space:nowrap}
.zoom-pill.active,.metric-pill.active{background:var(--gold)!important;color:#111827!important}

.map-legend{padding:12px 22px;display:flex;align-items:center;gap:16px}
.legend-scale{display:flex;height:14px;min-width:240px;flex:1}
.legend-scale span{flex:1}
.legend-stops{display:flex;justify-content:space-between;
              font-family:'JetBrains Mono',monospace;font-size:10px;color:var(--ink-soft)}

.data-table{width:100%;border-collapse:collapse;font-size:13px}
.data-table th{background:#020617!important;color:var(--ink-deep)!important;
              border-bottom:2px solid var(--gold);padding:12px 14px;text-align:left;
              font-family:'JetBrains Mono',monospace;font-size:10px}
.data-table th.num,.data-table td.num{text-align:right;
              font-family:'JetBrains Mono',monospace;font-size:13px}
.data-table td{padding:10px 14px;border-bottom:1px solid var(--rule-soft);background:var(--paper)!important}
.data-table tr:nth-child(even) td{background:var(--paper-2)!important}
.data-table tr:hover td{background:#1e293b!important}
.data-table tr.top12 td{background:rgba(251,191,36,.06)!important}
.data-table tr.top12:hover td{background:rgba(251,191,36,.12)!important}
.city-name{font-weight:700;white-space:nowrap}
.region-tag{display:inline-block;padding:1px 7px;border-radius:8px;font-family:'JetBrains Mono',monospace;
            font-size:9px;font-weight:700;letter-spacing:.06em;margin-left:8px;border:1px solid}
.region-tag.HP  {color:#fbbf24;border-color:#fbbf24}
.region-tag.RP  {color:#a78bfa;border-color:#a78bfa}
.region-tag.CB  {color:#06b6d4;border-color:#06b6d4}
.region-tag.LRGV{color:#22c55e;border-color:#22c55e}
.region-tag.FW  {color:#fb923c;border-color:#fb923c}
.region-tag.EP  {color:#f472b6;border-color:#f472b6}
/* Wildcard for any region tag not explicitly styled - uses gold */
.region-tag {color:#fbbf24;border-color:#fbbf24}
/* Specific Brazil regions (override wildcard) */
.region-tag.MT_NORTE  {color:#84cc16;border-color:#84cc16}
.region-tag.MT_MEDIO  {color:#22c55e;border-color:#22c55e}
.region-tag.MT_OESTE  {color:#10b981;border-color:#10b981}
.region-tag.MT_SUDESTE{color:#14b8a6;border-color:#14b8a6}
.region-tag.BA_OESTE  {color:#fbbf24;border-color:#fbbf24}
.region-tag.MS        {color:#06b6d4;border-color:#06b6d4}
.region-tag.GO        {color:#a78bfa;border-color:#a78bfa}
.region-tag.MG        {color:#fb923c;border-color:#fb923c}
.region-tag.MA        {color:#f472b6;border-color:#f472b6}
.region-tag.PI        {color:#ef4444;border-color:#ef4444}
/* China regions */
.region-tag[class*="NXJ"]  {color:#22d3ee;border-color:#22d3ee}
.region-tag[class*="SXJ"]  {color:#fb923c;border-color:#fb923c}
.region-tag[class*="YR_"]  {color:#a78bfa;border-color:#a78bfa}
.region-tag[class*="YZ_"]  {color:#f472b6;border-color:#f472b6}
/* India regions */
.region-tag[class*="GJ_"]  {color:#fbbf24;border-color:#fbbf24}
.region-tag[class*="MH_"]  {color:#22c55e;border-color:#22c55e}
.region-tag[class*="TG_"]  {color:#06b6d4;border-color:#06b6d4}
.region-tag[class*="AP_"]  {color:#a78bfa;border-color:#a78bfa}
.region-tag[class*="KA_"]  {color:#fb923c;border-color:#fb923c}
.region-tag[class*="MP_"]  {color:#f472b6;border-color:#f472b6}
.region-tag[class*="RJ_"]  {color:#84cc16;border-color:#84cc16}
/* Australia regions */
.region-tag[class*="NSW_"] {color:#22c55e;border-color:#22c55e}
.region-tag[class*="QLD_"] {color:#fb923c;border-color:#fb923c}
.region-tag[class*="WA_"]  {color:#a78bfa;border-color:#a78bfa}
/* Turkey regions */
.region-tag.TR_GAP        {color:#fbbf24;border-color:#fbbf24}
.region-tag.TR_CUKUROVA   {color:#06b6d4;border-color:#06b6d4}
.region-tag.TR_AEGEAN     {color:#22c55e;border-color:#22c55e}
/* Monsoon banner */
.monsoon-banner {background:linear-gradient(135deg,#1e3a8a 0%,#312e81 100%);
                 border:1px solid #6366f1;border-radius:4px;padding:14px 18px;margin:18px 0;
                 display:flex;align-items:center;gap:14px}
.monsoon-banner.active {border-color:#fbbf24;background:linear-gradient(135deg,#172554 0%,#1e3a8a 100%)}
.monsoon-icon {font-size:28px}
.monsoon-text {flex:1}
.monsoon-phase {font-family:'JetBrains Mono',monospace;font-size:10px;letter-spacing:.16em;
                color:#fbbf24!important;text-transform:uppercase;font-weight:700;margin:0 0 4px}
.monsoon-msg {color:#e5eefc!important;font-size:13px;margin:0;line-height:1.4}
.top12-star{color:var(--gold);margin-right:4px}
.region-dot{display:inline-block;width:8px;height:8px;border-radius:50%;margin-right:8px;vertical-align:middle}
.region-dot-west{background:var(--gold)}.region-dot-north{background:#2563eb}
.region-dot-central{background:#22c55e}.region-dot-east{background:#7c3aed}
.region-dot-gulf{background:#06b6d4}.region-dot-south{background:#db2777}
.region-label{font-family:'JetBrains Mono',monospace;font-size:10px;
              color:var(--ink-soft)!important;margin-left:8px;letter-spacing:.08em;text-transform:uppercase}

.balance-cell{font-weight:700}
.balance-cell.negative{color:#ef4444!important}
.balance-cell.positive{color:#22c55e!important}

.charts-grid{display:grid;grid-template-columns:1fr 1fr;gap:1px;background:var(--rule)}
.chart-cell{padding:18px 20px}
.chart-title{font-family:'JetBrains Mono',monospace;font-size:10px;letter-spacing:.14em;
             text-transform:uppercase;margin:0 0 4px;font-weight:700}
.chart-sub{font-family:'Fraunces',serif;font-size:18px;margin:0 0 12px}
.chart-canvas{height:240px;position:relative}
.chart-canvas canvas{width:100%!important;height:100%!important}

.forecast-strip{display:grid;grid-template-columns:repeat(auto-fit,minmax(110px,1fr));gap:1px;
                background:var(--rule);border:1px solid var(--rule);margin-top:12px}
.forecast-day{padding:12px 10px;text-align:center}
.forecast-day.night{background:#020617!important}
.forecast-temp{font-family:'Fraunces',serif;font-size:26px;font-weight:700}
.forecast-name,.forecast-wind,.forecast-precip{font-family:'JetBrains Mono',monospace;font-size:10px}
.forecast-precip{color:#38bdf8!important;font-weight:600}

.cotton-summary{background:linear-gradient(135deg,#172238 0%,#0c4a6e 100%);
                border-left:4px solid var(--water);padding:14px 18px;
                border-radius:0 3px 3px 0;margin-top:18px}
.cotton-summary-title{font-family:'JetBrains Mono',monospace;font-size:10px;
                      letter-spacing:.14em;text-transform:uppercase;color:#7dd3fc!important;
                      font-weight:700;margin:0 0 10px}
.cotton-stats{display:flex;gap:24px;flex-wrap:wrap}
.cotton-stat{display:flex;flex-direction:column}
.cotton-stat-value{font-family:'Fraunces',serif;font-size:24px;font-weight:700;
                   color:#f0f9ff!important;line-height:1}
.cotton-stat-label{font-family:'JetBrains Mono',monospace;font-size:10px;
                   letter-spacing:.10em;text-transform:uppercase;color:#7dd3fc!important;
                   font-weight:600;margin-top:4px}

.weather-marker{border:2px solid #fff;box-shadow:0 2px 6px rgba(0,0,0,.35);
                border-radius:18px;color:#fff;font-family:'JetBrains Mono',monospace;
                font-weight:800;font-size:11px;padding:3px 7px;white-space:nowrap;
                display:flex;align-items:center;justify-content:center}
.weather-marker.top12{border-color:var(--gold);border-width:3px;font-size:12px;padding:4px 8px}
.weather-marker.muted{background:#475569!important}

.popup-row{display:flex;justify-content:space-between;gap:14px;font-size:12px;padding:2px 0}
.popup-title{font-family:'Fraunces',serif;font-size:16px;font-weight:700;margin:0 0 4px}
.popup-cond{font-size:12px;margin:0 0 8px;padding-bottom:8px;border-bottom:1px solid var(--rule)!important}

.alerts-list{display:flex;flex-direction:column;gap:12px;max-height:800px;overflow-y:auto;padding-right:4px}
.alert-card{padding:14px 16px;border-radius:3px;border:1px solid var(--rule);
            border-left:4px solid var(--ink-soft)}
.alert-card.sev-extreme{border-left-color:#dc2626}
.alert-card.sev-severe {border-left-color:#ef4444}
.alert-card.sev-moderate{border-left-color:#f59e0b}
.alert-card.sev-minor{border-left-color:#64748b}
.alert-head{display:flex;justify-content:space-between;align-items:center;margin-bottom:6px}
.alert-event{font-family:'Fraunces',serif;font-size:15px;font-weight:700}
.alert-sev{font-family:'JetBrains Mono',monospace;font-size:10px;letter-spacing:.12em;
           font-weight:700;padding:2px 8px;border-radius:2px;color:#fff;background:#475569}
.sev-extreme .alert-sev{background:#dc2626}
.sev-severe  .alert-sev{background:#ef4444}
.sev-moderate .alert-sev{background:#f59e0b;color:#111827}
.alert-headline{font-size:13px;margin:0 0 6px}
.alert-area{font-size:12px;margin:0 0 4px}
.alert-meta{font-size:11px;font-family:'JetBrains Mono',monospace;margin:6px 0 0}
.alert-card pre{white-space:pre-wrap;font-size:12px;background:var(--paper-3)!important;
                padding:10px;border-radius:3px;font-family:'JetBrains Mono',monospace;
                border:1px solid var(--rule-soft)!important;margin:6px 0}
.alert-card details summary{cursor:pointer;color:var(--gold)!important;font-weight:600;font-size:12px}
.alert-sender{font-size:11px;color:var(--ink-soft)!important;margin:4px 0 0;font-style:italic}
.no-alerts{padding:24px;text-align:center;color:var(--good)!important;
           font-family:'Fraunces',serif;font-size:16px;border:1px solid var(--good)!important}

.city-selector{display:flex;align-items:center;gap:10px;margin-bottom:14px;flex-wrap:wrap}
.city-selector label{font-family:'JetBrains Mono',monospace;font-size:10px;
                     letter-spacing:.12em;text-transform:uppercase;color:var(--ink-soft);font-weight:600}
.city-selector select{padding:6px 30px 6px 12px;border:1px solid var(--rule)!important;
                      border-radius:3px;font-family:'Inter Tight';font-size:13px;font-weight:600;cursor:pointer}
.obs-meta{margin-left:auto;font-size:11px;font-family:'JetBrains Mono',monospace}

.footer{margin-top:36px;padding-top:16px;border-top:1px solid var(--rule);
        display:flex;justify-content:space-between;font-family:'JetBrains Mono',monospace;
        font-size:11px;color:var(--ink-soft);gap:8px;flex-wrap:wrap}

.table-filter{padding:14px 18px;background:var(--paper-2)!important;
              display:flex;gap:8px;align-items:center;flex-wrap:wrap;border-bottom:1px solid var(--rule)}
.table-filter button{background:transparent;border:1px solid var(--rule);color:var(--ink-soft);
                     padding:6px 12px;font-family:'JetBrains Mono',monospace;font-size:10px;
                     font-weight:700;letter-spacing:.08em;border-radius:3px;cursor:pointer}
.table-filter button.active{background:var(--gold);color:#111827;border-color:var(--gold)}
.table-filter span{font-family:'JetBrains Mono',monospace;font-size:10px;color:var(--ink-soft);letter-spacing:.08em;text-transform:uppercase;margin-right:6px}
</style>
</head>
<body>
<div class="container">

  <header class="masthead">
    <div class="masthead-meta">Generated <strong>__GENERATED__</strong><br/>
      Sources: NOAA / NWS · Open-Meteo (ET0, soil moisture)</div>
    <p class="masthead-eyebrow">AMAU · COTTON OPERATIONS</p>
    <h1 class="masthead-title">Global Cotton Weather Dashboard</h1>
    <p class="masthead-sub">Seven tabs covering the global cotton supply chain: <strong>NOAA Texas statewide</strong>, <strong>__N_COTTON__ Texas counties</strong>, <strong>__N_BR__ Brazil municipalities</strong>, <strong>__N_CN__ China sites</strong>, <strong>__N_IN__ India districts</strong>, <strong>__N_AU__ Australia sites</strong>, and <strong>__N_TR__ Turkey sites</strong>. Agronomic metrics: precipitation (3d / 7d past, 24h / 7d forecast), ET0 FAO, soil moisture, water balance, 10-day temperature outlook.</p>
  </header>

  <div class="tabs">
    <button class="tab-btn active" data-tab="noaa"><span class="tab-num">01</span>NOAA Texas</button>
    <button class="tab-btn"        data-tab="cotton"><span class="tab-num">02</span>Texas Cotton · __N_COTTON__</button>
    <button class="tab-btn"        data-tab="brazil"><span class="tab-num">03</span>Brazil · __N_BR__</button>
    <button class="tab-btn"        data-tab="china"><span class="tab-num">04</span>China · __N_CN__</button>
    <button class="tab-btn"        data-tab="india"><span class="tab-num">05</span>India · __N_IN__</button>
    <button class="tab-btn"        data-tab="australia"><span class="tab-num">06</span>Australia · __N_AU__</button>
    <button class="tab-btn"        data-tab="turkey"><span class="tab-num">07</span>Turkey · __N_TR__</button>
  </div>

  <!-- ========== NOAA TAB ========== -->
  <div id="tab-noaa" class="tab-content active">

    <div class="section-bar">
      <span>TEXAS — STATEWIDE NOAA SNAPSHOT</span>
      <span>LIVE · NOAA OBSERVATIONS · __N_TX_STATIONS__ STATIONS REPORTING</span>
    </div>

    <div class="kpi-grid">
      <div class="kpi"><p class="kpi-label">Hottest right now</p>__HOTTEST__</div>
      <div class="kpi"><p class="kpi-label">Coldest right now</p>__COLDEST__</div>
      <div class="kpi"><p class="kpi-label">Windiest right now</p>__WINDIEST__</div>
      <div class="kpi"><p class="kpi-label">Statewide avg temp</p>
        <span class="metric-value">__AVGTEMP__°F</span>
        <span class="metric-context">across __N_TEMPS__ cities</span></div>
      <div class="kpi rain-kpi"><p class="kpi-label">Wettest forecast 7d</p>
        <span class="metric-value">__WETTEST_MM__</span>
        <span class="metric-context">__WETTEST_NAME__</span></div>
      <div class="kpi alert-kpi"><p class="kpi-label">Severe+ alerts (TX)</p>
        <span class="metric-value">__SEVERE__</span>
        <span class="metric-context">__TOTAL_ALERTS__ total active</span></div>
    </div>

    <div class="section-h">
      <span class="h-eyebrow">SECTION 01 · GEOSPATIAL</span>
      <h2>Interactive map — West Texas / Texas / United States</h2>
    </div>
    <div class="card map-wrap">
      <div class="zoom-pills" id="zoomPillsNoaa">
        <button class="zoom-pill" data-zoom="west">West Texas</button>
        <button class="zoom-pill active" data-zoom="texas">Texas</button>
        <button class="zoom-pill" data-zoom="usa">USA</button>
      </div>
      <div class="metric-pills" id="metricPillsNoaa">
        <button class="metric-pill active" data-metric="temp">Temp</button>
        <button class="metric-pill" data-metric="wind">Wind</button>
        <button class="metric-pill" data-metric="precip1h">Precip 1h</button>
        <button class="metric-pill" data-metric="precip7">Precip 7d</button>
      </div>
      <div id="mapNoaa" class="weather-map"></div>
      <div class="map-legend">
        <span class="legend-title" id="legendTitleNoaa">Temperature scale</span>
        <div style="flex:1;min-width:240px">
          <div class="legend-scale" id="legendScaleNoaa"></div>
          <div class="legend-stops" id="legendStopsNoaa"></div>
        </div>
      </div>
    </div>

    <div class="section-h">
      <span class="h-eyebrow">SECTION 02 · CITY-BY-CITY</span>
      <h2>Texas observations and 7-day forecast precipitation</h2>
    </div>
    <div class="card">
      <table class="data-table">
        <thead><tr>
          <th>City</th><th>Conditions</th>
          <th class="num">Current °F</th><th class="num">7-day Min</th><th class="num">7-day Max</th>
          <th class="num">Humidity</th><th class="num">Wind</th>
          <th class="num">Current precip</th><th class="num">Forecast 7d precip</th>
        </tr></thead>
        <tbody>__TX_TABLE_ROWS__</tbody>
      </table>
    </div>

    <div class="layout">
      <div>
        <div class="section-h">
          <span class="h-eyebrow">SECTION 03 · ALERTS</span>
          <h2>Severe weather — active NWS alerts</h2>
        </div>
        <div class="card"><div class="card-body">
          <div class="alerts-list">__ALERTS_NOAA__</div>
        </div></div>
      </div>
      <div>
        <div class="section-h">
          <span class="h-eyebrow">SECTION 04 · QUICK INFO</span>
          <h2>About this tab</h2>
        </div>
        <div class="card"><div class="card-body">
          <p style="font-size:13px;color:var(--ink-soft)">This tab gives a statewide overview using NOAA NWS data plus Open-Meteo precipitation forecasts.</p>
          <p style="font-size:13px;color:var(--ink-soft);margin-top:14px">For cotton-specific operational metrics (ET0, soil moisture, water balance, 3-day retrospective by county), switch to the <strong style="color:var(--gold)">Cotton Belt Focus</strong> tab.</p>
        </div></div>
      </div>
    </div>
  </div>

  <!-- ========== COTTON TAB ========== -->
  <div id="tab-cotton" class="tab-content">

    <div class="section-bar">
      <span>COTTON BELT FOCUS · 74 PRODUCING COUNTIES</span>
      <span>3-DAY RETROSPECTIVE · 24H FORECAST · ET0 + SOIL MOISTURE</span>
    </div>

    <div class="kpi-grid">
      <div class="kpi rain-kpi"><p class="kpi-label">Wettest last 3 days</p>__C_WETTEST__</div>
      <div class="kpi dry-kpi"><p class="kpi-label">Driest last 3 days</p>__C_DRIEST__</div>
      <div class="kpi rain-kpi"><p class="kpi-label">Most rain expected 24h</p>__C_FORECAST_RAIN__</div>
      <div class="kpi balance-kpi"><p class="kpi-label">Avg water balance (3d)</p>__C_AVG_BALANCE__</div>
      <div class="kpi"><p class="kpi-label">Counties dry alert</p>
        <span class="metric-value">__C_DRY_COUNT__</span>
        <span class="metric-context">balance &lt; -10mm over 3 days</span></div>
      <div class="kpi"><p class="kpi-label">Counties wet alert</p>
        <span class="metric-value">__C_WET_COUNT__</span>
        <span class="metric-context">&gt; 20mm in past 3 days</span></div>
    </div>

    <div class="section-h">
      <span class="h-eyebrow">SECTION 01 · GEOSPATIAL</span>
      <h2>Cotton county map — by region or top-12 focus</h2>
    </div>
    <div class="card map-wrap">
      <div class="zoom-pills" id="zoomPillsCotton">
        <button class="zoom-pill active" data-zoom="all">All counties</button>
        <button class="zoom-pill" data-zoom="hp">High Plains</button>
        <button class="zoom-pill" data-zoom="top12">Top 12 USDA</button>
        <button class="zoom-pill" data-zoom="cb">Coastal Bend</button>
      </div>
      <div class="metric-pills" id="metricPillsCotton">
        <button class="metric-pill active" data-metric="past3_precip">Rain 3d</button>
        <button class="metric-pill" data-metric="past7_precip">Rain 7d</button>
        <button class="metric-pill" data-metric="forecast24_precip">Rain 24h fcst</button>
        <button class="metric-pill" data-metric="forecast7d_precip">Rain 7d fcst</button>
        <button class="metric-pill" data-metric="forecast24_pop">PoP 24h</button>
        <button class="metric-pill" data-metric="balance3d">Water balance</button>
        <button class="metric-pill" data-metric="soil">Soil moisture</button>
        <button class="metric-pill" data-metric="temp">Temp</button>
      </div>
      <div id="mapCotton" class="weather-map"></div>
      <div class="map-legend">
        <span class="legend-title" id="legendTitleCotton">Rainfall past 3 days (mm)</span>
        <div style="flex:1;min-width:240px">
          <div class="legend-scale" id="legendScaleCotton"></div>
          <div class="legend-stops" id="legendStopsCotton"></div>
        </div>
      </div>
    </div>

    <div class="section-h">
      <span class="h-eyebrow">SECTION 02 · COUNTY-BY-COUNTY</span>
      <h2>Cotton counties — 3-day retrospective and 24h forecast</h2>
    </div>
    <div class="card">
      <div class="table-filter">
        <span>Filter:</span>
        <button class="active" data-filter="all">All (__N_COTTON__)</button>
        <button data-filter="top12">Top 12 USDA</button>
        <button data-filter="HP">High Plains</button>
        <button data-filter="RP">Rolling Plains</button>
        <button data-filter="CB">Coastal Bend</button>
        <button data-filter="LRGV">Lower Rio Grande</button>
        <button data-filter="FW">Far West</button>
        <button data-filter="EP">Edwards Plateau</button>
      </div>
      <table class="data-table">
        <thead><tr>
          <th>County / Seat</th>
          <th class="num">Temp °F</th>
          <th class="num">Hum %</th>
          <th class="num">Rain J-3</th>
          <th class="num">Rain J-2</th>
          <th class="num">Rain J-1</th>
          <th class="num">Total 3d</th>
          <th class="num">Total 7d</th>
          <th class="num">ET0 3d</th>
          <th class="num">Balance</th>
          <th class="num">Soil</th>
          <th class="num">PoP 24h</th>
          <th class="num">Rain 24h fcst</th>
          <th class="num">Rain 7d fcst</th>
        </tr></thead>
        <tbody id="cottonTableBody">__COTTON_TABLE_ROWS__</tbody>
      </table>
    </div>

    <div class="layout">
      <div>
        <div class="section-h">
          <span class="h-eyebrow">SECTION 03 · COUNTY DETAIL</span>
          <h2>3-day retrospective + 24h forecast for a selected county</h2>
        </div>
        <div class="card"><div class="card-body">
          <div class="city-selector">
            <label>Focus county</label>
            <select id="countySelect"></select>
            <span class="obs-meta" id="obsMetaCotton">—</span>
          </div>

          <div class="cotton-summary">
            <p class="cotton-summary-title">Water balance — 3 days past + 24h forecast</p>
            <div class="cotton-stats" id="cottonStats"></div>
          </div>

          <div class="charts-grid" style="margin-top:18px">
            <div class="chart-cell">
              <p class="chart-title">Precipitation</p>
              <p class="chart-sub">Daily — past 3 days + forecast (mm)</p>
              <div class="chart-canvas"><canvas id="chartPrecipDaily"></canvas></div>
            </div>
            <div class="chart-cell">
              <p class="chart-title">Evapotranspiration</p>
              <p class="chart-sub">Daily ET0 FAO Penman-Monteith (mm)</p>
              <div class="chart-canvas"><canvas id="chartEt0Daily"></canvas></div>
            </div>
            <div class="chart-cell">
              <p class="chart-title">Hourly precip + PoP</p>
              <p class="chart-sub">Past 24h observed + next 24h forecast (mm, %)</p>
              <div class="chart-canvas"><canvas id="chartPrecipHourly"></canvas></div>
            </div>
            <div class="chart-cell">
              <p class="chart-title">Soil moisture</p>
              <p class="chart-sub">Root-zone average (0-27cm, m³/m³)</p>
              <div class="chart-canvas"><canvas id="chartSoil"></canvas></div>
            </div>
          </div>

          <div class="chart-cell" style="margin-top:1px;background:var(--paper)!important;border-top:1px solid var(--rule);padding:18px 20px">
            <p class="chart-title">10-day temperature forecast</p>
            <p class="chart-sub">Daily min/max range with mean (°F)</p>
            <div class="chart-canvas" style="height:280px"><canvas id="chartTemp10d"></canvas></div>
          </div>

          <!-- Seasonal climatology charts (top-12 USDA only) -->
          <div id="climatologyWrap" style="display:none;margin-top:18px">
            <div class="chart-cell" style="background:var(--paper)!important;border-top:1px solid var(--rule);padding:18px 20px">
              <div style="display:flex;justify-content:space-between;align-items:baseline;flex-wrap:wrap;gap:12px">
                <div>
                  <p class="chart-title">5-year climatology · top 12 USDA</p>
                  <p class="chart-sub" id="climoSub">Daily accumulated rainfall by year (mm)</p>
                </div>
                <div style="display:flex;gap:2px;padding:3px;background:rgba(15,23,42,.96);border-radius:4px">
                  <button class="metric-pill climo-period active" data-period="calendar">Jan-Dec</button>
                  <button class="metric-pill climo-period" data-period="growing">Apr-Nov (growing)</button>
                </div>
              </div>
              <div class="chart-canvas" style="height:300px;margin-top:14px"><canvas id="chartClimoRain"></canvas></div>
            </div>
            <div class="chart-cell" style="background:var(--paper)!important;border-top:1px solid var(--rule);padding:18px 20px">
              <p class="chart-title">Soil moisture climatology</p>
              <p class="chart-sub">Root-zone average (0-28cm, m³/m³) by year</p>
              <div class="chart-canvas" style="height:300px;margin-top:8px"><canvas id="chartClimoSoil"></canvas></div>
            </div>
            <p style="font-size:11px;color:var(--ink-soft);font-family:'JetBrains Mono',monospace;margin:6px 0 0;padding:0 6px">
              Source: Open-Meteo archive (ERA5 reanalysis). Bold gold line = current year (partial). Soil moisture is modeled, not measured.
            </p>
          </div>

          <p class="chart-title" style="margin-top:22px">NOAA 7-day forecast — selected county (top 12 only)</p>
          <div class="forecast-strip" id="forecastStripCotton"></div>
        </div></div>
      </div>
      <div>
        <div class="section-h">
          <span class="h-eyebrow">SECTION 04 · ALERTS</span>
          <h2>Severe weather (Texas-wide)</h2>
        </div>
        <div class="card"><div class="card-body">
          <p style="margin:0 0 14px;font-size:13px;color:var(--ink-soft)">
            Active NWS alerts for Texas affecting cotton-producing regions.
          </p>
          <div class="alerts-list">__ALERTS_COTTON__</div>
        </div></div>
      </div>
    </div>
  </div>

  
  <!-- ========== BRAZIL TAB ========== -->
  <div id="tab-brazil" class="tab-content">

    <div class="section-bar">
      <span>BRAZIL COTTON FOCUS · __N_BR__ MUNICIPALITIES</span>
      <span>7-DAY RETROSPECTIVE · 10-DAY FORECAST · ET0 + SOIL MOISTURE</span>
    </div>

    <div class="kpi-grid">
      <div class="kpi rain-kpi"><p class="kpi-label">Wettest last 7 days</p>__BR_WETTEST__</div>
      <div class="kpi dry-kpi"><p class="kpi-label">Driest last 7 days</p>__BR_DRIEST__</div>
      <div class="kpi rain-kpi"><p class="kpi-label">Most rain expected 7d</p>__BR_FORECAST_RAIN__</div>
      <div class="kpi balance-kpi"><p class="kpi-label">Avg water balance (3d)</p>__BR_AVG_BALANCE__</div>
      <div class="kpi"><p class="kpi-label">Top-tier producers</p>
        <span class="metric-value">__BR_TOP_COUNT__</span>
        <span class="metric-context">of __N_BR__ municipalities</span></div>
      <div class="kpi"><p class="kpi-label">States covered</p>
        <span class="metric-value">__BR_STATE_COUNT__</span>
        <span class="metric-context">MT, BA, MS, GO, MG, MA, PI</span></div>
    </div>

    <div class="section-h">
      <span class="h-eyebrow">SECTION 01 · GEOSPATIAL</span>
      <h2>Cotton municipality map — by region or top-tier focus</h2>
    </div>
    <div class="card map-wrap">
      <div class="zoom-pills" id="zoomPillsBrazil">
        <button class="zoom-pill active" data-zoom="all">All</button>
        <button class="zoom-pill" data-zoom="mt">Mato Grosso</button>
        <button class="zoom-pill" data-zoom="ba">Bahia (Oeste)</button>
        <button class="zoom-pill" data-zoom="top">Top tier</button>
        <button class="zoom-pill" data-zoom="matopiba">MATOPIBA</button>
      </div>
      <div class="metric-pills" id="metricPillsBrazil">
        <button class="metric-pill active" data-metric="past3_precip">Rain 3d</button>
        <button class="metric-pill" data-metric="past7_precip">Rain 7d</button>
        <button class="metric-pill" data-metric="forecast24_precip">Rain 24h fcst</button>
        <button class="metric-pill" data-metric="forecast7d_precip">Rain 7d fcst</button>
        <button class="metric-pill" data-metric="forecast24_pop">PoP 24h</button>
        <button class="metric-pill" data-metric="balance3d">Water balance</button>
        <button class="metric-pill" data-metric="soil">Soil moisture</button>
        <button class="metric-pill" data-metric="temp">Temp</button>
      </div>
      <div id="mapBrazil" class="weather-map"></div>
      <div class="map-legend">
        <span class="legend-title" id="legendTitleBrazil">Rainfall past 3 days (mm)</span>
        <div style="flex:1;min-width:240px">
          <div class="legend-scale" id="legendScaleBrazil"></div>
          <div class="legend-stops" id="legendStopsBrazil"></div>
        </div>
      </div>
    </div>

    <div class="section-h">
      <span class="h-eyebrow">SECTION 02 · MUNICIPALITY-BY-MUNICIPALITY</span>
      <h2>Brazil cotton municipalities — 7-day retrospective and forecast</h2>
    </div>
    <div class="card">
      <div class="table-filter">
        <span>Filter:</span>
        <button class="active" data-filter="all">All (__N_BR__)</button>
        <button data-filter="top_tier">Top tier</button>
        <button data-filter="MT">Mato Grosso</button>
        <button data-filter="BA">Bahia</button>
        <button data-filter="MS">MS</button>
        <button data-filter="GO">Goiás</button>
        <button data-filter="MG">MG</button>
        <button data-filter="MA">Maranhão</button>
        <button data-filter="PI">Piauí</button>
      </div>
      <table class="data-table">
        <thead><tr>
          <th>Municipality</th>
          <th class="num">Temp °F</th>
          <th class="num">Hum %</th>
          <th class="num">Rain J-3</th>
          <th class="num">Rain J-2</th>
          <th class="num">Rain J-1</th>
          <th class="num">Total 3d</th>
          <th class="num">Total 7d</th>
          <th class="num">ET0 3d</th>
          <th class="num">Balance</th>
          <th class="num">Soil</th>
          <th class="num">PoP 24h</th>
          <th class="num">Rain 24h fcst</th>
          <th class="num">Rain 7d fcst</th>
        </tr></thead>
        <tbody id="brazilTableBody">__BR_TABLE_ROWS__</tbody>
      </table>
    </div>

    <div class="layout">
      <div>
        <div class="section-h">
          <span class="h-eyebrow">SECTION 03 · MUNICIPALITY DETAIL</span>
          <h2>7-day retrospective + 10-day forecast for a selected municipality</h2>
        </div>
        <div class="card"><div class="card-body">
          <div class="city-selector">
            <label>Focus municipality</label>
            <select id="brazilSelect"></select>
            <span class="obs-meta" id="obsMetaBrazil">—</span>
          </div>

          <div class="cotton-summary">
            <p class="cotton-summary-title">Water balance — past 3 days + 24h/7d forecast</p>
            <div class="cotton-stats" id="brazilStats"></div>
          </div>

          <div class="charts-grid" style="margin-top:18px">
            <div class="chart-cell">
              <p class="chart-title">Precipitation</p>
              <p class="chart-sub">Daily — past 7 + 10-day forecast (mm)</p>
              <div class="chart-canvas"><canvas id="chartPrecipDailyBR"></canvas></div>
            </div>
            <div class="chart-cell">
              <p class="chart-title">Evapotranspiration</p>
              <p class="chart-sub">Daily ET0 FAO Penman-Monteith (mm)</p>
              <div class="chart-canvas"><canvas id="chartEt0DailyBR"></canvas></div>
            </div>
            <div class="chart-cell">
              <p class="chart-title">Hourly precip + PoP</p>
              <p class="chart-sub">Past 24h observed + next 24h forecast (mm, %)</p>
              <div class="chart-canvas"><canvas id="chartPrecipHourlyBR"></canvas></div>
            </div>
            <div class="chart-cell">
              <p class="chart-title">Soil moisture</p>
              <p class="chart-sub">Root-zone average (0-27cm, m³/m³)</p>
              <div class="chart-canvas"><canvas id="chartSoilBR"></canvas></div>
            </div>
          </div>

          <div class="chart-cell" style="margin-top:1px;background:var(--paper)!important;border-top:1px solid var(--rule);padding:18px 20px">
            <p class="chart-title">10-day temperature forecast</p>
            <p class="chart-sub">Daily min/max range with mean (°F)</p>
            <div class="chart-canvas" style="height:280px"><canvas id="chartTemp10dBR"></canvas></div>
          </div>
        </div></div>
      </div>
      <div>
        <div class="section-h">
          <span class="h-eyebrow">SECTION 04 · CROPPING CALENDAR</span>
          <h2>Brazilian cotton calendar</h2>
        </div>
        <div class="card"><div class="card-body">
          <p style="font-size:13px;color:var(--ink-soft);margin:0 0 14px">
            Brazilian cotton is grown in two main systems:
          </p>
          <p style="font-size:13px;color:var(--ink);margin:0 0 10px">
            <strong style="color:var(--gold)">Bahia (Oeste Baiano)</strong> — first crop, rainfed
          </p>
          <p style="font-size:12px;color:var(--ink-soft);margin:0 0 18px">
            Planting: November–December · Harvest: May–August. Water deficit risk concentrates in February–April (flowering through boll-filling).
          </p>
          <p style="font-size:13px;color:var(--ink);margin:0 0 10px">
            <strong style="color:var(--gold)">Mato Grosso</strong> — second crop (safrinha), planted after soybean harvest
          </p>
          <p style="font-size:12px;color:var(--ink-soft);margin:0 0 18px">
            Planting: January–February · Harvest: June–September. Critical water need in March–May.
          </p>
          <p style="font-size:13px;color:var(--ink);margin:0 0 10px">
            <strong style="color:var(--gold)">MATOPIBA frontier</strong> (MA, PI, parts of BA)
          </p>
          <p style="font-size:12px;color:var(--ink-soft);margin:0">
            Expanding cotton area on cerrado biome. Calendar between Bahia and Mato Grosso depending on latitude.
          </p>
        </div></div>
      </div>
    </div>
  </div>

    
  <!-- ========== CHINA TAB ========== -->
  <div id="tab-china" class="tab-content">

    <div class="section-bar">
      <span>CHINA COTTON FOCUS · __N_CN__ SITES</span>
      <span>7-DAY RETROSPECTIVE · 10-DAY FORECAST · ET0 + SOIL MOISTURE</span>
    </div>

    <div class="kpi-grid">
      <div class="kpi rain-kpi"><p class="kpi-label">Wettest last 7 days</p>__CHINA_WETTEST__</div>
      <div class="kpi dry-kpi"><p class="kpi-label">Driest last 7 days</p>__CHINA_DRIEST__</div>
      <div class="kpi rain-kpi"><p class="kpi-label">Most rain expected 7d</p>__CHINA_FORECAST__</div>
      <div class="kpi balance-kpi"><p class="kpi-label">Avg water balance (3d)</p>__CHINA_BALANCE__</div>
      <div class="kpi"><p class="kpi-label">Top-tier producers</p>
        <span class="metric-value">__CHINA_TOP_COUNT__</span>
        <span class="metric-context">of __N_CN__ sites</span></div>
      <div class="kpi"><p class="kpi-label">States/regions</p>
        <span class="metric-value">__CHINA_STATE_COUNT__</span>
        <span class="metric-context">covered</span></div>
    </div>

    <div class="section-h">
      <span class="h-eyebrow">SECTION 01 · GEOSPATIAL</span>
      <h2>China cotton map</h2>
    </div>
    <div class="card map-wrap">
      <div class="zoom-pills" id="zoomPills-china"></div>
      <div class="metric-pills" id="metricPills-china">
        <button class="metric-pill active" data-metric="past3_precip">Rain 3d</button>
        <button class="metric-pill" data-metric="past7_precip">Rain 7d</button>
        <button class="metric-pill" data-metric="forecast24_precip">Rain 24h fcst</button>
        <button class="metric-pill" data-metric="forecast7d_precip">Rain 7d fcst</button>
        <button class="metric-pill" data-metric="forecast24_pop">PoP 24h</button>
        <button class="metric-pill" data-metric="balance3d">Water balance</button>
        <button class="metric-pill" data-metric="soil">Soil moisture</button>
        <button class="metric-pill" data-metric="temp">Temp</button>
      </div>
      <div id="map-china" class="weather-map"></div>
      <div class="map-legend">
        <span class="legend-title" id="legendTitle-china">Rainfall past 3 days (mm)</span>
        <div style="flex:1;min-width:240px">
          <div class="legend-scale" id="legendScale-china"></div>
          <div class="legend-stops" id="legendStops-china"></div>
        </div>
      </div>
    </div>

    <div class="section-h">
      <span class="h-eyebrow">SECTION 02 · SITE-BY-SITE</span>
      <h2>China sites — 7-day retrospective and forecast</h2>
    </div>
    <div class="card">
      <div class="table-filter" id="tableFilter-china"></div>
      <table class="data-table">
        <thead><tr>
          <th>Site</th>
          <th class="num">Temp °F</th>
          <th class="num">Hum %</th>
          <th class="num">Rain J-3</th>
          <th class="num">Rain J-2</th>
          <th class="num">Rain J-1</th>
          <th class="num">Total 3d</th>
          <th class="num">Total 7d</th>
          <th class="num">ET0 3d</th>
          <th class="num">Balance</th>
          <th class="num">Soil</th>
          <th class="num">PoP 24h</th>
          <th class="num">Rain 24h fcst</th>
          <th class="num">Rain 7d fcst</th>
        </tr></thead>
        <tbody id="tableBody-china">__CHINA_TABLE_ROWS__</tbody>
      </table>
    </div>

    <div class="layout">
      <div>
        <div class="section-h">
          <span class="h-eyebrow">SECTION 03 · SITE DETAIL</span>
          <h2>7-day retrospective + 10-day forecast for a selected site</h2>
        </div>
        <div class="card"><div class="card-body">
          <div class="city-selector">
            <label>Focus site</label>
            <select id="select-china"></select>
            <span class="obs-meta" id="obsMeta-china">—</span>
          </div>

          <div class="cotton-summary">
            <p class="cotton-summary-title">Water balance — past + forecast</p>
            <div class="cotton-stats" id="stats-china"></div>
          </div>

          <div class="charts-grid" style="margin-top:18px">
            <div class="chart-cell">
              <p class="chart-title">Precipitation</p>
              <p class="chart-sub">Daily — past 7 + 10-day forecast (mm)</p>
              <div class="chart-canvas"><canvas id="chartPrecipDaily-china"></canvas></div>
            </div>
            <div class="chart-cell">
              <p class="chart-title">Evapotranspiration</p>
              <p class="chart-sub">Daily ET0 FAO Penman-Monteith (mm)</p>
              <div class="chart-canvas"><canvas id="chartEt0Daily-china"></canvas></div>
            </div>
            <div class="chart-cell">
              <p class="chart-title">Hourly precip + PoP</p>
              <p class="chart-sub">Past 24h + next 24h (mm, %)</p>
              <div class="chart-canvas"><canvas id="chartPrecipHourly-china"></canvas></div>
            </div>
            <div class="chart-cell">
              <p class="chart-title">Soil moisture</p>
              <p class="chart-sub">Root-zone average (0-27cm, m³/m³)</p>
              <div class="chart-canvas"><canvas id="chartSoil-china"></canvas></div>
            </div>
          </div>

          <div class="chart-cell" style="margin-top:1px;background:var(--paper)!important;border-top:1px solid var(--rule);padding:18px 20px">
            <p class="chart-title">10-day temperature forecast</p>
            <p class="chart-sub">Daily min/max range with mean (°F)</p>
            <div class="chart-canvas" style="height:280px"><canvas id="chartTemp10d-china"></canvas></div>
          </div>

          <!-- Long-range forecast (16-day deterministic + 35-day ensemble mean) -->
          <div class="chart-cell" style="margin-top:1px;background:var(--paper)!important;border-top:1px solid var(--rule);padding:18px 20px" id="longRangeWrap-china">
            <div style="display:flex;justify-content:space-between;align-items:baseline;flex-wrap:wrap;gap:12px;margin-bottom:8px">
              <div>
                <p class="chart-title">Long-range temperature forecast · 35 days</p>
                <p class="chart-sub">Days 1-10 high-skill · 11-16 GFS extension · 17-35 ensemble mean ± spread</p>
              </div>
              <div style="display:flex;gap:4px;padding:3px;background:rgba(15,23,42,.96);border-radius:4px">
                <button class="metric-pill lr-toggle active" data-lr="max">Tmax</button>
                <button class="metric-pill lr-toggle" data-lr="min">Tmin</button>
                <button class="metric-pill lr-toggle" data-lr="both">Both</button>
              </div>
            </div>
            <div class="chart-canvas" style="height:340px"><canvas id="chartLongRange-china"></canvas></div>
            <p style="font-size:11px;color:var(--ink-soft);font-family:'JetBrains Mono',monospace;margin:8px 0 0;padding:0 6px">
              Sources: Open-Meteo Forecast API (16-day) + Ensemble Mean / EC46 Seasonal API (35-day).
              Long-range skill degrades significantly after day 10; the 17-35 day band shows probable range only, not a firm forecast.
              Available for all Chinese sites.
            </p>
          </div>

          <!-- ============ REGIONAL AVERAGE FORECASTS (NXJ + SXJ) ============ -->
          <div class="section-h" style="margin-top:30px">
            <span class="h-eyebrow">SECTION 05 · REGIONAL OUTLOOK</span>
            <h2>Xinjiang regional average forecasts — North vs South</h2>
          </div>

          <div class="chart-cell" style="margin-top:1px;background:var(--paper)!important;border-top:1px solid var(--rule);padding:18px 20px">
            <div style="display:flex;justify-content:space-between;align-items:baseline;flex-wrap:wrap;gap:12px;margin-bottom:8px">
              <div>
                <p class="chart-title">North Xinjiang (NXJ) · average across <span id="nxjSiteCount">—</span> sites</p>
                <p class="chart-sub">Days 1-10 high-skill · 11-16 extended · 17-35 ensemble mean ± spread</p>
              </div>
              <div style="display:flex;gap:4px;padding:3px;background:rgba(15,23,42,.96);border-radius:4px">
                <button class="metric-pill nxj-toggle active" data-nxj="max">Tmax</button>
                <button class="metric-pill nxj-toggle" data-nxj="min">Tmin</button>
                <button class="metric-pill nxj-toggle" data-nxj="both">Both</button>
              </div>
            </div>
            <div class="chart-canvas" style="height:340px"><canvas id="chartRegionalNXJ"></canvas></div>
            <p style="font-size:11px;color:var(--ink-soft);font-family:'JetBrains Mono',monospace;margin:8px 0 0;padding:0 6px">
              Aggregation: mean of daily Tmax/Tmin/Tmean across all NXJ_* sites (Changji, Tacheng, Bortala, Karamay, Hami, Bingtuan Shihezi). Smoother than any single-site chart because local convective noise averages out.
            </p>
          </div>

          <div class="chart-cell" style="margin-top:1px;background:var(--paper)!important;border-top:1px solid var(--rule);padding:18px 20px">
            <div style="display:flex;justify-content:space-between;align-items:baseline;flex-wrap:wrap;gap:12px;margin-bottom:8px">
              <div>
                <p class="chart-title">South Xinjiang (SXJ) · average across <span id="sxjSiteCount">—</span> sites</p>
                <p class="chart-sub">Days 1-10 high-skill · 11-16 extended · 17-35 ensemble mean ± spread</p>
              </div>
              <div style="display:flex;gap:4px;padding:3px;background:rgba(15,23,42,.96);border-radius:4px">
                <button class="metric-pill sxj-toggle active" data-sxj="max">Tmax</button>
                <button class="metric-pill sxj-toggle" data-sxj="min">Tmin</button>
                <button class="metric-pill sxj-toggle" data-sxj="both">Both</button>
              </div>
            </div>
            <div class="chart-canvas" style="height:340px"><canvas id="chartRegionalSXJ"></canvas></div>
            <p style="font-size:11px;color:var(--ink-soft);font-family:'JetBrains Mono',monospace;margin:8px 0 0;padding:0 6px">
              Aggregation: mean of daily Tmax/Tmin/Tmean across all SXJ_* sites (Aksu, Bayingolin, Kashgar, Hotan, Bingtuan Alar/Tumxuk). Aksu prefecture (8 sites) is the largest cotton-producing region in China.
            </p>
          </div>

          <!-- ============ XINJIANG CLIMATE COMPARISON ============ -->
          <div class="section-h" style="margin-top:30px">
            <span class="h-eyebrow">SECTION 06 · CLIMATE COMPARISON</span>
            <h2>Xinjiang year-over-year comparison — current vs recent history</h2>
          </div>

          <div class="chart-cell" style="background:var(--paper)!important;border-top:1px solid var(--rule);padding:18px 20px">
            <div style="display:flex;justify-content:space-between;align-items:baseline;flex-wrap:wrap;gap:12px;margin-bottom:8px">
              <div>
                <p class="chart-title">Xinjiang region toggle</p>
                <p class="chart-sub">Switch all three charts below between North Xinjiang and South Xinjiang aggregates</p>
              </div>
              <div style="display:flex;gap:4px;padding:3px;background:rgba(15,23,42,.96);border-radius:4px">
                <button class="metric-pill xj-region-toggle active" data-xj="NXJ">North Xinjiang</button>
                <button class="metric-pill xj-region-toggle" data-xj="SXJ">South Xinjiang</button>
              </div>
            </div>
          </div>

          <!-- Chart 1: Cumulative heat days -->
          <div class="chart-cell" style="background:var(--paper)!important;border-top:1px solid var(--rule);padding:18px 20px">
            <div>
              <p class="chart-title">Cumulative heat days · <span id="xjHeatDaysRegion">NXJ</span></p>
              <p class="chart-sub">Year-to-date count of days with daily mean temp ≥ 86°F (30°C). Compare current year vs recent history.</p>
            </div>
            <div class="chart-canvas" style="height:280px;margin-top:12px"><canvas id="chartXjHeatDays"></canvas></div>
          </div>

          <!-- Chart 2: 30-day accumulated GDD -->
          <div class="chart-cell" style="background:var(--paper)!important;border-top:1px solid var(--rule);padding:18px 20px">
            <div>
              <p class="chart-title">30-day accumulated GDD · <span id="xjGddRegion">NXJ</span></p>
              <p class="chart-sub">Rolling 30-day sum of daily GDD = max(0, Tmean − 50°F). Growing degree units driving crop development.</p>
            </div>
            <div class="chart-canvas" style="height:280px;margin-top:12px"><canvas id="chartXjGdd"></canvas></div>
          </div>

          <!-- Chart 3: 30-day accumulated solar radiation -->
          <div class="chart-cell" style="background:var(--paper)!important;border-top:1px solid var(--rule);padding:18px 20px">
            <div>
              <p class="chart-title">30-day accumulated solar radiation · <span id="xjSolarRegion">NXJ</span></p>
              <p class="chart-sub">Rolling 30-day sum of daily shortwave radiation (MJ/m²). Higher = more photosynthetic potential.</p>
            </div>
            <div class="chart-canvas" style="height:280px;margin-top:12px"><canvas id="chartXjSolar"></canvas></div>
            <p style="font-size:11px;color:var(--ink-soft);font-family:'JetBrains Mono',monospace;margin:8px 0 0;padding:0 6px">
              Source: Open-Meteo archive (ERA5 reanalysis) aggregated across all Xinjiang sites in each region.
              Climate normal = mean of 2021-2025 · Year before last = 2024 · Last year = 2025 · Current year = 2026 (partial).
            </p>
          </div>
        </div></div>
      </div>
      <div>
        <div class="section-h">
          <span class="h-eyebrow">SECTION 04 · CONTEXT</span>
          <h2>China cotton context</h2>
        </div>
        <div class="card"><div class="card-body" id="context-china">__CHINA_CONTEXT__</div></div>
      </div>
    </div>
  </div>

  <!-- ========== INDIA TAB ========== -->
  <div id="tab-india" class="tab-content">

    <div class="section-bar">
      <span>INDIA COTTON FOCUS · __N_IN__ SITES</span>
      <span>7-DAY RETROSPECTIVE · 10-DAY FORECAST · ET0 + SOIL MOISTURE</span>
    </div>

    <div class="monsoon-banner" id="monsoonBanner">
      <div class="monsoon-icon">🌧️</div>
      <div class="monsoon-text">
        <p class="monsoon-phase" id="monsoonPhase">—</p>
        <p class="monsoon-msg" id="monsoonMsg">—</p>
      </div>
    </div>
    <div class="kpi-grid">
      <div class="kpi rain-kpi"><p class="kpi-label">Wettest last 7 days</p>__INDIA_WETTEST__</div>
      <div class="kpi dry-kpi"><p class="kpi-label">Driest last 7 days</p>__INDIA_DRIEST__</div>
      <div class="kpi rain-kpi"><p class="kpi-label">Most rain expected 7d</p>__INDIA_FORECAST__</div>
      <div class="kpi balance-kpi"><p class="kpi-label">Avg water balance (3d)</p>__INDIA_BALANCE__</div>
      <div class="kpi"><p class="kpi-label">Top-tier producers</p>
        <span class="metric-value">__INDIA_TOP_COUNT__</span>
        <span class="metric-context">of __N_IN__ sites</span></div>
      <div class="kpi"><p class="kpi-label">States/regions</p>
        <span class="metric-value">__INDIA_STATE_COUNT__</span>
        <span class="metric-context">covered</span></div>
    </div>

    <div class="section-h">
      <span class="h-eyebrow">SECTION 01 · GEOSPATIAL</span>
      <h2>India cotton map</h2>
    </div>
    <div class="card map-wrap">
      <div class="zoom-pills" id="zoomPills-india"></div>
      <div class="metric-pills" id="metricPills-india">
        <button class="metric-pill active" data-metric="past3_precip">Rain 3d</button>
        <button class="metric-pill" data-metric="past7_precip">Rain 7d</button>
        <button class="metric-pill" data-metric="forecast24_precip">Rain 24h fcst</button>
        <button class="metric-pill" data-metric="forecast7d_precip">Rain 7d fcst</button>
        <button class="metric-pill" data-metric="forecast24_pop">PoP 24h</button>
        <button class="metric-pill" data-metric="balance3d">Water balance</button>
        <button class="metric-pill" data-metric="soil">Soil moisture</button>
        <button class="metric-pill" data-metric="temp">Temp</button>
      </div>
      <div id="map-india" class="weather-map"></div>
      <div class="map-legend">
        <span class="legend-title" id="legendTitle-india">Rainfall past 3 days (mm)</span>
        <div style="flex:1;min-width:240px">
          <div class="legend-scale" id="legendScale-india"></div>
          <div class="legend-stops" id="legendStops-india"></div>
        </div>
      </div>
    </div>

    <div class="section-h">
      <span class="h-eyebrow">SECTION 02 · SITE-BY-SITE</span>
      <h2>India sites — 7-day retrospective and forecast</h2>
    </div>
    <div class="card">
      <div class="table-filter" id="tableFilter-india"></div>
      <table class="data-table">
        <thead><tr>
          <th>Site</th>
          <th class="num">Temp °F</th>
          <th class="num">Hum %</th>
          <th class="num">Rain J-3</th>
          <th class="num">Rain J-2</th>
          <th class="num">Rain J-1</th>
          <th class="num">Total 3d</th>
          <th class="num">Total 7d</th>
          <th class="num">ET0 3d</th>
          <th class="num">Balance</th>
          <th class="num">Soil</th>
          <th class="num">PoP 24h</th>
          <th class="num">Rain 24h fcst</th>
          <th class="num">Rain 7d fcst</th>
        </tr></thead>
        <tbody id="tableBody-india">__INDIA_TABLE_ROWS__</tbody>
      </table>
    </div>

    <div class="layout">
      <div>
        <div class="section-h">
          <span class="h-eyebrow">SECTION 03 · SITE DETAIL</span>
          <h2>7-day retrospective + 10-day forecast for a selected site</h2>
        </div>
        <div class="card"><div class="card-body">
          <div class="city-selector">
            <label>Focus site</label>
            <select id="select-india"></select>
            <span class="obs-meta" id="obsMeta-india">—</span>
          </div>

          <div class="cotton-summary">
            <p class="cotton-summary-title">Water balance — past + forecast</p>
            <div class="cotton-stats" id="stats-india"></div>
          </div>

          <div class="charts-grid" style="margin-top:18px">
            <div class="chart-cell">
              <p class="chart-title">Precipitation</p>
              <p class="chart-sub">Daily — past 7 + 10-day forecast (mm)</p>
              <div class="chart-canvas"><canvas id="chartPrecipDaily-india"></canvas></div>
            </div>
            <div class="chart-cell">
              <p class="chart-title">Evapotranspiration</p>
              <p class="chart-sub">Daily ET0 FAO Penman-Monteith (mm)</p>
              <div class="chart-canvas"><canvas id="chartEt0Daily-india"></canvas></div>
            </div>
            <div class="chart-cell">
              <p class="chart-title">Hourly precip + PoP</p>
              <p class="chart-sub">Past 24h + next 24h (mm, %)</p>
              <div class="chart-canvas"><canvas id="chartPrecipHourly-india"></canvas></div>
            </div>
            <div class="chart-cell">
              <p class="chart-title">Soil moisture</p>
              <p class="chart-sub">Root-zone average (0-27cm, m³/m³)</p>
              <div class="chart-canvas"><canvas id="chartSoil-india"></canvas></div>
            </div>
          </div>

          <div class="chart-cell" style="margin-top:1px;background:var(--paper)!important;border-top:1px solid var(--rule);padding:18px 20px">
            <p class="chart-title">10-day temperature forecast</p>
            <p class="chart-sub">Daily min/max range with mean (°F)</p>
            <div class="chart-canvas" style="height:280px"><canvas id="chartTemp10d-india"></canvas></div>
          </div>

          <!-- ============ INDIA CLIMATOLOGY (top_tier only) ============ -->
          <div id="climatologyWrapIndia" style="display:none;margin-top:18px">
            <div class="chart-cell" style="background:var(--paper)!important;border-top:1px solid var(--rule);padding:18px 20px">
              <div style="display:flex;justify-content:space-between;align-items:baseline;flex-wrap:wrap;gap:12px">
                <div>
                  <p class="chart-title">5-year climatology · top_tier India</p>
                  <p class="chart-sub" id="climoSubIndia">Daily accumulated rainfall by year (mm)</p>
                </div>
                <div style="display:flex;gap:2px;padding:3px;background:rgba(15,23,42,.96);border-radius:4px">
                  <button class="metric-pill climo-period-india active" data-period="calendar">Jan-Dec</button>
                  <button class="metric-pill climo-period-india" data-period="growing">Jun-Nov (kharif)</button>
                </div>
              </div>
              <div class="chart-canvas" style="height:300px;margin-top:14px"><canvas id="chartClimoRainIndia"></canvas></div>
            </div>
            <div class="chart-cell" style="background:var(--paper)!important;border-top:1px solid var(--rule);padding:18px 20px">
              <p class="chart-title">Soil moisture climatology</p>
              <p class="chart-sub">Root-zone average (0-28cm, m³/m³) by year</p>
              <div class="chart-canvas" style="height:300px;margin-top:8px"><canvas id="chartClimoSoilIndia"></canvas></div>
            </div>
            <p style="font-size:11px;color:var(--ink-soft);font-family:'JetBrains Mono',monospace;margin:6px 0 0;padding:0 6px">
              Source: Open-Meteo archive (ERA5 reanalysis). Bold gold line = current year (partial). Soil moisture is modeled, not measured. Kharif season = Jun-Nov (SW monsoon + cotton growing period).
            </p>
          </div>
        </div></div>
      </div>
      <div>
        <div class="section-h">
          <span class="h-eyebrow">SECTION 04 · CONTEXT</span>
          <h2>India cotton context</h2>
        </div>
        <div class="card"><div class="card-body" id="context-india">__INDIA_CONTEXT__</div></div>
      </div>
    </div>
  </div>

  <!-- ========== AUSTRALIA TAB ========== -->
  <div id="tab-australia" class="tab-content">

    <div class="section-bar">
      <span>AUSTRALIA COTTON FOCUS · __N_AU__ SITES</span>
      <span>7-DAY RETROSPECTIVE · 10-DAY FORECAST · ET0 + SOIL MOISTURE</span>
    </div>

    <div class="kpi-grid">
      <div class="kpi rain-kpi"><p class="kpi-label">Wettest last 7 days</p>__AUSTRALIA_WETTEST__</div>
      <div class="kpi dry-kpi"><p class="kpi-label">Driest last 7 days</p>__AUSTRALIA_DRIEST__</div>
      <div class="kpi rain-kpi"><p class="kpi-label">Most rain expected 7d</p>__AUSTRALIA_FORECAST__</div>
      <div class="kpi balance-kpi"><p class="kpi-label">Avg water balance (3d)</p>__AUSTRALIA_BALANCE__</div>
      <div class="kpi"><p class="kpi-label">Top-tier producers</p>
        <span class="metric-value">__AUSTRALIA_TOP_COUNT__</span>
        <span class="metric-context">of __N_AU__ sites</span></div>
      <div class="kpi"><p class="kpi-label">States/regions</p>
        <span class="metric-value">__AUSTRALIA_STATE_COUNT__</span>
        <span class="metric-context">covered</span></div>
    </div>

    <div class="section-h">
      <span class="h-eyebrow">SECTION 01 · GEOSPATIAL</span>
      <h2>Australia cotton map</h2>
    </div>
    <div class="card map-wrap">
      <div class="zoom-pills" id="zoomPills-australia"></div>
      <div class="metric-pills" id="metricPills-australia">
        <button class="metric-pill active" data-metric="past3_precip">Rain 3d</button>
        <button class="metric-pill" data-metric="past7_precip">Rain 7d</button>
        <button class="metric-pill" data-metric="forecast24_precip">Rain 24h fcst</button>
        <button class="metric-pill" data-metric="forecast7d_precip">Rain 7d fcst</button>
        <button class="metric-pill" data-metric="forecast24_pop">PoP 24h</button>
        <button class="metric-pill" data-metric="balance3d">Water balance</button>
        <button class="metric-pill" data-metric="soil">Soil moisture</button>
        <button class="metric-pill" data-metric="temp">Temp</button>
      </div>
      <div id="map-australia" class="weather-map"></div>
      <div class="map-legend">
        <span class="legend-title" id="legendTitle-australia">Rainfall past 3 days (mm)</span>
        <div style="flex:1;min-width:240px">
          <div class="legend-scale" id="legendScale-australia"></div>
          <div class="legend-stops" id="legendStops-australia"></div>
        </div>
      </div>
    </div>

    <div class="section-h">
      <span class="h-eyebrow">SECTION 02 · SITE-BY-SITE</span>
      <h2>Australia sites — 7-day retrospective and forecast</h2>
    </div>
    <div class="card">
      <div class="table-filter" id="tableFilter-australia"></div>
      <table class="data-table">
        <thead><tr>
          <th>Site</th>
          <th class="num">Temp °F</th>
          <th class="num">Hum %</th>
          <th class="num">Rain J-3</th>
          <th class="num">Rain J-2</th>
          <th class="num">Rain J-1</th>
          <th class="num">Total 3d</th>
          <th class="num">Total 7d</th>
          <th class="num">ET0 3d</th>
          <th class="num">Balance</th>
          <th class="num">Soil</th>
          <th class="num">PoP 24h</th>
          <th class="num">Rain 24h fcst</th>
          <th class="num">Rain 7d fcst</th>
        </tr></thead>
        <tbody id="tableBody-australia">__AUSTRALIA_TABLE_ROWS__</tbody>
      </table>
    </div>

    <div class="layout">
      <div>
        <div class="section-h">
          <span class="h-eyebrow">SECTION 03 · SITE DETAIL</span>
          <h2>7-day retrospective + 10-day forecast for a selected site</h2>
        </div>
        <div class="card"><div class="card-body">
          <div class="city-selector">
            <label>Focus site</label>
            <select id="select-australia"></select>
            <span class="obs-meta" id="obsMeta-australia">—</span>
          </div>

          <div class="cotton-summary">
            <p class="cotton-summary-title">Water balance — past + forecast</p>
            <div class="cotton-stats" id="stats-australia"></div>
          </div>

          <div class="charts-grid" style="margin-top:18px">
            <div class="chart-cell">
              <p class="chart-title">Precipitation</p>
              <p class="chart-sub">Daily — past 7 + 10-day forecast (mm)</p>
              <div class="chart-canvas"><canvas id="chartPrecipDaily-australia"></canvas></div>
            </div>
            <div class="chart-cell">
              <p class="chart-title">Evapotranspiration</p>
              <p class="chart-sub">Daily ET0 FAO Penman-Monteith (mm)</p>
              <div class="chart-canvas"><canvas id="chartEt0Daily-australia"></canvas></div>
            </div>
            <div class="chart-cell">
              <p class="chart-title">Hourly precip + PoP</p>
              <p class="chart-sub">Past 24h + next 24h (mm, %)</p>
              <div class="chart-canvas"><canvas id="chartPrecipHourly-australia"></canvas></div>
            </div>
            <div class="chart-cell">
              <p class="chart-title">Soil moisture</p>
              <p class="chart-sub">Root-zone average (0-27cm, m³/m³)</p>
              <div class="chart-canvas"><canvas id="chartSoil-australia"></canvas></div>
            </div>
          </div>

          <div class="chart-cell" style="margin-top:1px;background:var(--paper)!important;border-top:1px solid var(--rule);padding:18px 20px">
            <p class="chart-title">10-day temperature forecast</p>
            <p class="chart-sub">Daily min/max range with mean (°F)</p>
            <div class="chart-canvas" style="height:280px"><canvas id="chartTemp10d-australia"></canvas></div>
          </div>
        </div></div>
      </div>
      <div>
        <div class="section-h">
          <span class="h-eyebrow">SECTION 04 · CONTEXT</span>
          <h2>Australia cotton context</h2>
        </div>
        <div class="card"><div class="card-body" id="context-australia">__AUSTRALIA_CONTEXT__</div></div>
      </div>
    </div>
  </div>

  <!-- ========== TURKEY TAB ========== -->
  <div id="tab-turkey" class="tab-content">

    <div class="section-bar">
      <span>TURKEY COTTON FOCUS · __N_TR__ SITES</span>
      <span>7-DAY RETROSPECTIVE · 10-DAY FORECAST · ET0 + SOIL MOISTURE</span>
    </div>

    <div class="kpi-grid">
      <div class="kpi rain-kpi"><p class="kpi-label">Wettest last 7 days</p>__TURKEY_WETTEST__</div>
      <div class="kpi dry-kpi"><p class="kpi-label">Driest last 7 days</p>__TURKEY_DRIEST__</div>
      <div class="kpi rain-kpi"><p class="kpi-label">Most rain expected 7d</p>__TURKEY_FORECAST__</div>
      <div class="kpi balance-kpi"><p class="kpi-label">Avg water balance (3d)</p>__TURKEY_BALANCE__</div>
      <div class="kpi"><p class="kpi-label">Top-tier producers</p>
        <span class="metric-value">__TURKEY_TOP_COUNT__</span>
        <span class="metric-context">of __N_TR__ sites</span></div>
      <div class="kpi"><p class="kpi-label">States/regions</p>
        <span class="metric-value">__TURKEY_STATE_COUNT__</span>
        <span class="metric-context">covered</span></div>
    </div>

    <div class="section-h">
      <span class="h-eyebrow">SECTION 01 · GEOSPATIAL</span>
      <h2>Turkey cotton map</h2>
    </div>
    <div class="card map-wrap">
      <div class="zoom-pills" id="zoomPills-turkey"></div>
      <div class="metric-pills" id="metricPills-turkey">
        <button class="metric-pill active" data-metric="past3_precip">Rain 3d</button>
        <button class="metric-pill" data-metric="past7_precip">Rain 7d</button>
        <button class="metric-pill" data-metric="forecast24_precip">Rain 24h fcst</button>
        <button class="metric-pill" data-metric="forecast7d_precip">Rain 7d fcst</button>
        <button class="metric-pill" data-metric="forecast24_pop">PoP 24h</button>
        <button class="metric-pill" data-metric="balance3d">Water balance</button>
        <button class="metric-pill" data-metric="soil">Soil moisture</button>
        <button class="metric-pill" data-metric="temp">Temp</button>
      </div>
      <div id="map-turkey" class="weather-map"></div>
      <div class="map-legend">
        <span class="legend-title" id="legendTitle-turkey">Rainfall past 3 days (mm)</span>
        <div style="flex:1;min-width:240px">
          <div class="legend-scale" id="legendScale-turkey"></div>
          <div class="legend-stops" id="legendStops-turkey"></div>
        </div>
      </div>
    </div>

    <div class="section-h">
      <span class="h-eyebrow">SECTION 02 · SITE-BY-SITE</span>
      <h2>Turkey sites — 7-day retrospective and forecast</h2>
    </div>
    <div class="card">
      <div class="table-filter" id="tableFilter-turkey"></div>
      <table class="data-table">
        <thead><tr>
          <th>Site</th>
          <th class="num">Temp °F</th>
          <th class="num">Hum %</th>
          <th class="num">Rain J-3</th>
          <th class="num">Rain J-2</th>
          <th class="num">Rain J-1</th>
          <th class="num">Total 3d</th>
          <th class="num">Total 7d</th>
          <th class="num">ET0 3d</th>
          <th class="num">Balance</th>
          <th class="num">Soil</th>
          <th class="num">PoP 24h</th>
          <th class="num">Rain 24h fcst</th>
          <th class="num">Rain 7d fcst</th>
        </tr></thead>
        <tbody id="tableBody-turkey">__TURKEY_TABLE_ROWS__</tbody>
      </table>
    </div>

    <div class="layout">
      <div>
        <div class="section-h">
          <span class="h-eyebrow">SECTION 03 · SITE DETAIL</span>
          <h2>7-day retrospective + 10-day forecast for a selected site</h2>
        </div>
        <div class="card"><div class="card-body">
          <div class="city-selector">
            <label>Focus site</label>
            <select id="select-turkey"></select>
            <span class="obs-meta" id="obsMeta-turkey">—</span>
          </div>

          <div class="cotton-summary">
            <p class="cotton-summary-title">Water balance — past + forecast</p>
            <div class="cotton-stats" id="stats-turkey"></div>
          </div>

          <div class="charts-grid" style="margin-top:18px">
            <div class="chart-cell">
              <p class="chart-title">Precipitation</p>
              <p class="chart-sub">Daily — past 7 + 10-day forecast (mm)</p>
              <div class="chart-canvas"><canvas id="chartPrecipDaily-turkey"></canvas></div>
            </div>
            <div class="chart-cell">
              <p class="chart-title">Evapotranspiration</p>
              <p class="chart-sub">Daily ET0 FAO Penman-Monteith (mm)</p>
              <div class="chart-canvas"><canvas id="chartEt0Daily-turkey"></canvas></div>
            </div>
            <div class="chart-cell">
              <p class="chart-title">Hourly precip + PoP</p>
              <p class="chart-sub">Past 24h + next 24h (mm, %)</p>
              <div class="chart-canvas"><canvas id="chartPrecipHourly-turkey"></canvas></div>
            </div>
            <div class="chart-cell">
              <p class="chart-title">Soil moisture</p>
              <p class="chart-sub">Root-zone average (0-27cm, m³/m³)</p>
              <div class="chart-canvas"><canvas id="chartSoil-turkey"></canvas></div>
            </div>
          </div>

          <div class="chart-cell" style="margin-top:1px;background:var(--paper)!important;border-top:1px solid var(--rule);padding:18px 20px">
            <p class="chart-title">10-day temperature forecast</p>
            <p class="chart-sub">Daily min/max range with mean (°F)</p>
            <div class="chart-canvas" style="height:280px"><canvas id="chartTemp10d-turkey"></canvas></div>
          </div>
        </div></div>
      </div>
      <div>
        <div class="section-h">
          <span class="h-eyebrow">SECTION 04 · CONTEXT</span>
          <h2>Turkey cotton context</h2>
        </div>
        <div class="card"><div class="card-body" id="context-turkey">__TURKEY_CONTEXT__</div></div>
      </div>
    </div>
  </div>
  <div class="footer">
    <div>Data: <strong>NOAA / NWS</strong> (forecasts &amp; alerts), <strong>Open-Meteo</strong> (ET0 FAO + soil moisture). Maps: <strong>OpenStreetMap</strong> / CARTO.</div>
    <div>Dashboard auto-generated __GENERATED__</div>
  </div>
</div>

<script id="payload" type="application/json">__PAYLOAD__</script>
<script>
const DATA = JSON.parse(document.getElementById('payload').textContent);

// Utilities
function mix(a,b,t){
  const ah=a.replace('#',''),bh=b.replace('#','');
  const ar=parseInt(ah.substr(0,2),16),ag=parseInt(ah.substr(2,2),16),ab=parseInt(ah.substr(4,2),16);
  const br=parseInt(bh.substr(0,2),16),bg=parseInt(bh.substr(2,2),16),bb=parseInt(bh.substr(4,2),16);
  const r=Math.round(ar+(br-ar)*t).toString(16).padStart(2,'0');
  const g=Math.round(ag+(bg-ag)*t).toString(16).padStart(2,'0');
  const c=Math.round(ab+(bb-ab)*t).toString(16).padStart(2,'0');
  return '#'+r+g+c;
}
function interpolate(stops,v){
  if(v==null||isNaN(v))return '#475569';
  if(v<=stops[0][0])return stops[0][1];
  if(v>=stops[stops.length-1][0])return stops[stops.length-1][1];
  for(let i=0;i<stops.length-1;i++){
    if(v>=stops[i][0]&&v<=stops[i+1][0]){
      const t=(v-stops[i][0])/(stops[i+1][0]-stops[i][0]);
      return mix(stops[i][1],stops[i+1][1],t);
    }
  }
  return stops[stops.length-1][1];
}

const TEMP_STOPS=[[0,'#1e3a8a'],[32,'#3b82f6'],[50,'#06b6d4'],[65,'#22c55e'],
                  [75,'#eab308'],[85,'#f97316'],[95,'#dc2626'],[110,'#7f1d1d']];
const WIND_STOPS=[[0,'#dcfce7'],[10,'#86efac'],[20,'#fbbf24'],[30,'#f97316'],[40,'#dc2626'],[60,'#7f1d1d']];
const PRECIP1_STOPS=[[0,'#f8fafc'],[0.01,'#bae6fd'],[0.05,'#38bdf8'],[0.15,'#0284c7'],[0.5,'#312e81']];
const PRECIP7_STOPS=[[0,'#f8fafc'],[10,'#bae6fd'],[25,'#38bdf8'],[50,'#0284c7'],[100,'#312e81']];
const PRECIP3D_STOPS=[[0,'#fef3c7'],[2,'#bae6fd'],[10,'#38bdf8'],[25,'#0284c7'],[50,'#1e3a8a'],[100,'#312e81']];
const PRECIP7D_FCST_STOPS=[[0,'#fef3c7'],[5,'#bae6fd'],[15,'#38bdf8'],[40,'#0284c7'],[75,'#1e3a8a'],[150,'#312e81']];
const PRECIP24_STOPS=[[0,'#f1f5f9'],[0.5,'#bae6fd'],[2,'#38bdf8'],[5,'#0284c7'],[15,'#1e3a8a'],[30,'#312e81']];
const POP_STOPS=[[0,'#f1f5f9'],[20,'#bae6fd'],[40,'#7dd3fc'],[60,'#38bdf8'],[80,'#0284c7'],[100,'#1e3a8a']];
const BALANCE_STOPS=[[-30,'#7f1d1d'],[-15,'#dc2626'],[-5,'#f97316'],[0,'#fbbf24'],[10,'#86efac'],[25,'#22c55e'],[50,'#0c4a6e']];
const SOIL_STOPS=[[0.05,'#92400e'],[0.12,'#d97706'],[0.20,'#eab308'],[0.30,'#22c55e'],[0.45,'#0ea5e9']];

function parseWind(s){
  if(s==null)return null;
  if(typeof s==='number')return s;
  const m=String(s).match(/(\d+(?:\.\d+)?)/g);
  if(!m)return null;
  if(m.length===1)return +m[0];
  return (+m[0]+ +m[1])/2;
}
function fmtHourLabel(iso){
  if(!iso)return '';
  const d=new Date(iso);
  if(isNaN(d))return '';
  const h=d.getHours();
  return (((h+11)%12)+1)+(h>=12?'p':'a');
}

document.querySelectorAll('.tab-btn').forEach(btn=>{
  btn.addEventListener('click',()=>{
    document.querySelectorAll('.tab-btn').forEach(b=>b.classList.remove('active'));
    document.querySelectorAll('.tab-content').forEach(c=>c.classList.remove('active'));
    btn.classList.add('active');
    document.getElementById('tab-'+btn.dataset.tab).classList.add('active');
    setTimeout(()=>{
      if(btn.dataset.tab==='noaa' && window.mapNoaa) window.mapNoaa.invalidateSize();
      if(btn.dataset.tab==='cotton' && window.mapCotton) window.mapCotton.invalidateSize();
      if(btn.dataset.tab==='brazil' && window.mapBrazil) window.mapBrazil.invalidateSize();
      ['china','india','australia','turkey'].forEach(k=>{
        if(btn.dataset.tab===k && window['map_'+k]) window['map_'+k].invalidateSize();
      });
    },50);
  });
});

// =============================================================================
//                                 NOAA TAB
// =============================================================================
const METRIC_CFG_NOAA={
  temp:    {label:'Temperature (°F)',          stops:TEMP_STOPS,   domain:[0,110], unit:'°F',  getter:c=>c.current?.temp_f},
  wind:    {label:'Observed wind (mph)',       stops:WIND_STOPS,   domain:[0,60],  unit:' mph',getter:c=>c.current?.wind_mph},
  precip1h:{label:'Current precip 1h (in)',    stops:PRECIP1_STOPS,domain:[0,0.5], unit:' in', getter:c=>c.current?.precip_in_1h},
  precip7: {label:'7-day forecast precip (mm)',stops:PRECIP7_STOPS,domain:[0,100], unit:' mm', getter:c=>c.open_meteo?.forecast_7d_precip_mm},
};

window.mapNoaa=L.map('mapNoaa',{zoomControl:true,scrollWheelZoom:true}).setView([31,-99.5],6);
L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png',{
  attribution:'&copy; OpenStreetMap, &copy; CARTO',subdomains:'abcd',maxZoom:19
}).addTo(window.mapNoaa);
const layerNoaa=L.layerGroup().addTo(window.mapNoaa);
let currentMetricNoaa='temp',currentZoomNoaa='texas';

function fmtN(v,m){
  if(v==null||isNaN(v))return '—';
  if(m==='precip1h')return Number(v).toFixed(2);
  if(m==='precip7') return Number(v).toFixed(1);
  return Math.round(v);
}
function buildMarkersNoaa(){
  layerNoaa.clearLayers();
  const cfg=METRIC_CFG_NOAA[currentMetricNoaa];
  const cities=currentZoomNoaa==='usa'?[...DATA.noaa.texas,...DATA.noaa.us]:DATA.noaa.texas;
  cities.forEach(c=>{
    const v=cfg.getter(c);
    const muted=(v==null||isNaN(v));
    const color=muted?'#475569':interpolate(cfg.stops,v);
    const icon=L.divIcon({
      html:`<div class="weather-marker${muted?' muted':''}" style="background:${color};">${fmtN(v,currentMetricNoaa)}</div>`,
      className:'',iconSize:[60,28],iconAnchor:[30,14]
    });
    const cur=c.current||{};
    const popup=`<p class="popup-title">${c.name}</p>`
      +`<p class="popup-cond">${cur.text_description||'—'}</p>`
      +`<div class="popup-row"><span class="k">Temperature</span><span class="v">${cur.temp_f??'—'}°F</span></div>`
      +`<div class="popup-row"><span class="k">Wind</span><span class="v">${cur.wind_mph??'—'} mph</span></div>`
      +`<div class="popup-row"><span class="k">Current precip 1h</span><span class="v">${cur.precip_in_1h??'—'} in</span></div>`
      +`<div class="popup-row"><span class="k">Forecast 7d precip</span><span class="v">${c.open_meteo?.forecast_7d_precip_mm??'—'} mm</span></div>`;
    L.marker([c.lat,c.lon],{icon}).bindPopup(popup).addTo(layerNoaa);
  });
  updateLegendNoaa();
}
function updateLegendNoaa(){
  const cfg=METRIC_CFG_NOAA[currentMetricNoaa];
  document.getElementById('legendTitleNoaa').textContent=cfg.label;
  const scale=document.getElementById('legendScaleNoaa');
  const stops=document.getElementById('legendStopsNoaa');
  scale.innerHTML='';stops.innerHTML='';
  for(let i=0;i<32;i++){
    const t=i/31,v=cfg.domain[0]+t*(cfg.domain[1]-cfg.domain[0]);
    const s=document.createElement('span');s.style.background=interpolate(cfg.stops,v);scale.appendChild(s);
  }
  for(let i=0;i<=5;i++){
    const t=i/5,v=cfg.domain[0]+t*(cfg.domain[1]-cfg.domain[0]);
    const s=document.createElement('span');
    s.textContent=currentMetricNoaa==='precip1h'?v.toFixed(2)+cfg.unit:Math.round(v)+cfg.unit;
    stops.appendChild(s);
  }
}
const ZOOM_NOAA={west:{c:[32,-102.5],z:7},texas:{c:[31,-99.5],z:6},usa:{c:[39.5,-98.5],z:4}};
document.querySelectorAll('#zoomPillsNoaa .zoom-pill').forEach(b=>b.addEventListener('click',()=>{
  document.querySelectorAll('#zoomPillsNoaa .zoom-pill').forEach(x=>x.classList.remove('active'));
  b.classList.add('active');currentZoomNoaa=b.dataset.zoom;
  const v=ZOOM_NOAA[currentZoomNoaa];window.mapNoaa.flyTo(v.c,v.z,{duration:.9});
  buildMarkersNoaa();
}));
document.querySelectorAll('#metricPillsNoaa .metric-pill').forEach(b=>b.addEventListener('click',()=>{
  document.querySelectorAll('#metricPillsNoaa .metric-pill').forEach(x=>x.classList.remove('active'));
  b.classList.add('active');currentMetricNoaa=b.dataset.metric;buildMarkersNoaa();
}));
buildMarkersNoaa();

// =============================================================================
//                              COTTON BELT TAB
// =============================================================================
const METRIC_CFG_COTTON={
  past3_precip:     {label:'Rainfall past 3 days (mm)',  stops:PRECIP3D_STOPS, domain:[0,100], unit:'mm',  getter:c=>c.past3_precip_mm},
  past7_precip:     {label:'Rainfall past 7 days (mm)',  stops:PRECIP7D_FCST_STOPS,domain:[0,150],unit:'mm',getter:c=>c.past7_precip_mm},
  forecast24_precip:{label:'Rain forecast next 24h (mm)',stops:PRECIP24_STOPS, domain:[0,30],  unit:'mm',  getter:c=>c.forecast_24h?.total_mm},
  forecast7d_precip:{label:'Rain forecast next 7 days (mm)',stops:PRECIP7D_FCST_STOPS,domain:[0,150],unit:'mm',getter:c=>c.forecast_7d_precip_mm},
  forecast24_pop:   {label:'Max PoP next 24h (%)',       stops:POP_STOPS,      domain:[0,100], unit:'%',   getter:c=>c.forecast_24h?.max_prob},
  balance3d:        {label:'Water balance 3d (Precip-ET0, mm)',stops:BALANCE_STOPS,domain:[-30,50],unit:'mm',getter:c=>c.past3_balance_mm},
  soil:             {label:'Soil moisture (m³/m³)',       stops:SOIL_STOPS,    domain:[0.05,0.45],unit:'',getter:c=>c.current?.soil_root_zone},
  temp:             {label:'Temperature (°F)',           stops:TEMP_STOPS,    domain:[0,110], unit:'°F', getter:c=>c.current?.temp_f},
};

window.mapCotton=L.map('mapCotton',{zoomControl:true,scrollWheelZoom:true}).setView([32.5,-101.5],6);
L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png',{
  attribution:'&copy; OpenStreetMap, &copy; CARTO',subdomains:'abcd',maxZoom:19
}).addTo(window.mapCotton);
const layerCotton=L.layerGroup().addTo(window.mapCotton);
let currentMetricCotton='past3_precip',currentZoomCotton='all';

function fmtC(v,m){
  if(v==null||isNaN(v))return '—';
  if(m==='soil')return Number(v).toFixed(2);
  if(m==='past3_precip'||m==='past7_precip'||m==='forecast24_precip'||m==='forecast7d_precip'||m==='balance3d')return Math.round(v);
  return Math.round(v);
}
function buildMarkersCotton(){
  layerCotton.clearLayers();
  const cfg=METRIC_CFG_COTTON[currentMetricCotton];
  let counties=DATA.cotton.counties;
  if(currentZoomCotton==='hp')counties=counties.filter(c=>c.region==='HP');
  if(currentZoomCotton==='top12')counties=counties.filter(c=>c.top12);
  if(currentZoomCotton==='cb')counties=counties.filter(c=>c.region==='CB'||c.region==='LRGV');
  counties.forEach(c=>{
    const v=cfg.getter(c);
    const muted=(v==null||isNaN(v));
    const color=muted?'#475569':interpolate(cfg.stops,v);
    const cls='weather-marker'+(c.top12?' top12':'')+(muted?' muted':'');
    const icon=L.divIcon({
      html:`<div class="${cls}" style="background:${color};">${fmtC(v,currentMetricCotton)}</div>`,
      className:'',iconSize:[c.top12?64:56,c.top12?30:26],iconAnchor:[c.top12?32:28,c.top12?15:13]
    });
    const cur=c.current||{};
    const fc=c.forecast_24h||{};
    const popup=`<p class="popup-title">${c.top12?'⭐ ':''}${c.county} County</p>`
      +`<p class="popup-cond">Seat: <strong>${c.seat}</strong> · ${c.region}</p>`
      +`<div class="popup-row"><span class="k">Temperature</span><span class="v">${cur.temp_f!=null?Math.round(cur.temp_f)+'°F':'—'}</span></div>`
      +`<div class="popup-row"><span class="k">Humidity</span><span class="v">${cur.humidity!=null?Math.round(cur.humidity)+'%':'—'}</span></div>`
      +`<div class="popup-row"><span class="k">Wind</span><span class="v">${cur.wind_mph!=null?Math.round(cur.wind_mph)+' mph':'—'}</span></div>`
      +`<div class="popup-row" style="border-top:1px solid #334155;padding-top:6px;margin-top:6px"><span class="k"><strong>Rain past 3d</strong></span><span class="v"><strong>${c.past3_precip_mm??'—'} mm</strong></span></div>`
      +`<div class="popup-row"><span class="k"><strong>Rain past 7d</strong></span><span class="v"><strong>${c.past7_precip_mm??'—'} mm</strong></span></div>`
      +`<div class="popup-row"><span class="k">ET0 past 3d</span><span class="v">${c.past3_et0_mm??'—'} mm</span></div>`
      +`<div class="popup-row"><span class="k">Water balance</span><span class="v" style="color:${(c.past3_balance_mm??0)<0?'#ef4444':'#22c55e'}">${c.past3_balance_mm??'—'} mm</span></div>`
      +`<div class="popup-row"><span class="k">Soil moisture</span><span class="v">${cur.soil_root_zone!=null?cur.soil_root_zone.toFixed(2):'—'}</span></div>`
      +`<div class="popup-row" style="border-top:1px solid #334155;padding-top:6px;margin-top:6px"><span class="k"><strong>Max PoP 24h</strong></span><span class="v"><strong>${fc.max_prob??'—'}%</strong></span></div>`
      +`<div class="popup-row"><span class="k">Rain forecast 24h</span><span class="v">${fc.total_mm??'—'} mm</span></div>`
      +`<div class="popup-row"><span class="k"><strong>Rain forecast 7d</strong></span><span class="v"><strong>${c.forecast_7d_precip_mm??'—'} mm</strong></span></div>`;
    L.marker([c.lat,c.lon],{icon}).bindPopup(popup).addTo(layerCotton);
  });
  updateLegendCotton();
}
function updateLegendCotton(){
  const cfg=METRIC_CFG_COTTON[currentMetricCotton];
  document.getElementById('legendTitleCotton').textContent=cfg.label;
  const scale=document.getElementById('legendScaleCotton');
  const stops=document.getElementById('legendStopsCotton');
  scale.innerHTML='';stops.innerHTML='';
  for(let i=0;i<32;i++){
    const t=i/31,v=cfg.domain[0]+t*(cfg.domain[1]-cfg.domain[0]);
    const s=document.createElement('span');s.style.background=interpolate(cfg.stops,v);scale.appendChild(s);
  }
  for(let i=0;i<=5;i++){
    const t=i/5,v=cfg.domain[0]+t*(cfg.domain[1]-cfg.domain[0]);
    const s=document.createElement('span');
    s.textContent=currentMetricCotton==='soil'?v.toFixed(2):Math.round(v)+cfg.unit;
    stops.appendChild(s);
  }
}
const ZOOM_COTTON={all:{c:[32.5,-101.5],z:6},hp:{c:[34,-101.5],z:7},top12:{c:[33,-100],z:6},cb:{c:[27.5,-97.5],z:8}};
document.querySelectorAll('#zoomPillsCotton .zoom-pill').forEach(b=>b.addEventListener('click',()=>{
  document.querySelectorAll('#zoomPillsCotton .zoom-pill').forEach(x=>x.classList.remove('active'));
  b.classList.add('active');currentZoomCotton=b.dataset.zoom;
  const v=ZOOM_COTTON[currentZoomCotton];window.mapCotton.flyTo(v.c,v.z,{duration:.9});
  buildMarkersCotton();
}));
document.querySelectorAll('#metricPillsCotton .metric-pill').forEach(b=>b.addEventListener('click',()=>{
  document.querySelectorAll('#metricPillsCotton .metric-pill').forEach(x=>x.classList.remove('active'));
  b.classList.add('active');currentMetricCotton=b.dataset.metric;buildMarkersCotton();
}));
buildMarkersCotton();

// County table filter
document.querySelectorAll('.table-filter button').forEach(b=>b.addEventListener('click',()=>{
  document.querySelectorAll('.table-filter button').forEach(x=>x.classList.remove('active'));
  b.classList.add('active');
  const f=b.dataset.filter;
  document.querySelectorAll('#cottonTableBody tr').forEach(tr=>{
    if(f==='all')tr.style.display='';
    else if(f==='top12')tr.style.display=tr.classList.contains('top12')?'':'none';
    else tr.style.display=tr.dataset.region===f?'':'none';
  });
}));

// County detail
const countySelect=document.getElementById('countySelect');
DATA.cotton.counties.slice().sort((a,b)=>(b.top12?1:0)-(a.top12?1:0)||a.county.localeCompare(b.county)).forEach(c=>{
  const o=document.createElement('option');o.value=c.county;
  o.textContent=(c.top12?'⭐ ':'')+c.county+' · '+c.seat;
  countySelect.appendChild(o);
});
countySelect.value='Lubbock';

const chartsC={};
function mkChart(id,label,color,yLabel,type){
  const ctx=document.getElementById(id).getContext('2d');
  chartsC[id]=new Chart(ctx,{
    type:type||'line',
    data:{labels:[],datasets:[{label,data:[],borderColor:color,backgroundColor:type==='bar'?color+'88':color+'22',
          fill:true,tension:.35,borderWidth:2,pointRadius:0,pointHoverRadius:4}]},
    options:{responsive:true,maintainAspectRatio:false,
      plugins:{legend:{display:true,labels:{color:'#a9b7cc',font:{family:'JetBrains Mono',size:10}}},
               tooltip:{backgroundColor:'#0b1530'}},
      scales:{x:{ticks:{font:{family:'JetBrains Mono',size:9},maxRotation:0,autoSkip:true,maxTicksLimit:8,color:'#a9b7cc'},grid:{display:false}},
              y:{title:{display:true,text:yLabel,color:'#a9b7cc'},ticks:{color:'#a9b7cc'},grid:{color:'#1e293b'}}}}
  });
}
mkChart('chartPrecipDaily','Precipitation (mm)','#38bdf8','mm','bar');
mkChart('chartEt0Daily','ET0 (mm)','#f59e0b','mm','bar');
mkChart('chartPrecipHourly','Hourly precip (mm)','#38bdf8','mm');
mkChart('chartSoil','Root-zone soil moisture','#22c55e','m³/m³');

// 10-day temperature forecast - custom build (not via mkChart)
// because it needs a 3-dataset structure: min, max, mean (range band style)
const ctxTemp10 = document.getElementById('chartTemp10d').getContext('2d');
const chartTemp10d = new Chart(ctxTemp10, {
  type: 'line',
  data: {
    labels: [],
    datasets: [
      {
        label: 'Max',
        data: [],
        borderColor: '#ef4444',
        backgroundColor: '#ef444433',
        borderWidth: 2.5,
        pointRadius: 3,
        pointBackgroundColor: '#ef4444',
        tension: 0.35,
        fill: '+1',  // Fill area down to the next dataset (Min)
      },
      {
        label: 'Min',
        data: [],
        borderColor: '#3b82f6',
        backgroundColor: 'transparent',
        borderWidth: 2.5,
        pointRadius: 3,
        pointBackgroundColor: '#3b82f6',
        tension: 0.35,
        fill: false,
      },
      {
        label: 'Mean',
        data: [],
        borderColor: '#fbbf24',
        borderDash: [4, 4],
        borderWidth: 2,
        pointRadius: 2,
        pointBackgroundColor: '#fbbf24',
        tension: 0.35,
        fill: false,
      }
    ]
  },
  options: {
    responsive: true,
    maintainAspectRatio: false,
    interaction: { mode: 'index', intersect: false },
    plugins: {
      legend: {
        display: true,
        position: 'top',
        labels: { color: '#a9b7cc', font: { family: 'JetBrains Mono', size: 11 }, usePointStyle: true }
      },
      tooltip: {
        backgroundColor: '#0b1530',
        callbacks: {
          label: function(ctx) {
            return ctx.dataset.label + ': ' + Math.round(ctx.parsed.y) + '°F';
          }
        }
      }
    },
    scales: {
      x: {
        ticks: { font: { family: 'JetBrains Mono', size: 10 }, color: '#a9b7cc' },
        grid: { color: '#1e293b' }
      },
      y: {
        title: { display: true, text: 'Temperature (°F)', color: '#a9b7cc' },
        ticks: { color: '#a9b7cc' },
        grid: { color: '#1e293b' }
      }
    }
  }
});
chartsC['chartTemp10d'] = chartTemp10d;

// ===== Climatology charts =====
// Year color palette: 5 history years in muted shades + current year in bold gold
const CURRENT_YEAR_CLIMO = new Date().getFullYear();
const YEAR_COLORS = {
  // Past years get cool / neutral colors; current year is bold gold
  // Computed below based on selected county's available years
};
function climoYearColor(year, isCurrent){
  if(isCurrent) return '#fbbf24';
  const palette = ['#64748b','#94a3b8','#7dd3fc','#22d3ee','#a78bfa','#fb923c','#22c55e'];
  // Map years to palette index deterministically
  return palette[(year - 2020) % palette.length];
}

function makeClimoChart(canvasId, yLabel) {
  const ctx = document.getElementById(canvasId).getContext('2d');
  return new Chart(ctx, {
    type:'line',
    data:{labels:[], datasets:[]},
    options:{
      responsive:true, maintainAspectRatio:false,
      interaction:{mode:'index', intersect:false},
      plugins:{
        legend:{display:true, position:'top',
                labels:{color:'#a9b7cc', font:{family:'JetBrains Mono', size:10},
                        usePointStyle:true, padding:14}},
        tooltip:{backgroundColor:'#0b1530'}
      },
      scales:{
        x:{ticks:{font:{family:'JetBrains Mono', size:9}, color:'#a9b7cc',
                  maxRotation:0, autoSkip:true, maxTicksLimit:12},
           grid:{color:'#1e293b', drawTicks:false},
           title:{display:true, text:'Date (month-day)', color:'#a9b7cc',
                  font:{family:'JetBrains Mono', size:10}}},
        y:{title:{display:true, text:yLabel, color:'#a9b7cc'},
           ticks:{color:'#a9b7cc'},
           grid:{color:'#1e293b'}}
      }
    }
  });
}

const chartClimoRain = makeClimoChart('chartClimoRain', 'Cumulative precipitation (mm)');
const chartClimoSoil = makeClimoChart('chartClimoSoil', 'Soil moisture (m³/m³)');

let _climoPeriod = 'calendar';  // 'calendar' or 'growing'

// Convert day-of-year to a "MM-DD" label
function doyToLabel(doy){
  // Use 2025 (non-leap proxy) as the reference calendar to map doy -> MM-DD
  const d = new Date(2025, 0, doy);
  const mm = String(d.getMonth() + 1).padStart(2, '0');
  const dd = String(d.getDate()).padStart(2, '0');
  return `${mm}-${dd}`;
}

function renderClimatology(countyName){
  const c = DATA.cotton.counties.find(x => x.county === countyName);
  const wrap = document.getElementById('climatologyWrap');
  if (!c || !c.top12 || !c.climatology || !Object.keys(c.climatology).length) {
    wrap.style.display = 'none';
    return;
  }
  wrap.style.display = 'block';

  // X axis: day-of-year range based on period toggle
  const doyStart = _climoPeriod === 'growing' ? 91 : 1;       // ~Apr 1 (DOY 91)
  const doyEnd   = _climoPeriod === 'growing' ? 334 : 366;    // ~Nov 30 (DOY 334)
  const labels = [];
  for (let d = doyStart; d <= doyEnd; d += 1) labels.push(doyToLabel(d));

  const years = Object.keys(c.climatology).map(Number).sort();
  // Build datasets for both charts
  const rainSets = [];
  const soilSets = [];

  // If growing-season view, we need to RE-cumulate precipitation from the first growing-season day
  years.forEach(year => {
    const days = c.climatology[year] || [];
    const isCurrent = (year === CURRENT_YEAR_CLIMO);
    const color = climoYearColor(year, isCurrent);

    // Build full arrays of length labels.length (sparse: undefined where no data)
    const rainArr = new Array(labels.length).fill(null);
    const soilArr = new Array(labels.length).fill(null);

    if (_climoPeriod === 'calendar') {
      // Just use the pre-computed cumulative values
      days.forEach(d => {
        const idx = d.doy - doyStart;
        if (idx >= 0 && idx < labels.length) {
          rainArr[idx] = d.precip_cumulative_mm;
          if (d.soil != null) soilArr[idx] = d.soil;
        }
      });
    } else {
      // Re-cumulate precip starting from doyStart for the growing-season view
      let cum = 0;
      days.forEach(d => {
        if (d.doy < doyStart || d.doy > doyEnd) return;
        if (typeof d.precip_mm === 'number') cum += d.precip_mm;
        const idx = d.doy - doyStart;
        rainArr[idx] = Math.round(cum * 100) / 100;
        if (d.soil != null) soilArr[idx] = d.soil;
      });
    }

    const baseSet = {
      label: String(year),
      borderColor: color,
      backgroundColor: color + (isCurrent ? '33' : '11'),
      borderWidth: isCurrent ? 3.5 : 1.5,
      pointRadius: 0,
      pointHoverRadius: isCurrent ? 4 : 3,
      tension: 0.2,
      fill: false,
      spanGaps: true,
    };
    rainSets.push({...baseSet, data: rainArr});
    soilSets.push({...baseSet, data: soilArr});
  });

  chartClimoRain.data.labels = labels;
  chartClimoRain.data.datasets = rainSets;
  chartClimoRain.update();

  chartClimoSoil.data.labels = labels;
  chartClimoSoil.data.datasets = soilSets;
  chartClimoSoil.update();

  // Update subtitle text based on toggle
  document.getElementById('climoSub').textContent =
    _climoPeriod === 'growing'
      ? 'Daily accumulated rainfall, growing season (Apr–Nov), by year (mm)'
      : 'Daily accumulated rainfall, calendar year, by year (mm)';
}

// Wire toggle buttons
document.querySelectorAll('.climo-period').forEach(btn => {
  btn.addEventListener('click', () => {
    document.querySelectorAll('.climo-period').forEach(b => b.classList.remove('active'));
    btn.classList.add('active');
    _climoPeriod = btn.dataset.period;
    renderClimatology(countySelect.value);
  });
});

// Add PoP secondary dataset to hourly chart
chartsC.chartPrecipHourly.data.datasets.push({
  label:'PoP (%)',data:[],borderColor:'#a78bfa',backgroundColor:'#a78bfa22',
  yAxisID:'y2',borderWidth:2,pointRadius:0,fill:false,tension:.35
});
chartsC.chartPrecipHourly.options.scales.y2={
  type:'linear',position:'right',title:{display:true,text:'PoP %',color:'#a78bfa'},
  ticks:{color:'#a78bfa'},grid:{display:false},min:0,max:100
};
chartsC.chartPrecipHourly.update();

function renderCountyDetail(county){
  const c=DATA.cotton.counties.find(x=>x.county===county);
  if(!c)return;

  // Daily precip + ET0 bars: past 3 + forecast 2 = 5 days
  const daily=c.daily||[];
  const labels=daily.map(d=>d.date.substring(5));
  chartsC.chartPrecipDaily.data.labels=labels;
  chartsC.chartPrecipDaily.data.datasets[0].data=daily.map(d=>d.precip_mm);
  // Color forecast days differently
  chartsC.chartPrecipDaily.data.datasets[0].backgroundColor=daily.map(d=>
    d.period==='past'?'#38bdf8cc':'#7dd3fcaa');
  chartsC.chartPrecipDaily.update();

  chartsC.chartEt0Daily.data.labels=labels;
  chartsC.chartEt0Daily.data.datasets[0].data=daily.map(d=>d.et0_mm);
  chartsC.chartEt0Daily.data.datasets[0].backgroundColor=daily.map(d=>
    d.period==='past'?'#f59e0bcc':'#fcd34daa');
  chartsC.chartEt0Daily.update();

  // Hourly: past 24h observed + next 24h forecast
  const hourly=c.hourly||[];
  const hLabels=hourly.map(h=>fmtHourLabel(h.time));
  chartsC.chartPrecipHourly.data.labels=hLabels;
  chartsC.chartPrecipHourly.data.datasets[0].data=hourly.map(h=>h.precipitation);
  chartsC.chartPrecipHourly.data.datasets[1].data=hourly.map(h=>h.precipitation_probability);
  chartsC.chartPrecipHourly.update();

  // Soil moisture
  chartsC.chartSoil.data.labels=hLabels;
  chartsC.chartSoil.data.datasets[0].data=hourly.map(h=>h.soil_root_zone);
  chartsC.chartSoil.update();

  // Cotton summary stats
  const fc=c.forecast_24h||{};
  const stats=document.getElementById('cottonStats');
  const balCls=(c.past3_balance_mm??0)<0?'negative':'positive';
  stats.innerHTML=
     `<div class="cotton-stat"><span class="cotton-stat-value">${c.past3_precip_mm??'—'}mm</span><span class="cotton-stat-label">Rain past 3 days</span></div>`
    +`<div class="cotton-stat"><span class="cotton-stat-value">${c.past7_precip_mm??'—'}mm</span><span class="cotton-stat-label">Rain past 7 days</span></div>`
    +`<div class="cotton-stat"><span class="cotton-stat-value">${c.past3_et0_mm??'—'}mm</span><span class="cotton-stat-label">ET0 past 3 days</span></div>`
    +`<div class="cotton-stat"><span class="cotton-stat-value" style="color:${balCls==='negative'?'#fca5a5':'#86efac'}">${c.past3_balance_mm??'—'}mm</span><span class="cotton-stat-label">3d water balance</span></div>`
    +`<div class="cotton-stat"><span class="cotton-stat-value">${fc.max_prob??'—'}${fc.max_prob!=null?'%':''}</span><span class="cotton-stat-label">Peak PoP 24h</span></div>`
    +`<div class="cotton-stat"><span class="cotton-stat-value">${fc.total_mm??'—'}${fc.total_mm!=null?'mm':''}</span><span class="cotton-stat-label">Expected rain 24h</span></div>`
    +`<div class="cotton-stat"><span class="cotton-stat-value">${fc.hours_pop_gt_50??0}h</span><span class="cotton-stat-label">Hours PoP ≥50%</span></div>`
    +`<div class="cotton-stat"><span class="cotton-stat-value" style="color:#7dd3fc">${c.forecast_7d_precip_mm??'—'}${c.forecast_7d_precip_mm!=null?'mm':''}</span><span class="cotton-stat-label">Expected rain 7 days</span></div>`;

  document.getElementById('obsMetaCotton').textContent=
    `${c.county} County · seat ${c.seat} · ${c.region} · ${c.lat.toFixed(2)},${c.lon.toFixed(2)} · ${c.current?.time||'no recent obs'}`;

  // 10-day temperature forecast (min/max band + mean line)
  // Use ALL daily entries that are in the future + today, from Open-Meteo
  const tempDaily = (c.daily || []).filter(d => d.period === 'forecast');
  const tempLabels = tempDaily.map(d => {
    // Format as "Mon 06/02"
    const dt = new Date(d.date);
    const weekday = ['Sun','Mon','Tue','Wed','Thu','Fri','Sat'][dt.getUTCDay()];
    return weekday + ' ' + d.date.substring(5);
  });
  const tempMax = tempDaily.map(d => d.tmax_f);
  const tempMin = tempDaily.map(d => d.tmin_f);
  const tempMean = tempDaily.map(d =>
    (typeof d.tmax_f === 'number' && typeof d.tmin_f === 'number')
      ? Math.round(((d.tmax_f + d.tmin_f) / 2) * 10) / 10
      : null
  );
  chartsC.chartTemp10d.data.labels = tempLabels;
  chartsC.chartTemp10d.data.datasets[0].data = tempMax;
  chartsC.chartTemp10d.data.datasets[1].data = tempMin;
  chartsC.chartTemp10d.data.datasets[2].data = tempMean;
  chartsC.chartTemp10d.update();

  // 7-day NOAA forecast (only top 12 have it)
  const strip=document.getElementById('forecastStripCotton');strip.innerHTML='';
  if(c.noaa_forecast && c.noaa_forecast.length){
    c.noaa_forecast.slice(0,14).forEach(p=>{
      const div=document.createElement('div');
      div.className='forecast-day'+(p.is_day===false?' night':'');
      div.innerHTML=`<div class="forecast-name">${p.name||''}</div>`
        +`<div class="forecast-temp">${p.temp??'—'}°</div>`
        +`<div class="forecast-cond">${p.short||''}</div>`
        +((p.precip_prob!=null&&p.precip_prob>0)?`<div class="forecast-precip">${p.precip_prob}% precip</div>`:'')
        +`<div class="forecast-wind">${p.wind_dir||''} ${p.wind||''}</div>`;
      strip.appendChild(div);
    });
  }else{
    strip.innerHTML='<div style="padding:14px;color:var(--ink-soft);font-size:12px">NOAA 7-day forecast loaded only for top-12 USDA counties to keep load times reasonable.</div>';
  }
}
countySelect.addEventListener('change',()=>{
  renderCountyDetail(countySelect.value);
  renderClimatology(countySelect.value);
});
renderCountyDetail(countySelect.value);
renderClimatology(countySelect.value);

// =============================================================================
//                              BRAZIL TAB
// =============================================================================
const METRIC_CFG_BRAZIL = {
  past3_precip:     {label:'Rainfall past 3 days (mm)',  stops:PRECIP3D_STOPS, domain:[0,100], unit:'mm',  getter:c=>c.past3_precip_mm},
  past7_precip:     {label:'Rainfall past 7 days (mm)',  stops:PRECIP7D_FCST_STOPS,domain:[0,150],unit:'mm',getter:c=>c.past7_precip_mm},
  forecast24_precip:{label:'Rain forecast next 24h (mm)',stops:PRECIP24_STOPS, domain:[0,30],  unit:'mm',  getter:c=>c.forecast_24h?.total_mm},
  forecast7d_precip:{label:'Rain forecast next 7 days (mm)',stops:PRECIP7D_FCST_STOPS,domain:[0,150],unit:'mm',getter:c=>c.forecast_7d_precip_mm},
  forecast24_pop:   {label:'Max PoP next 24h (%)',       stops:POP_STOPS,      domain:[0,100], unit:'%',   getter:c=>c.forecast_24h?.max_prob},
  balance3d:        {label:'Water balance 3d (Precip-ET0, mm)',stops:BALANCE_STOPS,domain:[-30,50],unit:'mm',getter:c=>c.past3_balance_mm},
  soil:             {label:'Soil moisture (m³/m³)',       stops:SOIL_STOPS,    domain:[0.05,0.45],unit:'',getter:c=>c.current?.soil_root_zone},
  temp:             {label:'Temperature (°F)',           stops:TEMP_STOPS,    domain:[0,110], unit:'°F', getter:c=>c.current?.temp_f},
};

window.mapBrazil = L.map('mapBrazil',{zoomControl:true,scrollWheelZoom:true}).setView([-13,-50],5);
L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png',{
  attribution:'&copy; OpenStreetMap, &copy; CARTO',subdomains:'abcd',maxZoom:19
}).addTo(window.mapBrazil);
const layerBrazil = L.layerGroup().addTo(window.mapBrazil);
let currentMetricBrazil='past3_precip', currentZoomBrazil='all';

function fmtBR(v,m){
  if(v==null||isNaN(v))return '—';
  if(m==='soil')return Number(v).toFixed(2);
  return Math.round(v);
}
function buildMarkersBrazil(){
  layerBrazil.clearLayers();
  const cfg = METRIC_CFG_BRAZIL[currentMetricBrazil];
  let muns = DATA.brazil.municipalities;
  if(currentZoomBrazil==='mt') muns = muns.filter(c => c.state==='MT');
  if(currentZoomBrazil==='ba') muns = muns.filter(c => c.state==='BA');
  if(currentZoomBrazil==='top') muns = muns.filter(c => c.top_tier);
  if(currentZoomBrazil==='matopiba') muns = muns.filter(c => ['MA','PI'].includes(c.state) || c.region==='BA_OESTE');

  muns.forEach(c => {
    const v = cfg.getter(c);
    const muted = (v==null||isNaN(v));
    const color = muted?'#475569':interpolate(cfg.stops,v);
    const cls = 'weather-marker' + (c.top_tier?' top12':'') + (muted?' muted':'');
    const icon = L.divIcon({
      html:`<div class="${cls}" style="background:${color};">${fmtBR(v,currentMetricBrazil)}</div>`,
      className:'', iconSize:[c.top_tier?64:56,c.top_tier?30:26],
      iconAnchor:[c.top_tier?32:28,c.top_tier?15:13]
    });
    const cur = c.current||{};
    const fc = c.forecast_24h||{};
    const popup = `<p class="popup-title">${c.top_tier?'⭐ ':''}${c.county}</p>`
      +`<p class="popup-cond">State: <strong>${c.state}</strong> · ${c.region}</p>`
      +`<div class="popup-row"><span class="k">Temperature</span><span class="v">${cur.temp_f!=null?Math.round(cur.temp_f)+'°F':'—'}</span></div>`
      +`<div class="popup-row"><span class="k">Humidity</span><span class="v">${cur.humidity!=null?Math.round(cur.humidity)+'%':'—'}</span></div>`
      +`<div class="popup-row"><span class="k">Wind</span><span class="v">${cur.wind_mph!=null?Math.round(cur.wind_mph)+' mph':'—'}</span></div>`
      +`<div class="popup-row" style="border-top:1px solid #334155;padding-top:6px;margin-top:6px"><span class="k"><strong>Rain past 3d</strong></span><span class="v"><strong>${c.past3_precip_mm??'—'} mm</strong></span></div>`
      +`<div class="popup-row"><span class="k"><strong>Rain past 7d</strong></span><span class="v"><strong>${c.past7_precip_mm??'—'} mm</strong></span></div>`
      +`<div class="popup-row"><span class="k">ET0 past 3d</span><span class="v">${c.past3_et0_mm??'—'} mm</span></div>`
      +`<div class="popup-row"><span class="k">Water balance</span><span class="v" style="color:${(c.past3_balance_mm??0)<0?'#ef4444':'#22c55e'}">${c.past3_balance_mm??'—'} mm</span></div>`
      +`<div class="popup-row"><span class="k">Soil moisture</span><span class="v">${cur.soil_root_zone!=null?cur.soil_root_zone.toFixed(2):'—'}</span></div>`
      +`<div class="popup-row" style="border-top:1px solid #334155;padding-top:6px;margin-top:6px"><span class="k"><strong>Max PoP 24h</strong></span><span class="v"><strong>${fc.max_prob??'—'}%</strong></span></div>`
      +`<div class="popup-row"><span class="k">Rain forecast 24h</span><span class="v">${fc.total_mm??'—'} mm</span></div>`
      +`<div class="popup-row"><span class="k"><strong>Rain forecast 7d</strong></span><span class="v"><strong>${c.forecast_7d_precip_mm??'—'} mm</strong></span></div>`;
    L.marker([c.lat,c.lon],{icon}).bindPopup(popup).addTo(layerBrazil);
  });
  updateLegendBrazil();
}
function updateLegendBrazil(){
  const cfg = METRIC_CFG_BRAZIL[currentMetricBrazil];
  document.getElementById('legendTitleBrazil').textContent = cfg.label;
  const scale = document.getElementById('legendScaleBrazil');
  const stops = document.getElementById('legendStopsBrazil');
  scale.innerHTML=''; stops.innerHTML='';
  for(let i=0;i<32;i++){
    const t=i/31, v=cfg.domain[0]+t*(cfg.domain[1]-cfg.domain[0]);
    const s=document.createElement('span'); s.style.background=interpolate(cfg.stops,v);
    scale.appendChild(s);
  }
  for(let i=0;i<=5;i++){
    const t=i/5, v=cfg.domain[0]+t*(cfg.domain[1]-cfg.domain[0]);
    const s=document.createElement('span');
    s.textContent = currentMetricBrazil==='soil' ? v.toFixed(2) : Math.round(v)+cfg.unit;
    stops.appendChild(s);
  }
}
const ZOOM_BRAZIL = {
  all:      {c:[-13,-50], z:5},
  mt:       {c:[-13,-56], z:6},
  ba:       {c:[-12.5,-45], z:7},
  top:      {c:[-13,-50], z:5},
  matopiba: {c:[-10,-46], z:6}
};
document.querySelectorAll('#zoomPillsBrazil .zoom-pill').forEach(b=>b.addEventListener('click',()=>{
  document.querySelectorAll('#zoomPillsBrazil .zoom-pill').forEach(x=>x.classList.remove('active'));
  b.classList.add('active'); currentZoomBrazil=b.dataset.zoom;
  const v = ZOOM_BRAZIL[currentZoomBrazil]; window.mapBrazil.flyTo(v.c, v.z, {duration:.9});
  buildMarkersBrazil();
}));
document.querySelectorAll('#metricPillsBrazil .metric-pill').forEach(b=>b.addEventListener('click',()=>{
  document.querySelectorAll('#metricPillsBrazil .metric-pill').forEach(x=>x.classList.remove('active'));
  b.classList.add('active'); currentMetricBrazil=b.dataset.metric;
  buildMarkersBrazil();
}));
buildMarkersBrazil();

// Brazil table filter
document.querySelectorAll('#tab-brazil .table-filter button').forEach(b=>b.addEventListener('click',()=>{
  document.querySelectorAll('#tab-brazil .table-filter button').forEach(x=>x.classList.remove('active'));
  b.classList.add('active');
  const f = b.dataset.filter;
  document.querySelectorAll('#brazilTableBody tr').forEach(tr=>{
    if(f==='all') tr.style.display='';
    else if(f==='top_tier') tr.style.display = tr.classList.contains('top12') ? '' : 'none';
    else tr.style.display = tr.dataset.state===f ? '' : 'none';
  });
}));

// Brazil detail panel
const brazilSelect = document.getElementById('brazilSelect');
DATA.brazil.municipalities.slice()
  .sort((a,b)=>(b.top_tier?1:0)-(a.top_tier?1:0) || a.county.localeCompare(b.county))
  .forEach(c=>{
    const o = document.createElement('option'); o.value = c.county;
    o.textContent = (c.top_tier?'⭐ ':'') + c.county + ' · ' + c.state;
    brazilSelect.appendChild(o);
  });

const chartsBR = {};
function mkChartBR(id, label, color, yLabel, type){
  const ctx = document.getElementById(id).getContext('2d');
  chartsBR[id] = new Chart(ctx, {
    type: type||'line',
    data:{labels:[],datasets:[{label,data:[],borderColor:color,backgroundColor:type==='bar'?color+'88':color+'22',
          fill:true,tension:.35,borderWidth:2,pointRadius:0,pointHoverRadius:4}]},
    options:{responsive:true,maintainAspectRatio:false,
      plugins:{legend:{display:true,labels:{color:'#a9b7cc',font:{family:'JetBrains Mono',size:10}}},
               tooltip:{backgroundColor:'#0b1530'}},
      scales:{x:{ticks:{font:{family:'JetBrains Mono',size:9},maxRotation:0,autoSkip:true,maxTicksLimit:8,color:'#a9b7cc'},grid:{display:false}},
              y:{title:{display:true,text:yLabel,color:'#a9b7cc'},ticks:{color:'#a9b7cc'},grid:{color:'#1e293b'}}}}
  });
}
mkChartBR('chartPrecipDailyBR','Precipitation (mm)','#38bdf8','mm','bar');
mkChartBR('chartEt0DailyBR','ET0 (mm)','#f59e0b','mm','bar');
mkChartBR('chartPrecipHourlyBR','Hourly precip (mm)','#38bdf8','mm');
mkChartBR('chartSoilBR','Root-zone soil moisture','#22c55e','m³/m³');

chartsBR.chartPrecipHourlyBR.data.datasets.push({
  label:'PoP (%)',data:[],borderColor:'#a78bfa',backgroundColor:'#a78bfa22',
  yAxisID:'y2',borderWidth:2,pointRadius:0,fill:false,tension:.35
});
chartsBR.chartPrecipHourlyBR.options.scales.y2 = {
  type:'linear',position:'right',title:{display:true,text:'PoP %',color:'#a78bfa'},
  ticks:{color:'#a78bfa'},grid:{display:false},min:0,max:100
};
chartsBR.chartPrecipHourlyBR.update();

// 10-day temperature chart (Brazil)
const ctxTempBR = document.getElementById('chartTemp10dBR').getContext('2d');
const chartTemp10dBR = new Chart(ctxTempBR, {
  type:'line',
  data:{labels:[],datasets:[
    {label:'Max',data:[],borderColor:'#ef4444',backgroundColor:'#ef444433',borderWidth:2.5,pointRadius:3,pointBackgroundColor:'#ef4444',tension:0.35,fill:'+1'},
    {label:'Min',data:[],borderColor:'#3b82f6',backgroundColor:'transparent',borderWidth:2.5,pointRadius:3,pointBackgroundColor:'#3b82f6',tension:0.35,fill:false},
    {label:'Mean',data:[],borderColor:'#fbbf24',borderDash:[4,4],borderWidth:2,pointRadius:2,pointBackgroundColor:'#fbbf24',tension:0.35,fill:false}
  ]},
  options:{responsive:true,maintainAspectRatio:false,interaction:{mode:'index',intersect:false},
    plugins:{legend:{display:true,position:'top',labels:{color:'#a9b7cc',font:{family:'JetBrains Mono',size:11},usePointStyle:true}},
             tooltip:{backgroundColor:'#0b1530',callbacks:{label:ctx=>ctx.dataset.label+': '+Math.round(ctx.parsed.y)+'°F'}}},
    scales:{x:{ticks:{font:{family:'JetBrains Mono',size:10},color:'#a9b7cc'},grid:{color:'#1e293b'}},
            y:{title:{display:true,text:'Temperature (°F)',color:'#a9b7cc'},ticks:{color:'#a9b7cc'},grid:{color:'#1e293b'}}}}
});
chartsBR['chartTemp10dBR'] = chartTemp10dBR;

function renderBrazilDetail(name){
  const c = DATA.brazil.municipalities.find(x=>x.county===name);
  if(!c) return;

  const daily = c.daily||[];
  const labels = daily.map(d => d.date.substring(5));
  chartsBR.chartPrecipDailyBR.data.labels = labels;
  chartsBR.chartPrecipDailyBR.data.datasets[0].data = daily.map(d=>d.precip_mm);
  chartsBR.chartPrecipDailyBR.data.datasets[0].backgroundColor = daily.map(d => d.period==='past' ? '#38bdf8cc' : '#7dd3fcaa');
  chartsBR.chartPrecipDailyBR.update();

  chartsBR.chartEt0DailyBR.data.labels = labels;
  chartsBR.chartEt0DailyBR.data.datasets[0].data = daily.map(d=>d.et0_mm);
  chartsBR.chartEt0DailyBR.data.datasets[0].backgroundColor = daily.map(d => d.period==='past' ? '#f59e0bcc' : '#fcd34daa');
  chartsBR.chartEt0DailyBR.update();

  const hourly = c.hourly||[];
  const hLabels = hourly.map(h => fmtHourLabel(h.time));
  chartsBR.chartPrecipHourlyBR.data.labels = hLabels;
  chartsBR.chartPrecipHourlyBR.data.datasets[0].data = hourly.map(h=>h.precipitation);
  chartsBR.chartPrecipHourlyBR.data.datasets[1].data = hourly.map(h=>h.precipitation_probability);
  chartsBR.chartPrecipHourlyBR.update();

  chartsBR.chartSoilBR.data.labels = hLabels;
  chartsBR.chartSoilBR.data.datasets[0].data = hourly.map(h=>h.soil_root_zone);
  chartsBR.chartSoilBR.update();

  const fc = c.forecast_24h||{};
  const balCls = (c.past3_balance_mm??0)<0 ? 'negative' : 'positive';
  const stats = document.getElementById('brazilStats');
  stats.innerHTML =
     `<div class="cotton-stat"><span class="cotton-stat-value">${c.past3_precip_mm??'—'}mm</span><span class="cotton-stat-label">Rain past 3 days</span></div>`
    +`<div class="cotton-stat"><span class="cotton-stat-value">${c.past7_precip_mm??'—'}mm</span><span class="cotton-stat-label">Rain past 7 days</span></div>`
    +`<div class="cotton-stat"><span class="cotton-stat-value">${c.past3_et0_mm??'—'}mm</span><span class="cotton-stat-label">ET0 past 3 days</span></div>`
    +`<div class="cotton-stat"><span class="cotton-stat-value" style="color:${balCls==='negative'?'#fca5a5':'#86efac'}">${c.past3_balance_mm??'—'}mm</span><span class="cotton-stat-label">3d water balance</span></div>`
    +`<div class="cotton-stat"><span class="cotton-stat-value">${fc.max_prob??'—'}${fc.max_prob!=null?'%':''}</span><span class="cotton-stat-label">Peak PoP 24h</span></div>`
    +`<div class="cotton-stat"><span class="cotton-stat-value">${fc.total_mm??'—'}${fc.total_mm!=null?'mm':''}</span><span class="cotton-stat-label">Expected rain 24h</span></div>`
    +`<div class="cotton-stat"><span class="cotton-stat-value">${fc.hours_pop_gt_50??0}h</span><span class="cotton-stat-label">Hours PoP ≥50%</span></div>`
    +`<div class="cotton-stat"><span class="cotton-stat-value" style="color:#7dd3fc">${c.forecast_7d_precip_mm??'—'}${c.forecast_7d_precip_mm!=null?'mm':''}</span><span class="cotton-stat-label">Expected rain 7 days</span></div>`;

  document.getElementById('obsMetaBrazil').textContent =
    `${c.county} · ${c.state} · ${c.region} · ${c.lat.toFixed(2)},${c.lon.toFixed(2)} · ${c.current?.time||'no recent obs'}`;

  // 10-day temperature
  const tempDaily = daily.filter(d => d.period === 'forecast');
  const tempLabels = tempDaily.map(d => {
    const dt = new Date(d.date);
    const weekday = ['Sun','Mon','Tue','Wed','Thu','Fri','Sat'][dt.getUTCDay()];
    return weekday + ' ' + d.date.substring(5);
  });
  chartsBR.chartTemp10dBR.data.labels = tempLabels;
  chartsBR.chartTemp10dBR.data.datasets[0].data = tempDaily.map(d => d.tmax_f);
  chartsBR.chartTemp10dBR.data.datasets[1].data = tempDaily.map(d => d.tmin_f);
  chartsBR.chartTemp10dBR.data.datasets[2].data = tempDaily.map(d =>
    (typeof d.tmax_f === 'number' && typeof d.tmin_f === 'number')
      ? Math.round(((d.tmax_f + d.tmin_f) / 2) * 10) / 10
      : null
  );
  chartsBR.chartTemp10dBR.update();
}
brazilSelect.addEventListener('change', () => renderBrazilDetail(brazilSelect.value));

// =============================================================================
//             GENERIC COUNTRY TAB FACTORY (China/India/Australia/Turkey)
// =============================================================================
function mountCountryTab(cfg) {
  // cfg = {
  //   key:         'china' | 'india' | 'australia' | 'turkey',
  //   sites:       DATA.china.sites,
  //   mapCenter:   [lat, lon],
  //   mapZoom:     5,
  //   zoomPresets: [{key:'all',label:'All',c:[..],z:5}, ...],
  //   filters:     [{key:'all',label:'All (N)'}, {key:'top_tier',label:'Top tier'}, ...]
  // }
  const key = cfg.key;
  const sites = cfg.sites || [];

  // Helper formatters
  function fmtC(v, m) {
    if (v == null || isNaN(v)) return '—';
    if (m === 'soil') return Number(v).toFixed(2);
    return Math.round(v);
  }

  // Metric configurations (same across countries)
  const METRIC_CFG = {
    past3_precip:     {label:'Rainfall past 3 days (mm)',  stops:PRECIP3D_STOPS, domain:[0,100], unit:'mm',  getter:c=>c.past3_precip_mm},
    past7_precip:     {label:'Rainfall past 7 days (mm)',  stops:PRECIP7D_FCST_STOPS,domain:[0,150],unit:'mm',getter:c=>c.past7_precip_mm},
    forecast24_precip:{label:'Rain forecast next 24h (mm)',stops:PRECIP24_STOPS, domain:[0,30],  unit:'mm',  getter:c=>c.forecast_24h?.total_mm},
    forecast7d_precip:{label:'Rain forecast next 7 days (mm)',stops:PRECIP7D_FCST_STOPS,domain:[0,150],unit:'mm',getter:c=>c.forecast_7d_precip_mm},
    forecast24_pop:   {label:'Max PoP next 24h (%)',       stops:POP_STOPS,      domain:[0,100], unit:'%',   getter:c=>c.forecast_24h?.max_prob},
    balance3d:        {label:'Water balance 3d (Precip-ET0, mm)',stops:BALANCE_STOPS,domain:[-30,50],unit:'mm',getter:c=>c.past3_balance_mm},
    soil:             {label:'Soil moisture (m³/m³)',       stops:SOIL_STOPS,    domain:[0.05,0.45],unit:'',getter:c=>c.current?.soil_root_zone},
    temp:             {label:'Temperature (°F)',           stops:TEMP_STOPS,    domain:[0,110], unit:'°F', getter:c=>c.current?.temp_f},
  };

  // Build zoom pills HTML
  const zoomEl = document.getElementById('zoomPills-'+key);
  cfg.zoomPresets.forEach((z, i) => {
    const b = document.createElement('button');
    b.className = 'zoom-pill' + (i === 0 ? ' active' : '');
    b.dataset.zoom = z.key;
    b.textContent = z.label;
    zoomEl.appendChild(b);
  });

  // Build filters
  const filterEl = document.getElementById('tableFilter-'+key);
  filterEl.innerHTML = '<span>Filter:</span>';
  cfg.filters.forEach((f, i) => {
    const b = document.createElement('button');
    b.className = i === 0 ? 'active' : '';
    b.dataset.filter = f.key;
    b.textContent = f.label;
    filterEl.appendChild(b);
  });

  // Build map
  const map = L.map('map-'+key, {zoomControl:true, scrollWheelZoom:true})
    .setView(cfg.mapCenter, cfg.mapZoom);
  L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png',
    {attribution:'&copy; OpenStreetMap, &copy; CARTO', subdomains:'abcd', maxZoom:19}).addTo(map);
  window['map_'+key] = map;
  const layer = L.layerGroup().addTo(map);

  let currentMetric = 'past3_precip';
  let currentZoom = cfg.zoomPresets[0].key;

  function buildMarkers() {
    layer.clearLayers();
    const mcfg = METRIC_CFG[currentMetric];
    let filtered = sites;
    const zoom = cfg.zoomPresets.find(z => z.key === currentZoom);
    if (zoom && zoom.filter) {
      filtered = sites.filter(zoom.filter);
    }
    filtered.forEach(c => {
      const v = mcfg.getter(c);
      const muted = (v == null || isNaN(v));
      const color = muted ? '#475569' : interpolate(mcfg.stops, v);
      const cls = 'weather-marker' + (c.top_tier ? ' top12' : '') + (muted ? ' muted' : '');
      const icon = L.divIcon({
        html: `<div class="${cls}" style="background:${color};">${fmtC(v, currentMetric)}</div>`,
        className: '',
        iconSize: [c.top_tier ? 64 : 56, c.top_tier ? 30 : 26],
        iconAnchor: [c.top_tier ? 32 : 28, c.top_tier ? 15 : 13]
      });
      const cur = c.current || {};
      const fc = c.forecast_24h || {};
      const popup = `<p class="popup-title">${c.top_tier ? '⭐ ' : ''}${c.county}</p>`
        + `<p class="popup-cond">State: <strong>${c.state}</strong> · ${c.region}</p>`
        + `<div class="popup-row"><span class="k">Temperature</span><span class="v">${cur.temp_f!=null?Math.round(cur.temp_f)+'°F':'—'}</span></div>`
        + `<div class="popup-row"><span class="k">Humidity</span><span class="v">${cur.humidity!=null?Math.round(cur.humidity)+'%':'—'}</span></div>`
        + `<div class="popup-row"><span class="k">Wind</span><span class="v">${cur.wind_mph!=null?Math.round(cur.wind_mph)+' mph':'—'}</span></div>`
        + `<div class="popup-row" style="border-top:1px solid #334155;padding-top:6px;margin-top:6px"><span class="k"><strong>Rain past 3d</strong></span><span class="v"><strong>${c.past3_precip_mm??'—'} mm</strong></span></div>`
        + `<div class="popup-row"><span class="k"><strong>Rain past 7d</strong></span><span class="v"><strong>${c.past7_precip_mm??'—'} mm</strong></span></div>`
        + `<div class="popup-row"><span class="k">ET0 past 3d</span><span class="v">${c.past3_et0_mm??'—'} mm</span></div>`
        + `<div class="popup-row"><span class="k">Water balance</span><span class="v" style="color:${(c.past3_balance_mm??0)<0?'#ef4444':'#22c55e'}">${c.past3_balance_mm??'—'} mm</span></div>`
        + `<div class="popup-row"><span class="k">Soil moisture</span><span class="v">${cur.soil_root_zone!=null?cur.soil_root_zone.toFixed(2):'—'}</span></div>`
        + `<div class="popup-row" style="border-top:1px solid #334155;padding-top:6px;margin-top:6px"><span class="k"><strong>Max PoP 24h</strong></span><span class="v"><strong>${fc.max_prob??'—'}%</strong></span></div>`
        + `<div class="popup-row"><span class="k">Rain forecast 24h</span><span class="v">${fc.total_mm??'—'} mm</span></div>`
        + `<div class="popup-row"><span class="k"><strong>Rain forecast 7d</strong></span><span class="v"><strong>${c.forecast_7d_precip_mm??'—'} mm</strong></span></div>`;
      L.marker([c.lat, c.lon], {icon}).bindPopup(popup).addTo(layer);
    });
    updateLegend();
  }

  function updateLegend() {
    const mcfg = METRIC_CFG[currentMetric];
    document.getElementById('legendTitle-'+key).textContent = mcfg.label;
    const scale = document.getElementById('legendScale-'+key);
    const stops = document.getElementById('legendStops-'+key);
    scale.innerHTML = ''; stops.innerHTML = '';
    for (let i = 0; i < 32; i++) {
      const t = i / 31, v = mcfg.domain[0] + t * (mcfg.domain[1] - mcfg.domain[0]);
      const s = document.createElement('span');
      s.style.background = interpolate(mcfg.stops, v);
      scale.appendChild(s);
    }
    for (let i = 0; i <= 5; i++) {
      const t = i / 5, v = mcfg.domain[0] + t * (mcfg.domain[1] - mcfg.domain[0]);
      const s = document.createElement('span');
      s.textContent = currentMetric === 'soil' ? v.toFixed(2) : Math.round(v) + mcfg.unit;
      stops.appendChild(s);
    }
  }

  // Wire zoom pills
  document.querySelectorAll('#zoomPills-' + key + ' .zoom-pill').forEach(b => b.addEventListener('click', () => {
    document.querySelectorAll('#zoomPills-' + key + ' .zoom-pill').forEach(x => x.classList.remove('active'));
    b.classList.add('active'); currentZoom = b.dataset.zoom;
    const zoom = cfg.zoomPresets.find(z => z.key === currentZoom);
    if (zoom) map.flyTo(zoom.c, zoom.z, {duration: .9});
    buildMarkers();
  }));

  // Wire metric pills
  document.querySelectorAll('#metricPills-' + key + ' .metric-pill').forEach(b => b.addEventListener('click', () => {
    document.querySelectorAll('#metricPills-' + key + ' .metric-pill').forEach(x => x.classList.remove('active'));
    b.classList.add('active'); currentMetric = b.dataset.metric;
    buildMarkers();
  }));

  // Wire filters
  document.querySelectorAll('#tableFilter-' + key + ' button').forEach(b => b.addEventListener('click', () => {
    document.querySelectorAll('#tableFilter-' + key + ' button').forEach(x => x.classList.remove('active'));
    b.classList.add('active');
    const f = b.dataset.filter;
    document.querySelectorAll('#tableBody-' + key + ' tr').forEach(tr => {
      if (f === 'all') {
        tr.style.display = '';
      } else if (f === 'top_tier') {
        tr.style.display = tr.classList.contains('top12') ? '' : 'none';
      } else {
        // Match filter against (a) state code (XJ, HE, SD, NSW, QLD, GJ, MH, etc.)
        // or (b) region prefix before the first underscore (NXJ_HAMI -> NXJ,
        // SXJ_AKSU -> SXJ, YR_HEBEI -> YR, YZ_HUBEI -> YZ, MT_NORTE -> MT etc.)
        const regionPrefix = (tr.dataset.region || '').split('_')[0];
        const matches = (tr.dataset.state === f) || (regionPrefix === f);
        tr.style.display = matches ? '' : 'none';
      }
    });
  }));

  buildMarkers();

  // ---- Detail panel ----
  const select = document.getElementById('select-' + key);
  sites.slice()
    .sort((a, b) => (b.top_tier ? 1 : 0) - (a.top_tier ? 1 : 0) || a.county.localeCompare(b.county))
    .forEach(c => {
      const o = document.createElement('option');
      o.value = c.county;
      o.textContent = (c.top_tier ? '⭐ ' : '') + c.county + ' · ' + c.state;
      select.appendChild(o);
    });

  const charts = {};
  function mkChart(id, label, color, yLabel, type) {
    const ctx = document.getElementById(id).getContext('2d');
    charts[id] = new Chart(ctx, {
      type: type || 'line',
      data: {labels: [], datasets: [{label, data: [], borderColor: color,
        backgroundColor: type === 'bar' ? color + '88' : color + '22',
        fill: true, tension: .35, borderWidth: 2, pointRadius: 0, pointHoverRadius: 4}]},
      options: {responsive: true, maintainAspectRatio: false,
        plugins: {legend: {display: true, labels: {color: '#a9b7cc', font: {family: 'JetBrains Mono', size: 10}}}, tooltip: {backgroundColor: '#0b1530'}},
        scales: {x: {ticks: {font: {family: 'JetBrains Mono', size: 9}, maxRotation: 0, autoSkip: true, maxTicksLimit: 8, color: '#a9b7cc'}, grid: {display: false}},
                 y: {title: {display: true, text: yLabel, color: '#a9b7cc'}, ticks: {color: '#a9b7cc'}, grid: {color: '#1e293b'}}}}
    });
  }
  mkChart('chartPrecipDaily-' + key, 'Precipitation (mm)', '#38bdf8', 'mm', 'bar');
  mkChart('chartEt0Daily-' + key, 'ET0 (mm)', '#f59e0b', 'mm', 'bar');
  mkChart('chartPrecipHourly-' + key, 'Hourly precip (mm)', '#38bdf8', 'mm');
  mkChart('chartSoil-' + key, 'Root-zone soil moisture', '#22c55e', 'm³/m³');

  charts['chartPrecipHourly-' + key].data.datasets.push({
    label: 'PoP (%)', data: [], borderColor: '#a78bfa', backgroundColor: '#a78bfa22',
    yAxisID: 'y2', borderWidth: 2, pointRadius: 0, fill: false, tension: .35
  });
  charts['chartPrecipHourly-' + key].options.scales.y2 = {
    type: 'linear', position: 'right',
    title: {display: true, text: 'PoP %', color: '#a78bfa'},
    ticks: {color: '#a78bfa'}, grid: {display: false}, min: 0, max: 100
  };
  charts['chartPrecipHourly-' + key].update();

  // 10-day temperature chart
  const ctxTemp = document.getElementById('chartTemp10d-' + key).getContext('2d');
  charts['chartTemp10d-' + key] = new Chart(ctxTemp, {
    type: 'line',
    data: {labels: [], datasets: [
      {label: 'Max', data: [], borderColor: '#ef4444', backgroundColor: '#ef444433', borderWidth: 2.5, pointRadius: 3, pointBackgroundColor: '#ef4444', tension: 0.35, fill: '+1'},
      {label: 'Min', data: [], borderColor: '#3b82f6', backgroundColor: 'transparent', borderWidth: 2.5, pointRadius: 3, pointBackgroundColor: '#3b82f6', tension: 0.35, fill: false},
      {label: 'Mean', data: [], borderColor: '#fbbf24', borderDash: [4, 4], borderWidth: 2, pointRadius: 2, pointBackgroundColor: '#fbbf24', tension: 0.35, fill: false}
    ]},
    options: {responsive: true, maintainAspectRatio: false, interaction: {mode: 'index', intersect: false},
      plugins: {legend: {display: true, position: 'top', labels: {color: '#a9b7cc', font: {family: 'JetBrains Mono', size: 11}, usePointStyle: true}},
        tooltip: {backgroundColor: '#0b1530', callbacks: {label: ctx => ctx.dataset.label + ': ' + Math.round(ctx.parsed.y) + '°F'}}},
      scales: {x: {ticks: {font: {family: 'JetBrains Mono', size: 10}, color: '#a9b7cc'}, grid: {color: '#1e293b'}},
        y: {title: {display: true, text: 'Temperature (°F)', color: '#a9b7cc'}, ticks: {color: '#a9b7cc'}, grid: {color: '#1e293b'}}}}
  });

  function renderDetail(name) {
    const c = sites.find(x => x.county === name);
    if (!c) return;

    const daily = c.daily || [];
    const labels = daily.map(d => d.date.substring(5));
    charts['chartPrecipDaily-' + key].data.labels = labels;
    charts['chartPrecipDaily-' + key].data.datasets[0].data = daily.map(d => d.precip_mm);
    charts['chartPrecipDaily-' + key].data.datasets[0].backgroundColor = daily.map(d => d.period === 'past' ? '#38bdf8cc' : '#7dd3fcaa');
    charts['chartPrecipDaily-' + key].update();

    charts['chartEt0Daily-' + key].data.labels = labels;
    charts['chartEt0Daily-' + key].data.datasets[0].data = daily.map(d => d.et0_mm);
    charts['chartEt0Daily-' + key].data.datasets[0].backgroundColor = daily.map(d => d.period === 'past' ? '#f59e0bcc' : '#fcd34daa');
    charts['chartEt0Daily-' + key].update();

    const hourly = c.hourly || [];
    const hLabels = hourly.map(h => fmtHourLabel(h.time));
    charts['chartPrecipHourly-' + key].data.labels = hLabels;
    charts['chartPrecipHourly-' + key].data.datasets[0].data = hourly.map(h => h.precipitation);
    charts['chartPrecipHourly-' + key].data.datasets[1].data = hourly.map(h => h.precipitation_probability);
    charts['chartPrecipHourly-' + key].update();

    charts['chartSoil-' + key].data.labels = hLabels;
    charts['chartSoil-' + key].data.datasets[0].data = hourly.map(h => h.soil_root_zone);
    charts['chartSoil-' + key].update();

    const fc = c.forecast_24h || {};
    const balCls = (c.past3_balance_mm ?? 0) < 0 ? 'negative' : 'positive';
    document.getElementById('stats-' + key).innerHTML =
       `<div class="cotton-stat"><span class="cotton-stat-value">${c.past3_precip_mm??'—'}mm</span><span class="cotton-stat-label">Rain past 3 days</span></div>`
      +`<div class="cotton-stat"><span class="cotton-stat-value">${c.past7_precip_mm??'—'}mm</span><span class="cotton-stat-label">Rain past 7 days</span></div>`
      +`<div class="cotton-stat"><span class="cotton-stat-value">${c.past3_et0_mm??'—'}mm</span><span class="cotton-stat-label">ET0 past 3 days</span></div>`
      +`<div class="cotton-stat"><span class="cotton-stat-value" style="color:${balCls==='negative'?'#fca5a5':'#86efac'}">${c.past3_balance_mm??'—'}mm</span><span class="cotton-stat-label">3d water balance</span></div>`
      +`<div class="cotton-stat"><span class="cotton-stat-value">${fc.max_prob??'—'}${fc.max_prob!=null?'%':''}</span><span class="cotton-stat-label">Peak PoP 24h</span></div>`
      +`<div class="cotton-stat"><span class="cotton-stat-value">${fc.total_mm??'—'}${fc.total_mm!=null?'mm':''}</span><span class="cotton-stat-label">Expected rain 24h</span></div>`
      +`<div class="cotton-stat"><span class="cotton-stat-value">${fc.hours_pop_gt_50??0}h</span><span class="cotton-stat-label">Hours PoP ≥50%</span></div>`
      +`<div class="cotton-stat"><span class="cotton-stat-value" style="color:#7dd3fc">${c.forecast_7d_precip_mm??'—'}${c.forecast_7d_precip_mm!=null?'mm':''}</span><span class="cotton-stat-label">Expected rain 7 days</span></div>`;

    document.getElementById('obsMeta-' + key).textContent =
      `${c.county} · ${c.state} · ${c.region} · ${c.lat.toFixed(2)},${c.lon.toFixed(2)} · ${c.current?.time||'no recent obs'}`;

    const tempDaily = daily.filter(d => d.period === 'forecast');
    const tempLabels = tempDaily.map(d => {
      const dt = new Date(d.date);
      const weekday = ['Sun','Mon','Tue','Wed','Thu','Fri','Sat'][dt.getUTCDay()];
      return weekday + ' ' + d.date.substring(5);
    });
    charts['chartTemp10d-' + key].data.labels = tempLabels;
    charts['chartTemp10d-' + key].data.datasets[0].data = tempDaily.map(d => d.tmax_f);
    charts['chartTemp10d-' + key].data.datasets[1].data = tempDaily.map(d => d.tmin_f);
    charts['chartTemp10d-' + key].data.datasets[2].data = tempDaily.map(d =>
      (typeof d.tmax_f === 'number' && typeof d.tmin_f === 'number')
        ? Math.round(((d.tmax_f + d.tmin_f) / 2) * 10) / 10
        : null
    );
    charts['chartTemp10d-' + key].update();
  }

  select.addEventListener('change', () => renderDetail(select.value));
  const defaultName = cfg.defaultSite || (sites.find(s => s.top_tier) || sites[0])?.county;
  if (defaultName) {
    select.value = defaultName;
    renderDetail(defaultName);
  }
}

// Mount the 4 new country tabs
mountCountryTab({
  key: 'china',
  sites: DATA.china.sites,
  mapCenter: [40, 84], mapZoom: 5,
  zoomPresets: [
    {key: 'all',  label: 'All',           c: [40, 84],  z: 5},
    {key: 'nxj',  label: 'North Xinjiang',c: [44, 86],  z: 7, filter: s => s.region.startsWith('NXJ')},
    {key: 'sxj',  label: 'South Xinjiang',c: [40, 80],  z: 6, filter: s => s.region.startsWith('SXJ')},
    {key: 'top',  label: 'Top tier',      c: [40, 84],  z: 5, filter: s => s.top_tier},
    {key: 'residual', label: 'Yellow+Yangtze', c: [33, 116], z: 6, filter: s => s.region.startsWith('YR') || s.region.startsWith('YZ')},
  ],
  filters: [
    {key: 'all',      label: 'All (' + DATA.china.sites.length + ')'},
    {key: 'top_tier', label: 'Top tier'},
    {key: 'XJ',       label: 'All Xinjiang'},
    {key: 'NXJ',      label: 'North Xinjiang'},
    {key: 'SXJ',      label: 'South Xinjiang'},
    {key: 'HE',       label: 'Hebei'},
    {key: 'SD',       label: 'Shandong'},
    {key: 'HA',       label: 'Henan'},
    {key: 'HB',       label: 'Hubei'},
    {key: 'JS',       label: 'Jiangsu'},
    {key: 'AH',       label: 'Anhui'},
  ],
  defaultSite: 'Aksu',
});

mountCountryTab({
  key: 'india',
  sites: DATA.india.sites,
  mapCenter: [22, 78], mapZoom: 5,
  zoomPresets: [
    {key: 'all',     label: 'All',         c: [22, 78],   z: 5},
    {key: 'west',    label: 'Gujarat',     c: [22, 71],   z: 7, filter: s => s.state === 'GJ'},
    {key: 'central', label: 'Maharashtra', c: [20, 77],   z: 7, filter: s => s.state === 'MH'},
    {key: 'south',   label: 'Telangana/AP/KA', c: [17, 78], z: 6, filter: s => ['TG','AP','KA'].includes(s.state)},
    {key: 'north',   label: 'North (PB/HR/RJ)',c: [29, 75], z: 6, filter: s => ['PB','HR','RJ'].includes(s.state)},
    {key: 'top',     label: 'Top tier',    c: [22, 78],   z: 5, filter: s => s.top_tier},
  ],
  filters: [
    {key: 'all', label: 'All (' + DATA.india.sites.length + ')'},
    {key: 'top_tier', label: 'Top tier'},
    {key: 'GJ', label: 'Gujarat'},
    {key: 'MH', label: 'Maharashtra'},
    {key: 'TG', label: 'Telangana'},
    {key: 'AP', label: 'AP'},
    {key: 'KA', label: 'Karnataka'},
    {key: 'MP', label: 'MP'},
    {key: 'RJ', label: 'Rajasthan'},
    {key: 'HR', label: 'Haryana'},
    {key: 'PB', label: 'Punjab'},
    {key: 'TN', label: 'Tamil Nadu'},
    {key: 'OD', label: 'Odisha'},
  ],
  defaultSite: 'Yavatmal',
});

// India monsoon banner
const monsoonStatus = DATA.india_monsoon || {};
const banner = document.getElementById('monsoonBanner');
if (banner) {
  if (monsoonStatus.active) {
    banner.classList.add('active');
    document.getElementById('monsoonPhase').textContent =
      `MONSOON ${monsoonStatus.phase || 'active'} PHASE · KHARIF COTTON CRITICAL WINDOW`;
  } else {
    document.getElementById('monsoonPhase').textContent =
      `MONSOON OFF-SEASON · ${monsoonStatus.phase || ''}`;
  }
  document.getElementById('monsoonMsg').textContent = monsoonStatus.message || '';
}

mountCountryTab({
  key: 'australia',
  sites: DATA.australia.sites,
  mapCenter: [-30, 148], mapZoom: 5,
  zoomPresets: [
    {key: 'all', label: 'All',       c: [-29, 148], z: 5},
    {key: 'nsw', label: 'NSW',       c: [-31, 147], z: 6, filter: s => s.state === 'NSW'},
    {key: 'qld', label: 'QLD',       c: [-26, 149], z: 6, filter: s => s.state === 'QLD'},
    {key: 'top', label: 'Top tier',  c: [-29, 148], z: 5, filter: s => s.top_tier},
    {key: 'north', label: 'Northern Aus', c: [-17, 138], z: 5, filter: s => s.state === 'WA' || s.region === 'QLD_BURDEKIN'},
  ],
  filters: [
    {key: 'all', label: 'All (' + DATA.australia.sites.length + ')'},
    {key: 'top_tier', label: 'Top tier'},
    {key: 'NSW', label: 'NSW'},
    {key: 'QLD', label: 'QLD'},
    {key: 'WA', label: 'WA (Ord)'},
  ],
  defaultSite: 'Moree',
});

mountCountryTab({
  key: 'turkey',
  sites: DATA.turkey.sites,
  mapCenter: [38, 36], mapZoom: 6,
  zoomPresets: [
    {key: 'all',   label: 'All',           c: [38, 36], z: 6},
    {key: 'gap',   label: 'GAP / Southeast', c: [37.4, 39.5], z: 7, filter: s => s.region === 'TR_GAP'},
    {key: 'cuk',   label: 'Cukurova / South', c: [37, 35.5], z: 7, filter: s => s.region === 'TR_CUKUROVA'},
    {key: 'aegean',label: 'Aegean / West',  c: [38.3, 28], z: 7, filter: s => s.region === 'TR_AEGEAN'},
    {key: 'top',   label: 'Top tier',      c: [38, 36], z: 6, filter: s => s.top_tier},
  ],
  filters: [
    {key: 'all', label: 'All (' + DATA.turkey.sites.length + ')'},
    {key: 'top_tier', label: 'Top tier'},
    {key: 'GAP', label: 'GAP / Southeast'},
    {key: 'MED', label: 'Cukurova / South'},
    {key: 'AEG', label: 'Aegean / West'},
  ],
  defaultSite: 'Şanlıurfa',
});

// =============================================================================
//                CHINA LONG-RANGE FORECAST CHART (16d + 35d)
// =============================================================================
(function setupChinaLongRange() {
  const chinaSites = DATA.china?.sites || [];
  // Map from county name -> long_range data (undefined if not top_tier)
  const lrByCounty = {};
  chinaSites.forEach(s => { if (s.long_range) lrByCounty[s.county] = s.long_range; });

  const canvas = document.getElementById('chartLongRange-china');
  if (!canvas) return;
  const ctx = canvas.getContext('2d');

  let lrMode = 'max';  // 'max', 'min', or 'both'

  const chart = new Chart(ctx, {
    type: 'line',
    data: { labels: [], datasets: [] },
    options: {
      responsive: true, maintainAspectRatio: false,
      interaction: { mode: 'index', intersect: false },
      plugins: {
        legend: {
          display: true, position: 'top',
          labels: { color: '#a9b7cc', font: { family: 'JetBrains Mono', size: 10 },
                    usePointStyle: true, padding: 12, boxWidth: 8 }
        },
        tooltip: {
          backgroundColor: '#0b1530',
          callbacks: {
            label: ctx => {
              const v = ctx.parsed.y;
              if (v == null) return '';
              return ctx.dataset.label + ': ' + Math.round(v) + '°F';
            }
          }
        },
        annotation: {}  // placeholder if we add plugin
      },
      scales: {
        x: {
          ticks: { color: '#a9b7cc', font: { family: 'JetBrains Mono', size: 9 },
                   maxRotation: 0, autoSkip: true, maxTicksLimit: 12 },
          grid: { color: '#1e293b' },
          title: { display: true, text: 'Date', color: '#a9b7cc',
                   font: { family: 'JetBrains Mono', size: 10 } }
        },
        y: {
          ticks: { color: '#a9b7cc' },
          grid: { color: '#1e293b' },
          title: { display: true, text: 'Temperature (°F)', color: '#a9b7cc' }
        }
      }
    }
  });

  function renderLongRange(countyName) {
    const wrap = document.getElementById('longRangeWrap-china');
    const lr = lrByCounty[countyName];

    if (!lr || (!lr.deterministic_16d?.length && !lr.ensemble_35d?.length)) {
      // Long-range data not available for this site (fetch failed or upstream API unavailable)
      chart.data.labels = [];
      chart.data.datasets = [];
      chart.update();
      wrap.innerHTML = `
        <p class="chart-title">Long-range temperature forecast · 35 days</p>
        <p class="chart-sub" style="color:var(--ink-soft);font-style:italic;padding:24px 0;text-align:center">
          Long-range forecast data unavailable for this site.
          The upstream Open-Meteo endpoint likely failed during the last run.
          Retry the notebook to refresh; the disk cache preserves already-fetched sites.
        </p>
      `;
      return;
    }

    // Merge the two data sources into a single unified timeline
    const detByDate = {};
    (lr.deterministic_16d || []).forEach(d => { detByDate[d.date] = d; });
    const ensByDate = {};
    (lr.ensemble_35d || []).forEach(d => { ensByDate[d.date] = d; });

    // Union of all dates, sorted
    const allDates = Array.from(new Set([
      ...Object.keys(detByDate),
      ...Object.keys(ensByDate),
    ])).sort();

    const labels = allDates.map(d => {
      const dt = new Date(d);
      const weekday = ['Sun','Mon','Tue','Wed','Thu','Fri','Sat'][dt.getUTCDay()];
      return weekday + ' ' + d.substring(5);
    });

    const showMax = (lrMode === 'max' || lrMode === 'both');
    const showMin = (lrMode === 'min' || lrMode === 'both');

    // Deterministic 1-10 (high skill zone)
    const detMax_1_10 = allDates.map((d, i) => (i < 10 && detByDate[d]) ? detByDate[d].tmax_f : null);
    const detMin_1_10 = allDates.map((d, i) => (i < 10 && detByDate[d]) ? detByDate[d].tmin_f : null);

    // Deterministic 11-16 (extended, degraded skill)
    const detMax_11_16 = allDates.map((d, i) => (i >= 10 && i < 16 && detByDate[d]) ? detByDate[d].tmax_f : null);
    const detMin_11_16 = allDates.map((d, i) => (i >= 10 && i < 16 && detByDate[d]) ? detByDate[d].tmin_f : null);

    // Ensemble mean 15-35, with spread as ribbon
    // We show Tmean as the p50, and Tmean ± spread as p10/p90 approximation
    const ensMean = allDates.map((d, i) => (i >= 14 && ensByDate[d]) ? ensByDate[d].tmean_f : null);
    const ensUpper = allDates.map((d, i) => {
      if (i < 14 || !ensByDate[d]) return null;
      return ensByDate[d].tmean_f + ensByDate[d].tmean_spread_f;
    });
    const ensLower = allDates.map((d, i) => {
      if (i < 14 || !ensByDate[d]) return null;
      return ensByDate[d].tmean_f - ensByDate[d].tmean_spread_f;
    });

    const datasets = [];

    if (showMax) {
      datasets.push({
        label: 'Tmax (day 1-10)', data: detMax_1_10,
        borderColor: '#ef4444', backgroundColor: 'transparent',
        borderWidth: 3, pointRadius: 3, pointBackgroundColor: '#ef4444',
        tension: 0.3, spanGaps: false,
      });
      datasets.push({
        label: 'Tmax (day 11-16, extended)', data: detMax_11_16,
        borderColor: '#ef4444', backgroundColor: 'transparent',
        borderWidth: 2, borderDash: [6, 4],
        pointRadius: 2, pointBackgroundColor: '#ef4444', pointStyle: 'triangle',
        tension: 0.3, spanGaps: false,
      });
    }
    if (showMin) {
      datasets.push({
        label: 'Tmin (day 1-10)', data: detMin_1_10,
        borderColor: '#3b82f6', backgroundColor: 'transparent',
        borderWidth: 3, pointRadius: 3, pointBackgroundColor: '#3b82f6',
        tension: 0.3, spanGaps: false,
      });
      datasets.push({
        label: 'Tmin (day 11-16, extended)', data: detMin_11_16,
        borderColor: '#3b82f6', backgroundColor: 'transparent',
        borderWidth: 2, borderDash: [6, 4],
        pointRadius: 2, pointBackgroundColor: '#3b82f6', pointStyle: 'triangle',
        tension: 0.3, spanGaps: false,
      });
    }

    // Ensemble band (always shown when in scope)
    // Upper bound of ribbon
    datasets.push({
      label: 'Ensemble upper (mean + spread)', data: ensUpper,
      borderColor: 'rgba(251, 191, 36, 0.4)',
      backgroundColor: 'rgba(251, 191, 36, 0.15)',
      borderWidth: 1, borderDash: [2, 3],
      pointRadius: 0, fill: '+1', tension: 0.3, spanGaps: true,
    });
    // Lower bound of ribbon (fills between this and upper)
    datasets.push({
      label: 'Ensemble lower (mean - spread)', data: ensLower,
      borderColor: 'rgba(251, 191, 36, 0.4)',
      backgroundColor: 'transparent',
      borderWidth: 1, borderDash: [2, 3],
      pointRadius: 0, fill: false, tension: 0.3, spanGaps: true,
    });
    // Ensemble mean line on top
    datasets.push({
      label: 'Ensemble mean (day 15-35)', data: ensMean,
      borderColor: '#fbbf24', backgroundColor: 'transparent',
      borderWidth: 2.5, pointRadius: 2, pointBackgroundColor: '#fbbf24',
      tension: 0.3, spanGaps: true,
    });

    chart.data.labels = labels;
    chart.data.datasets = datasets;
    chart.update();
  }

  // Toggle Tmax/Tmin/Both
  document.querySelectorAll('.lr-toggle').forEach(btn => {
    btn.addEventListener('click', () => {
      document.querySelectorAll('.lr-toggle').forEach(b => b.classList.remove('active'));
      btn.classList.add('active');
      lrMode = btn.dataset.lr;
      const currentCounty = document.getElementById('select-china').value;
      renderLongRange(currentCounty);
    });
  });

  // Hook into the china selector to refresh when the selection changes.
  // The mountCountryTab factory already wires 'change' events, we just add ours.
  const chinaSelect = document.getElementById('select-china');
  if (chinaSelect) {
    chinaSelect.addEventListener('change', () => renderLongRange(chinaSelect.value));
    // Initial render
    renderLongRange(chinaSelect.value);
  }
})();

// =============================================================================
//              CHINA REGIONAL AVERAGE CHARTS (NXJ + SXJ)
// =============================================================================
(function setupChinaRegionalCharts() {
  const chinaSites = DATA.china?.sites || [];

  // Helper: mean of an array, ignoring null/NaN
  function mean(arr) {
    const vals = arr.filter(v => v != null && !isNaN(v));
    if (!vals.length) return null;
    return vals.reduce((a, b) => a + b, 0) / vals.length;
  }

  // Aggregate sites by region prefix
  // For each date across all sites, compute the mean of tmax/tmin/tmean.
  function aggregateRegion(regionPrefix) {
    const sites = chinaSites.filter(s => (s.region || '').startsWith(regionPrefix));

    // Build maps: date -> array of values (one per site)
    const detByDate = {};   // day 1-16 deterministic
    const ensByDate = {};   // day 1-35 ensemble mean

    sites.forEach(s => {
      const lr = s.long_range;
      if (!lr) return;
      (lr.deterministic_16d || []).forEach(d => {
        if (!detByDate[d.date]) detByDate[d.date] = { tmax: [], tmin: [], tmean: [] };
        if (d.tmax_f != null) detByDate[d.date].tmax.push(d.tmax_f);
        if (d.tmin_f != null) detByDate[d.date].tmin.push(d.tmin_f);
        if (d.tmean_f != null) detByDate[d.date].tmean.push(d.tmean_f);
      });
      (lr.ensemble_35d || []).forEach(d => {
        if (!ensByDate[d.date]) ensByDate[d.date] = { mean: [], spread: [] };
        if (d.tmean_f != null) ensByDate[d.date].mean.push(d.tmean_f);
        if (d.tmean_spread_f != null) ensByDate[d.date].spread.push(d.tmean_spread_f);
      });
    });

    // Sort dates
    const detDates = Object.keys(detByDate).sort();
    const ensDates = Object.keys(ensByDate).sort();

    const deterministic_16d = detDates.map(d => ({
      date: d,
      tmax_f:  mean(detByDate[d].tmax),
      tmin_f:  mean(detByDate[d].tmin),
      tmean_f: mean(detByDate[d].tmean),
    }));

    const ensemble_35d = ensDates.map(d => ({
      date: d,
      tmean_f:        mean(ensByDate[d].mean),
      tmean_spread_f: mean(ensByDate[d].spread),
    }));

    return {
      site_count: sites.length,
      deterministic_16d,
      ensemble_35d,
    };
  }

  // Build the aggregated data for both regions
  const nxjData = aggregateRegion('NXJ');
  const sxjData = aggregateRegion('SXJ');

  // Update the site count labels
  const nxjCountEl = document.getElementById('nxjSiteCount');
  const sxjCountEl = document.getElementById('sxjSiteCount');
  if (nxjCountEl) nxjCountEl.textContent = nxjData.site_count;
  if (sxjCountEl) sxjCountEl.textContent = sxjData.site_count;

  // Chart factory (same styling as the per-site long-range chart)
  function makeRegionalChart(canvasId) {
    const ctx = document.getElementById(canvasId).getContext('2d');
    return new Chart(ctx, {
      type: 'line',
      data: { labels: [], datasets: [] },
      options: {
        responsive: true, maintainAspectRatio: false,
        interaction: { mode: 'index', intersect: false },
        plugins: {
          legend: {
            display: true, position: 'top',
            labels: { color: '#a9b7cc', font: { family: 'JetBrains Mono', size: 10 },
                      usePointStyle: true, padding: 12, boxWidth: 8 }
          },
          tooltip: {
            backgroundColor: '#0b1530',
            callbacks: {
              label: ctx => {
                const v = ctx.parsed.y;
                if (v == null) return '';
                return ctx.dataset.label + ': ' + Math.round(v) + '°F';
              }
            }
          },
        },
        scales: {
          x: {
            ticks: { color: '#a9b7cc', font: { family: 'JetBrains Mono', size: 9 },
                     maxRotation: 0, autoSkip: true, maxTicksLimit: 12 },
            grid: { color: '#1e293b' },
            title: { display: true, text: 'Date', color: '#a9b7cc',
                     font: { family: 'JetBrains Mono', size: 10 } }
          },
          y: {
            ticks: { color: '#a9b7cc' },
            grid: { color: '#1e293b' },
            title: { display: true, text: 'Temperature (°F, region mean)', color: '#a9b7cc' }
          }
        }
      }
    });
  }

  // Render a regional chart in the same 3-zone visual style as the per-site chart
  function renderRegional(chart, data, mode) {
    if (!data || (!data.deterministic_16d?.length && !data.ensemble_35d?.length)) {
      chart.data.labels = [];
      chart.data.datasets = [];
      chart.update();
      return;
    }

    const detByDate = {};
    (data.deterministic_16d || []).forEach(d => { detByDate[d.date] = d; });
    const ensByDate = {};
    (data.ensemble_35d || []).forEach(d => { ensByDate[d.date] = d; });

    const allDates = Array.from(new Set([
      ...Object.keys(detByDate),
      ...Object.keys(ensByDate),
    ])).sort();

    const labels = allDates.map(d => {
      const dt = new Date(d);
      const weekday = ['Sun','Mon','Tue','Wed','Thu','Fri','Sat'][dt.getUTCDay()];
      return weekday + ' ' + d.substring(5);
    });

    const showMax = (mode === 'max' || mode === 'both');
    const showMin = (mode === 'min' || mode === 'both');

    // Deterministic zones (1-10 and 11-16)
    const detMax_1_10 = allDates.map((d, i) => (i < 10 && detByDate[d]) ? detByDate[d].tmax_f : null);
    const detMin_1_10 = allDates.map((d, i) => (i < 10 && detByDate[d]) ? detByDate[d].tmin_f : null);
    const detMax_11_16 = allDates.map((d, i) => (i >= 10 && i < 16 && detByDate[d]) ? detByDate[d].tmax_f : null);
    const detMin_11_16 = allDates.map((d, i) => (i >= 10 && i < 16 && detByDate[d]) ? detByDate[d].tmin_f : null);

    // Ensemble mean 15-35 with spread ribbon
    const ensMean  = allDates.map((d, i) => (i >= 14 && ensByDate[d]) ? ensByDate[d].tmean_f : null);
    const ensUpper = allDates.map((d, i) => {
      if (i < 14 || !ensByDate[d]) return null;
      return ensByDate[d].tmean_f + ensByDate[d].tmean_spread_f;
    });
    const ensLower = allDates.map((d, i) => {
      if (i < 14 || !ensByDate[d]) return null;
      return ensByDate[d].tmean_f - ensByDate[d].tmean_spread_f;
    });

    const datasets = [];

    if (showMax) {
      datasets.push({
        label: 'Tmax (day 1-10)', data: detMax_1_10,
        borderColor: '#ef4444', backgroundColor: 'transparent',
        borderWidth: 3, pointRadius: 3, pointBackgroundColor: '#ef4444',
        tension: 0.3, spanGaps: false,
      });
      datasets.push({
        label: 'Tmax (day 11-16, extended)', data: detMax_11_16,
        borderColor: '#ef4444', backgroundColor: 'transparent',
        borderWidth: 2, borderDash: [6, 4],
        pointRadius: 2, pointBackgroundColor: '#ef4444', pointStyle: 'triangle',
        tension: 0.3, spanGaps: false,
      });
    }
    if (showMin) {
      datasets.push({
        label: 'Tmin (day 1-10)', data: detMin_1_10,
        borderColor: '#3b82f6', backgroundColor: 'transparent',
        borderWidth: 3, pointRadius: 3, pointBackgroundColor: '#3b82f6',
        tension: 0.3, spanGaps: false,
      });
      datasets.push({
        label: 'Tmin (day 11-16, extended)', data: detMin_11_16,
        borderColor: '#3b82f6', backgroundColor: 'transparent',
        borderWidth: 2, borderDash: [6, 4],
        pointRadius: 2, pointBackgroundColor: '#3b82f6', pointStyle: 'triangle',
        tension: 0.3, spanGaps: false,
      });
    }

    // Ensemble band + mean line
    datasets.push({
      label: 'Ensemble upper (mean + spread)', data: ensUpper,
      borderColor: 'rgba(251, 191, 36, 0.4)',
      backgroundColor: 'rgba(251, 191, 36, 0.15)',
      borderWidth: 1, borderDash: [2, 3],
      pointRadius: 0, fill: '+1', tension: 0.3, spanGaps: true,
    });
    datasets.push({
      label: 'Ensemble lower (mean - spread)', data: ensLower,
      borderColor: 'rgba(251, 191, 36, 0.4)',
      backgroundColor: 'transparent',
      borderWidth: 1, borderDash: [2, 3],
      pointRadius: 0, fill: false, tension: 0.3, spanGaps: true,
    });
    datasets.push({
      label: 'Ensemble mean (day 15-35)', data: ensMean,
      borderColor: '#fbbf24', backgroundColor: 'transparent',
      borderWidth: 2.5, pointRadius: 2, pointBackgroundColor: '#fbbf24',
      tension: 0.3, spanGaps: true,
    });

    chart.data.labels = labels;
    chart.data.datasets = datasets;
    chart.update();
  }

  // Init both charts
  const nxjChart = makeRegionalChart('chartRegionalNXJ');
  const sxjChart = makeRegionalChart('chartRegionalSXJ');

  let nxjMode = 'max';
  let sxjMode = 'max';

  // Wire toggles
  document.querySelectorAll('.nxj-toggle').forEach(btn => {
    btn.addEventListener('click', () => {
      document.querySelectorAll('.nxj-toggle').forEach(b => b.classList.remove('active'));
      btn.classList.add('active');
      nxjMode = btn.dataset.nxj;
      renderRegional(nxjChart, nxjData, nxjMode);
    });
  });
  document.querySelectorAll('.sxj-toggle').forEach(btn => {
    btn.addEventListener('click', () => {
      document.querySelectorAll('.sxj-toggle').forEach(b => b.classList.remove('active'));
      btn.classList.add('active');
      sxjMode = btn.dataset.sxj;
      renderRegional(sxjChart, sxjData, sxjMode);
    });
  });

  // Initial render
  renderRegional(nxjChart, nxjData, nxjMode);
  renderRegional(sxjChart, sxjData, sxjMode);
})();

// =============================================================================
//              INDIA CLIMATOLOGY CHARTS (5-year, top_tier only)
// =============================================================================
(function setupIndiaClimatology() {
  const indiaSites = DATA.india?.sites || [];
  const CURRENT_YEAR_INDIA = new Date().getFullYear();

  // Year color palette
  function climoYearColorIndia(year, isCurrent) {
    if (isCurrent) return '#fbbf24';
    const palette = ['#64748b','#94a3b8','#7dd3fc','#22d3ee','#a78bfa','#fb923c','#22c55e'];
    return palette[(year - 2020) % palette.length];
  }

  function makeIndiaClimoChart(canvasId, yLabel) {
    const ctx = document.getElementById(canvasId).getContext('2d');
    return new Chart(ctx, {
      type: 'line',
      data: { labels: [], datasets: [] },
      options: {
        responsive: true, maintainAspectRatio: false,
        interaction: { mode: 'index', intersect: false },
        plugins: {
          legend: { display: true, position: 'top',
                    labels: { color: '#a9b7cc', font: { family: 'JetBrains Mono', size: 10 },
                              usePointStyle: true, padding: 14 } },
          tooltip: { backgroundColor: '#0b1530' }
        },
        scales: {
          x: { ticks: { font: { family: 'JetBrains Mono', size: 9 }, color: '#a9b7cc',
                        maxRotation: 0, autoSkip: true, maxTicksLimit: 12 },
               grid: { color: '#1e293b', drawTicks: false },
               title: { display: true, text: 'Date (month-day)', color: '#a9b7cc',
                        font: { family: 'JetBrains Mono', size: 10 } } },
          y: { title: { display: true, text: yLabel, color: '#a9b7cc' },
               ticks: { color: '#a9b7cc' },
               grid: { color: '#1e293b' } }
        }
      }
    });
  }

  const chartClimoRainIndia = makeIndiaClimoChart('chartClimoRainIndia', 'Cumulative precipitation (mm)');
  const chartClimoSoilIndia = makeIndiaClimoChart('chartClimoSoilIndia', 'Soil moisture (m³/m³)');

  let _climoPeriodIndia = 'calendar';

  function doyToLabelIndia(doy) {
    const d = new Date(2025, 0, doy);
    const mm = String(d.getMonth() + 1).padStart(2, '0');
    const dd = String(d.getDate()).padStart(2, '0');
    return `${mm}-${dd}`;
  }

  function renderClimatologyIndia(siteName) {
    const s = indiaSites.find(x => x.county === siteName);
    const wrap = document.getElementById('climatologyWrapIndia');
    if (!s || !s.top_tier || !s.climatology || !Object.keys(s.climatology).length) {
      wrap.style.display = 'none';
      return;
    }
    wrap.style.display = 'block';

    // Kharif season: Jun 1 (DOY 152) to Nov 30 (DOY 334)
    const doyStart = _climoPeriodIndia === 'growing' ? 152 : 1;
    const doyEnd   = _climoPeriodIndia === 'growing' ? 334 : 366;
    const labels = [];
    for (let d = doyStart; d <= doyEnd; d += 1) labels.push(doyToLabelIndia(d));

    const years = Object.keys(s.climatology).map(Number).sort();
    const rainSets = [];
    const soilSets = [];

    years.forEach(year => {
      const days = s.climatology[year] || [];
      const isCurrent = (year === CURRENT_YEAR_INDIA);
      const color = climoYearColorIndia(year, isCurrent);

      const rainArr = new Array(labels.length).fill(null);
      const soilArr = new Array(labels.length).fill(null);

      if (_climoPeriodIndia === 'calendar') {
        days.forEach(d => {
          const idx = d.doy - doyStart;
          if (idx >= 0 && idx < labels.length) {
            rainArr[idx] = d.precip_cumulative_mm;
            if (d.soil != null) soilArr[idx] = d.soil;
          }
        });
      } else {
        // Kharif: re-cumulate from Jun 1
        let cum = 0;
        days.forEach(d => {
          if (d.doy < doyStart || d.doy > doyEnd) return;
          if (typeof d.precip_mm === 'number') cum += d.precip_mm;
          const idx = d.doy - doyStart;
          rainArr[idx] = Math.round(cum * 100) / 100;
          if (d.soil != null) soilArr[idx] = d.soil;
        });
      }

      const baseSet = {
        label: String(year),
        borderColor: color,
        backgroundColor: color + (isCurrent ? '33' : '11'),
        borderWidth: isCurrent ? 3.5 : 1.5,
        pointRadius: 0,
        pointHoverRadius: isCurrent ? 4 : 3,
        tension: 0.2,
        fill: false,
        spanGaps: true,
      };
      rainSets.push({...baseSet, data: rainArr});
      soilSets.push({...baseSet, data: soilArr});
    });

    chartClimoRainIndia.data.labels = labels;
    chartClimoRainIndia.data.datasets = rainSets;
    chartClimoRainIndia.update();

    chartClimoSoilIndia.data.labels = labels;
    chartClimoSoilIndia.data.datasets = soilSets;
    chartClimoSoilIndia.update();

    document.getElementById('climoSubIndia').textContent =
      _climoPeriodIndia === 'growing'
        ? 'Daily accumulated rainfall, kharif season (Jun-Nov), by year (mm)'
        : 'Daily accumulated rainfall, calendar year, by year (mm)';
  }

  // Wire toggle buttons
  document.querySelectorAll('.climo-period-india').forEach(btn => {
    btn.addEventListener('click', () => {
      document.querySelectorAll('.climo-period-india').forEach(b => b.classList.remove('active'));
      btn.classList.add('active');
      _climoPeriodIndia = btn.dataset.period;
      const currentSite = document.getElementById('select-india').value;
      renderClimatologyIndia(currentSite);
    });
  });

  // Hook into the India selector to refresh when selection changes
  const indiaSelect = document.getElementById('select-india');
  if (indiaSelect) {
    indiaSelect.addEventListener('change', () => renderClimatologyIndia(indiaSelect.value));
    renderClimatologyIndia(indiaSelect.value);
  }
})();

// =============================================================================
//              XINJIANG CLIMATE COMPARISON (NXJ / SXJ)
// =============================================================================
(function setupXinjiangClimateComparison() {
  const xjData = DATA.china?.xj_climate;
  if (!xjData) return;

  const CURRENT_YEAR_XJ = new Date().getFullYear();

  let currentRegion = 'NXJ';   // 'NXJ' or 'SXJ'

  // Update all region labels
  function updateRegionLabels(region) {
    ['xjHeatDaysRegion', 'xjGddRegion', 'xjSolarRegion'].forEach(id => {
      const el = document.getElementById(id);
      if (el) el.textContent = region;
    });
  }

  // ---- Derived metric calculators ----

  // Cumulative running count of days where Tmean >= 86°F, indexed by DOY
  function cumulativeHeatDays(records) {
    if (!records || !records.length) return [];
    let count = 0;
    return records.map(r => {
      if (r.tmean_f != null && r.tmean_f >= 86) count += 1;
      return { doy: r.doy, value: count };
    });
  }

  // 30-day rolling sum of GDD = max(0, Tmean_F - 50)
  function rollingGdd(records, window = 30) {
    if (!records || !records.length) return [];
    const gdd = records.map(r => (r.tmean_f != null ? Math.max(0, r.tmean_f - 50) : 0));
    const out = [];
    let sum = 0;
    for (let i = 0; i < gdd.length; i++) {
      sum += gdd[i];
      if (i >= window) sum -= gdd[i - window];
      out.push({ doy: records[i].doy, value: Math.round(sum * 10) / 10 });
    }
    return out;
  }

  // 30-day rolling sum of solar_mj
  function rollingSolar(records, window = 30) {
    if (!records || !records.length) return [];
    const solar = records.map(r => (r.solar_mj != null ? r.solar_mj : 0));
    const out = [];
    let sum = 0;
    for (let i = 0; i < solar.length; i++) {
      sum += solar[i];
      if (i >= window) sum -= solar[i - window];
      out.push({ doy: records[i].doy, value: Math.round(sum * 10) / 10 });
    }
    return out;
  }

  // Convert DOY to a MM-DD label
  function doyToMD(doy) {
    const d = new Date(2025, 0, doy);  // 2025 = non-leap reference
    const mm = String(d.getMonth() + 1).padStart(2, '0');
    const dd = String(d.getDate()).padStart(2, '0');
    return `${mm}-${dd}`;
  }

  // Compute climate normal (mean across years) for a given per-year metric
  function computeNormal(perYearData) {
    // perYearData: { year: [{doy, value}, ...] }
    // Result: array [{doy, value}] where value = mean over years for that doy
    const doyMap = {};   // doy -> [values across years]
    for (const year in perYearData) {
      perYearData[year].forEach(pt => {
        if (pt.value == null) return;
        if (!doyMap[pt.doy]) doyMap[pt.doy] = [];
        doyMap[pt.doy].push(pt.value);
      });
    }
    return Object.keys(doyMap).map(Number).sort((a, b) => a - b).map(doy => {
      const vals = doyMap[doy];
      const mean = vals.reduce((a, b) => a + b, 0) / vals.length;
      return { doy, value: Math.round(mean * 10) / 10 };
    });
  }

  // ---- Chart factory ----
  function makeXjChart(canvasId, yLabel) {
    const ctx = document.getElementById(canvasId).getContext('2d');
    return new Chart(ctx, {
      type: 'line',
      data: { labels: [], datasets: [] },
      options: {
        responsive: true, maintainAspectRatio: false,
        interaction: { mode: 'index', intersect: false },
        plugins: {
          legend: {
            display: true, position: 'top',
            labels: { color: '#a9b7cc', font: { family: 'JetBrains Mono', size: 10 },
                      usePointStyle: true, padding: 12, boxWidth: 8 }
          },
          tooltip: { backgroundColor: '#0b1530' }
        },
        scales: {
          x: {
            ticks: { color: '#a9b7cc', font: { family: 'JetBrains Mono', size: 9 },
                     maxRotation: 0, autoSkip: true, maxTicksLimit: 12 },
            grid: { color: '#1e293b' },
            title: { display: true, text: 'Date (month-day)', color: '#a9b7cc',
                     font: { family: 'JetBrains Mono', size: 10 } }
          },
          y: {
            ticks: { color: '#a9b7cc' },
            grid: { color: '#1e293b' },
            title: { display: true, text: yLabel, color: '#a9b7cc' }
          }
        }
      }
    });
  }

  // ---- Prepare chart instances ----
  const chartHeatDays = makeXjChart('chartXjHeatDays', 'Days (count)');
  const chartGdd      = makeXjChart('chartXjGdd',      '30-day GDD (°F·d)');
  const chartSolar    = makeXjChart('chartXjSolar',    '30-day radiation (MJ/m²)');

  // ---- Render logic ----
  function renderXjCharts(region) {
    const regionData = xjData[region];
    if (!regionData || !regionData.by_year) {
      console.warn('No XJ climate data for region', region);
      return;
    }

    const byYear = regionData.by_year;
    const years = Object.keys(byYear).map(Number).sort();
    if (!years.length) return;

    const currentYear = years[years.length - 1];   // most recent = current
    const lastYear    = currentYear - 1;
    const yearBeforeLast = currentYear - 2;

    // Compute each metric per year
    const heatDaysPerYear = {};
    const gddPerYear      = {};
    const solarPerYear    = {};
    years.forEach(y => {
      const records = byYear[y] || [];
      heatDaysPerYear[y] = cumulativeHeatDays(records);
      gddPerYear[y]      = rollingGdd(records, 30);
      solarPerYear[y]    = rollingSolar(records, 30);
    });

    // Climate normal = mean across all past years (exclude current if partial)
    const historicalYears = years.filter(y => y < currentYear);
    const normalHeat  = computeNormal(Object.fromEntries(historicalYears.map(y => [y, heatDaysPerYear[y]])));
    const normalGdd   = computeNormal(Object.fromEntries(historicalYears.map(y => [y, gddPerYear[y]])));
    const normalSolar = computeNormal(Object.fromEntries(historicalYears.map(y => [y, solarPerYear[y]])));

    // Build labels (DOY 1 to 365)
    const maxDoy = 366;
    const labels = [];
    for (let d = 1; d <= maxDoy; d++) labels.push(doyToMD(d));

    // Convert per-metric per-year data to fixed-length arrays aligned to DOY 1-366
    function toArr(perYear, y) {
      const data = perYear[y] || [];
      const arr = new Array(maxDoy).fill(null);
      data.forEach(pt => { if (pt.doy >= 1 && pt.doy <= maxDoy) arr[pt.doy - 1] = pt.value; });
      return arr;
    }
    function toArrNormal(normalData) {
      const arr = new Array(maxDoy).fill(null);
      normalData.forEach(pt => { if (pt.doy >= 1 && pt.doy <= maxDoy) arr[pt.doy - 1] = pt.value; });
      return arr;
    }

    // Common dataset styling
    const styles = {
      normal:  { color: '#94a3b8', dash: [6, 4], width: 2,   points: 0 },
      y2ago:   { color: '#fb923c', dash: [],     width: 1.8, points: 0 },
      yLast:   { color: '#3b82f6', dash: [],     width: 2,   points: 0 },
      yCur:    { color: '#ef4444', dash: [],     width: 3,   points: 2.5 },
    };

    function buildDatasets(perYear, normalArr) {
      const sets = [];
      sets.push({
        label: 'Climate normal (' + historicalYears[0] + '-' + historicalYears[historicalYears.length - 1] + ')',
        data: normalArr,
        borderColor: styles.normal.color,
        backgroundColor: 'transparent',
        borderDash: styles.normal.dash,
        borderWidth: styles.normal.width,
        pointRadius: styles.normal.points,
        tension: 0.2, spanGaps: true,
      });
      if (perYear[yearBeforeLast]) {
        sets.push({
          label: 'Year before last (' + yearBeforeLast + ')',
          data: toArr(perYear, yearBeforeLast),
          borderColor: styles.y2ago.color,
          backgroundColor: 'transparent',
          borderWidth: styles.y2ago.width,
          pointRadius: styles.y2ago.points,
          tension: 0.2, spanGaps: true,
        });
      }
      if (perYear[lastYear]) {
        sets.push({
          label: 'Last year (' + lastYear + ')',
          data: toArr(perYear, lastYear),
          borderColor: styles.yLast.color,
          backgroundColor: 'transparent',
          borderWidth: styles.yLast.width,
          pointRadius: styles.yLast.points,
          tension: 0.2, spanGaps: true,
        });
      }
      if (perYear[currentYear]) {
        sets.push({
          label: 'Current year (' + currentYear + ')',
          data: toArr(perYear, currentYear),
          borderColor: styles.yCur.color,
          backgroundColor: styles.yCur.color + '22',
          borderWidth: styles.yCur.width,
          pointRadius: styles.yCur.points,
          pointBackgroundColor: styles.yCur.color,
          tension: 0.2, spanGaps: true,
        });
      }
      return sets;
    }

    chartHeatDays.data.labels = labels;
    chartHeatDays.data.datasets = buildDatasets(heatDaysPerYear, toArrNormal(normalHeat));
    chartHeatDays.update();

    chartGdd.data.labels = labels;
    chartGdd.data.datasets = buildDatasets(gddPerYear, toArrNormal(normalGdd));
    chartGdd.update();

    chartSolar.data.labels = labels;
    chartSolar.data.datasets = buildDatasets(solarPerYear, toArrNormal(normalSolar));
    chartSolar.update();

    updateRegionLabels(region);
  }

  // ---- Wire region toggle ----
  document.querySelectorAll('.xj-region-toggle').forEach(btn => {
    btn.addEventListener('click', () => {
      document.querySelectorAll('.xj-region-toggle').forEach(b => b.classList.remove('active'));
      btn.classList.add('active');
      currentRegion = btn.dataset.xj;
      renderXjCharts(currentRegion);
    });
  });

  // ---- Initial render ----
  renderXjCharts('NXJ');
})();





// Default to the most prominent Brazilian cotton municipality
const defaultBR = ['Sapezal','São Desidério','Sorriso','Luís Eduardo Magalhães','Lucas do Rio Verde']
  .find(name => DATA.brazil.municipalities.some(c=>c.county===name)) || DATA.brazil.municipalities[0]?.county;
if(defaultBR){ brazilSelect.value = defaultBR; renderBrazilDetail(defaultBR); }

</script>
</body>
</html>
"""


In [32]:
def _severity_class(sev):
    return f"sev-{(sev or 'unknown').lower()}"


def _alert_card(a):
    sev_cls = _severity_class(a.get("severity"))
    instr_html = (f"<pre><strong>Instruction:</strong> {esc(a.get('instruction',''))}</pre>"
                  if a.get("instruction") else "")
    return (
        f"<article class='alert-card {sev_cls}'>"
        f"<div class='alert-head'>"
        f"<span class='alert-event'>{esc(a.get('event','Alert'))}</span>"
        f"<span class='alert-sev'>{esc((a.get('severity') or 'Unknown').upper())}</span>"
        f"</div>"
        f"<p class='alert-headline'>{esc(a.get('headline',''))}</p>"
        f"<p class='alert-area'><strong>Area:</strong> {esc(a.get('area',''))}</p>"
        f"<p class='alert-meta'>"
        f"<span>Urgency: {esc(a.get('urgency','—'))}</span> · "
        f"<span>Certainty: {esc(a.get('certainty','—'))}</span> · "
        f"<span>Expires: {esc((a.get('expires') or '')[:16].replace('T',' '))}</span>"
        f"</p>"
        f"<details><summary>Full description &amp; instructions</summary>"
        f"<pre>{esc(a.get('description',''))}</pre>"
        f"{instr_html}"
        f"<p class='alert-sender'>— {esc(a.get('sender',''))}</p>"
        f"</details></article>"
    )


def render_html(data):
    payload = json.dumps(data, ensure_ascii=False, default=str)
    generated = datetime.now(timezone.utc).strftime("%B %d, %Y · %H:%M UTC")

    # ============== NOAA TAB ==============
    noaa = data["noaa"]
    currents = [c.get("current") for c in noaa["texas"] if c.get("current")]
    temps    = [c.get("temp_f")  for c in currents if isinstance(c.get("temp_f"), (int, float))]
    avg_temp = sum(temps) / len(temps) if temps else 0

    def _k_temp(c):
        v = (c.get("current") or {}).get("temp_f"); return v if isinstance(v,(int,float)) else -999
    def _k_temp_min(c):
        v = (c.get("current") or {}).get("temp_f"); return v if isinstance(v,(int,float)) else 999
    def _k_wind(c):
        v = (c.get("current") or {}).get("wind_mph"); return v if isinstance(v,(int,float)) else -1
    def _k_wet(c):
        v = (c.get("open_meteo") or {}).get("forecast_7d_precip_mm"); return v if isinstance(v,(int,float)) else -1

    cities = noaa["texas"]
    hottest  = max(cities, key=_k_temp,    default={})
    coldest  = min(cities, key=_k_temp_min, default={})
    windiest = max(cities, key=_k_wind,    default={})
    wettest  = max(cities, key=_k_wet,     default={})

    severe = sum(1 for a in noaa["alerts_tx"]
                 if (a.get("severity") or "").lower() in ("severe","extreme"))

    def metric(city, key, suffix, decimals=0):
        return (f"<span class='metric-value'>"
                f"{fmt_num((city.get('current') or {}).get(key), suffix, decimals)}"
                f"</span><span class='metric-context'>{esc(city.get('name','—'))}</span>")

    wettest_mm = (wettest.get("open_meteo") or {}).get("forecast_7d_precip_mm")

    tx_rows = []
    for c in noaa["texas"]:
        cur  = c.get("current")    or {}
        om   = c.get("open_meteo") or {}
        tx_rows.append(
            f"<tr>"
            f"<td class='city-name'>{esc(c.get('name'))}"
            f"<span class='region-label'>{esc(c.get('region',''))}</span></td>"
            f"<td>{esc(cur.get('text_description') or '—')}</td>"
            f"<td class='num strong'>{fmt_temp(cur.get('temp_f'))}</td>"
            f"<td class='num'>{fmt_temp(c.get('forecast_min_f'))}</td>"
            f"<td class='num'>{fmt_temp(c.get('forecast_max_f'))}</td>"
            f"<td class='num'>{fmt_num(cur.get('humidity'), '%', 0)}</td>"
            f"<td class='num strong'>{fmt_num(cur.get('wind_mph'), ' mph', 1)}</td>"
            f"<td class='num'>{fmt_num(cur.get('precip_in_1h'), ' in', 2)}</td>"
            f"<td class='num'>{fmt_num(om.get('forecast_7d_precip_mm'), ' mm', 1)}</td>"
            f"</tr>"
        )
    alerts_noaa = [_alert_card(a) for a in noaa["alerts_tx"][:30]]

    # ============== COTTON TAB ==============
    cotton = data["cotton"]
    counties = cotton["counties"]

    def _c_past3(c):
        v = c.get("past3_precip_mm"); return v if isinstance(v,(int,float)) else -1
    def _c_past3_min(c):
        v = c.get("past3_precip_mm"); return v if isinstance(v,(int,float)) else 9999
    def _c_fc24(c):
        v = (c.get("forecast_24h") or {}).get("total_mm"); return v if isinstance(v,(int,float)) else -1

    wettest_c   = max(counties, key=_c_past3,    default={})
    driest_c    = min(counties, key=_c_past3_min, default={})
    most_fc24_c = max(counties, key=_c_fc24,     default={})

    balances = [c.get("past3_balance_mm") for c in counties
                if isinstance(c.get("past3_balance_mm"), (int, float))]
    avg_balance = sum(balances) / len(balances) if balances else 0
    dry_count = sum(1 for c in counties
                    if isinstance(c.get("past3_balance_mm"), (int, float))
                    and c["past3_balance_mm"] < -10)
    wet_count = sum(1 for c in counties
                    if isinstance(c.get("past3_precip_mm"), (int, float))
                    and c["past3_precip_mm"] > 20)

    def c_metric(c, value_key, label_template, suffix, decimals=0):
        d = c
        for k in value_key.split("."):
            d = (d or {}).get(k) if isinstance(d, dict) else None
        return (f"<span class='metric-value'>{fmt_num(d, suffix, decimals)}</span>"
                f"<span class='metric-context'>{esc(label_template.format(**c))}</span>")

    c_wettest_html = c_metric(wettest_c, "past3_precip_mm", "{county} County · {seat}", " mm", 0)
    c_driest_html  = c_metric(driest_c,  "past3_precip_mm", "{county} County · {seat}", " mm", 0)
    c_fc24_html    = c_metric(most_fc24_c, "forecast_24h.total_mm", "{county} County · {seat}", " mm", 0)
    bal_cls = "negative" if avg_balance < 0 else "positive"
    c_balance_html = (f"<span class='metric-value {bal_cls}'>{avg_balance:+.0f}mm</span>"
                      f"<span class='metric-context'>across {len(balances)} counties</span>")

    # Cotton table rows
    cotton_rows = []
    for c in counties:
        cur = c.get("current") or {}
        fc  = c.get("forecast_24h") or {}
        daily = c.get("daily") or []
        past  = [d for d in daily if d.get("period") == "past"]
        # Get J-3, J-2, J-1 daily precip (the 3 most recent past days)
        # past_days=7 means past[] has up to 7 entries chronologically; we want
        # the 3 most recent ones, not the 3 oldest.
        past_last_3 = past[-3:] if len(past) >= 3 else past
        p3 = past_last_3[0].get("precip_mm") if len(past_last_3) > 0 else None
        p2 = past_last_3[1].get("precip_mm") if len(past_last_3) > 1 else None
        p1 = past_last_3[2].get("precip_mm") if len(past_last_3) > 2 else None
        bal = c.get("past3_balance_mm")
        bal_cls = "balance-cell " + ("negative" if isinstance(bal,(int,float)) and bal<0 else "positive")
        top12_cls = "top12" if c.get("top12") else ""
        star = "<span class='top12-star'>⭐</span>" if c.get("top12") else ""
        cotton_rows.append(
            f"<tr class='{top12_cls}' data-region='{esc(c.get('region',''))}'>"
            f"<td class='city-name'>{star}{esc(c['county'])}"
            f"<span class='region-tag {esc(c.get('region',''))}'>{esc(c.get('region',''))}</span>"
            f"<span class='region-label'>{esc(c['seat'])}</span></td>"
            f"<td class='num strong'>{fmt_num(cur.get('temp_f'), '', 0)}</td>"
            f"<td class='num'>{fmt_num(cur.get('humidity'), '', 0)}</td>"
            f"<td class='num'>{fmt_num(p3, '', 1)}</td>"
            f"<td class='num'>{fmt_num(p2, '', 1)}</td>"
            f"<td class='num'>{fmt_num(p1, '', 1)}</td>"
            f"<td class='num strong'>{fmt_num(c.get('past3_precip_mm'), '', 1)}</td>"
            f"<td class='num strong'>{fmt_num(c.get('past7_precip_mm'), '', 1)}</td>"
            f"<td class='num'>{fmt_num(c.get('past3_et0_mm'), '', 1)}</td>"
            f"<td class='num {bal_cls}'>{fmt_num(bal, '', 0)}</td>"
            f"<td class='num'>{fmt_num(cur.get('soil_root_zone'), '', 2)}</td>"
            f"<td class='num'>{fmt_num(fc.get('max_prob'), '%', 0)}</td>"
            f"<td class='num strong'>{fmt_num(fc.get('total_mm'), '', 1)}</td>"
            f"<td class='num strong'>{fmt_num(c.get('forecast_7d_precip_mm'), '', 1)}</td>"
            f"</tr>"
        )
    alerts_cotton = [_alert_card(a) for a in cotton["alerts_tx"][:30]]


    # ============== BRAZIL TAB ==============
    brazil = data.get("brazil", {})
    muns = brazil.get("municipalities", [])

    def _b_past7(c):
        v = c.get("past7_precip_mm"); return v if isinstance(v,(int,float)) else -1
    def _b_past7_min(c):
        v = c.get("past7_precip_mm"); return v if isinstance(v,(int,float)) else 9999
    def _b_fc7d(c):
        v = c.get("forecast_7d_precip_mm"); return v if isinstance(v,(int,float)) else -1

    br_wettest = max(muns, key=_b_past7, default={})
    br_driest  = min(muns, key=_b_past7_min, default={})
    br_most_fc = max(muns, key=_b_fc7d, default={})

    br_balances = [c.get("past3_balance_mm") for c in muns
                   if isinstance(c.get("past3_balance_mm"), (int, float))]
    br_avg_balance = sum(br_balances)/len(br_balances) if br_balances else 0

    def br_metric(c, value_key, label_template, suffix, decimals=0):
        d = c
        for k in value_key.split("."):
            d = (d or {}).get(k) if isinstance(d, dict) else None
        ctx_text = label_template.format(**c) if c else "—"
        return (f"<span class='metric-value'>{fmt_num(d, suffix, decimals)}</span>"
                f"<span class='metric-context'>{esc(ctx_text)}</span>")

    br_wettest_html  = br_metric(br_wettest, "past7_precip_mm",       "{county} · {state}", " mm", 0)
    br_driest_html   = br_metric(br_driest,  "past7_precip_mm",       "{county} · {state}", " mm", 0)
    br_fc_html       = br_metric(br_most_fc, "forecast_7d_precip_mm", "{county} · {state}", " mm", 0)
    br_bal_cls = "negative" if br_avg_balance < 0 else "positive"
    br_balance_html  = (f"<span class='metric-value {br_bal_cls}'>{br_avg_balance:+.0f}mm</span>"
                        f"<span class='metric-context'>across {len(br_balances)} municipalities</span>")

    br_top_count   = sum(1 for c in muns if c.get("top_tier"))
    br_state_count = len({c.get("state") for c in muns if c.get("state")})

    br_rows = []
    for c in muns:
        cur = c.get("current") or {}
        fc  = c.get("forecast_24h") or {}
        daily = c.get("daily") or []
        past = [d for d in daily if d.get("period") == "past"]
        last_3 = past[-3:] if len(past) >= 3 else past
        p3 = last_3[0].get("precip_mm") if len(last_3) > 0 else None
        p2 = last_3[1].get("precip_mm") if len(last_3) > 1 else None
        p1 = last_3[2].get("precip_mm") if len(last_3) > 2 else None
        bal = c.get("past3_balance_mm")
        bal_cls = "balance-cell " + ("negative" if isinstance(bal,(int,float)) and bal<0 else "positive")
        top_cls = "top12" if c.get("top_tier") else ""
        star = "<span class='top12-star'>⭐</span>" if c.get("top_tier") else ""
        br_rows.append(
            f"<tr class='{top_cls}' data-state='{esc(c.get('state',''))}' data-region='{esc(c.get('region',''))}'>"
            f"<td class='city-name'>{star}{esc(c['county'])}"
            f"<span class='region-tag {esc(c.get('region',''))}'>{esc(c.get('region',''))}</span>"
            f"<span class='region-label'>{esc(c.get('state',''))}</span></td>"
            f"<td class='num strong'>{fmt_num(cur.get('temp_f'), '', 0)}</td>"
            f"<td class='num'>{fmt_num(cur.get('humidity'), '', 0)}</td>"
            f"<td class='num'>{fmt_num(p3, '', 1)}</td>"
            f"<td class='num'>{fmt_num(p2, '', 1)}</td>"
            f"<td class='num'>{fmt_num(p1, '', 1)}</td>"
            f"<td class='num strong'>{fmt_num(c.get('past3_precip_mm'), '', 1)}</td>"
            f"<td class='num strong'>{fmt_num(c.get('past7_precip_mm'), '', 1)}</td>"
            f"<td class='num'>{fmt_num(c.get('past3_et0_mm'), '', 1)}</td>"
            f"<td class='num {bal_cls}'>{fmt_num(bal, '', 0)}</td>"
            f"<td class='num'>{fmt_num(cur.get('soil_root_zone'), '', 2)}</td>"
            f"<td class='num'>{fmt_num(fc.get('max_prob'), '%', 0)}</td>"
            f"<td class='num strong'>{fmt_num(fc.get('total_mm'), '', 1)}</td>"
            f"<td class='num strong'>{fmt_num(c.get('forecast_7d_precip_mm'), '', 1)}</td>"
            f"</tr>"
        )


    # ============== GENERIC COUNTRY RENDERING (CN, IN, AU, TR) ==============
    def render_country(data_key, country_code, context_html=""):
        country_data = data.get(data_key, {})
        sites = country_data.get("sites", [])

        def _w7(c):
            v = c.get("past7_precip_mm"); return v if isinstance(v,(int,float)) else -1
        def _w7_min(c):
            v = c.get("past7_precip_mm"); return v if isinstance(v,(int,float)) else 9999
        def _fc7(c):
            v = c.get("forecast_7d_precip_mm"); return v if isinstance(v,(int,float)) else -1

        wettest = max(sites, key=_w7, default={})
        driest  = min(sites, key=_w7_min, default={})
        most_fc = max(sites, key=_fc7, default={})

        balances = [c.get("past3_balance_mm") for c in sites
                    if isinstance(c.get("past3_balance_mm"), (int, float))]
        avg_bal = sum(balances)/len(balances) if balances else 0

        def metric_html(c, key, suffix, decimals=0):
            d = c
            for k in key.split("."):
                d = (d or {}).get(k) if isinstance(d, dict) else None
            ctx = f"{c.get('county','—')} · {c.get('state','—')}" if c else "—"
            return (f"<span class='metric-value'>{fmt_num(d, suffix, decimals)}</span>"
                    f"<span class='metric-context'>{esc(ctx)}</span>")

        wettest_html = metric_html(wettest, "past7_precip_mm",       " mm", 0)
        driest_html  = metric_html(driest,  "past7_precip_mm",       " mm", 0)
        fc_html      = metric_html(most_fc, "forecast_7d_precip_mm", " mm", 0)
        bal_cls = "negative" if avg_bal < 0 else "positive"
        balance_html = (f"<span class='metric-value {bal_cls}'>{avg_bal:+.0f}mm</span>"
                        f"<span class='metric-context'>across {len(balances)} sites</span>")
        top_count = sum(1 for c in sites if c.get("top_tier"))
        state_count = len({c.get("state") for c in sites if c.get("state")})

        rows = []
        for c in sites:
            cur = c.get("current") or {}
            fc  = c.get("forecast_24h") or {}
            daily = c.get("daily") or []
            past = [d for d in daily if d.get("period") == "past"]
            last3 = past[-3:] if len(past) >= 3 else past
            p3 = last3[0].get("precip_mm") if len(last3) > 0 else None
            p2 = last3[1].get("precip_mm") if len(last3) > 1 else None
            p1 = last3[2].get("precip_mm") if len(last3) > 2 else None
            bal = c.get("past3_balance_mm")
            bal_cls = "balance-cell " + ("negative" if isinstance(bal,(int,float)) and bal<0 else "positive")
            top_cls = "top12" if c.get("top_tier") else ""
            star = "<span class='top12-star'>⭐</span>" if c.get("top_tier") else ""
            rows.append(
                f"<tr class='{top_cls}' data-state='{esc(c.get('state',''))}' data-region='{esc(c.get('region',''))}'>"
                f"<td class='city-name'>{star}{esc(c['county'])}"
                f"<span class='region-tag {esc(c.get('region',''))}'>{esc(c.get('region',''))}</span>"
                f"<span class='region-label'>{esc(c.get('state',''))}</span></td>"
                f"<td class='num strong'>{fmt_num(cur.get('temp_f'), '', 0)}</td>"
                f"<td class='num'>{fmt_num(cur.get('humidity'), '', 0)}</td>"
                f"<td class='num'>{fmt_num(p3, '', 1)}</td>"
                f"<td class='num'>{fmt_num(p2, '', 1)}</td>"
                f"<td class='num'>{fmt_num(p1, '', 1)}</td>"
                f"<td class='num strong'>{fmt_num(c.get('past3_precip_mm'), '', 1)}</td>"
                f"<td class='num strong'>{fmt_num(c.get('past7_precip_mm'), '', 1)}</td>"
                f"<td class='num'>{fmt_num(c.get('past3_et0_mm'), '', 1)}</td>"
                f"<td class='num {bal_cls}'>{fmt_num(bal, '', 0)}</td>"
                f"<td class='num'>{fmt_num(cur.get('soil_root_zone'), '', 2)}</td>"
                f"<td class='num'>{fmt_num(fc.get('max_prob'), '%', 0)}</td>"
                f"<td class='num strong'>{fmt_num(fc.get('total_mm'), '', 1)}</td>"
                f"<td class='num strong'>{fmt_num(c.get('forecast_7d_precip_mm'), '', 1)}</td>"
                f"</tr>"
            )
        return {
            "N":         str(len(sites)),
            "WETTEST":   wettest_html,
            "DRIEST":    driest_html,
            "FORECAST":  fc_html,
            "BALANCE":   balance_html,
            "TOP_COUNT": str(top_count),
            "STATE_COUNT":str(state_count),
            "TABLE_ROWS":"".join(rows),
            "CONTEXT":   context_html,
        }

    CN_CONTEXT = """<p style='font-size:13px;color:var(--ink-soft);margin:0 0 12px'><strong style='color:var(--gold)'>Xinjiang</strong> produces ~92% of China's cotton. Planting begins April-May; harvest October. The northern (NXJ) and southern (SXJ) Xinjiang regions are separated by the Tianshan Mountains. NXJ has shorter frost-free periods (140-180 days); SXJ has longer growing seasons and higher quality long-staple cotton.</p>
    <p style='font-size:13px;color:var(--ink-soft);margin:0'>The <strong style='color:var(--gold)'>Aksu prefecture</strong> in SXJ is the single largest cotton-producing region in China. Key risk: spring cold damage (SpCD) during emergence (May).</p>"""

    IN_CONTEXT = """<p style='font-size:13px;color:var(--ink-soft);margin:0 0 10px'><strong style='color:var(--gold)'>Kharif crop</strong> sown with southwest monsoon arrival (June-July), harvested October-December.</p>
    <p style='font-size:13px;color:var(--ink-soft);margin:0 0 10px'>Top states: <strong>Gujarat</strong> (~28%, Saurashtra rainfed black cotton soil), <strong>Maharashtra</strong> (~25%, Vidarbha + Marathwada), <strong>Telangana</strong> (~15%).</p>
    <p style='font-size:13px;color:var(--ink-soft);margin:0'>Northern states (Punjab, Haryana, Rajasthan) are irrigated, less dependent on monsoon timing.</p>"""

    AU_CONTEXT = """<p style='font-size:13px;color:var(--ink-soft);margin:0 0 10px'><strong style='color:var(--gold)'>Murray-Darling Basin = 91%</strong> of Australian cotton. Planting September-October, harvest March-April.</p>
    <p style='font-size:13px;color:var(--ink-soft);margin:0 0 10px'>NSW (~66%): Gwydir, Namoi, Macquarie, Murrumbidgee valleys.</p>
    <p style='font-size:13px;color:var(--ink-soft);margin:0'>QLD (~33%): Darling Downs, St George, Dirranbandi, Central Highlands.</p>"""

    TR_CONTEXT = """<p style='font-size:13px;color:var(--ink-soft);margin:0 0 10px'><strong style='color:var(--gold)'>GAP / Southeast Anatolia</strong> ~60% (Şanlıurfa alone 42%). Irrigated via Atatürk Dam network. Planting April, harvest September-October.</p>
    <p style='font-size:13px;color:var(--ink-soft);margin:0 0 10px'><strong style='color:var(--gold)'>Cukurova / Mediterranean</strong> ~20% (Adana, Hatay). Long staple, declining due to citrus competition.</p>
    <p style='font-size:13px;color:var(--ink-soft);margin:0'><strong style='color:var(--gold)'>Aegean / West</strong> ~20% (Aydın, İzmir). Premium long-staple cotton, highest quality.</p>"""

    cn = render_country("china",     "CN", CN_CONTEXT)
    inn = render_country("india",     "IN", IN_CONTEXT)
    au = render_country("australia", "AU", AU_CONTEXT)
    tr = render_country("turkey",    "TR", TR_CONTEXT)

    return (HTML_TEMPLATE
        .replace("__PAYLOAD__",         payload)
        .replace("__GENERATED__",       generated)
        .replace("__N_COTTON__",        str(len(counties)))
        .replace("__N_TX_STATIONS__",   str(len(currents)))
        .replace("__HOTTEST__",         metric(hottest,  'temp_f', '°F'))
        .replace("__COLDEST__",         metric(coldest,  'temp_f', '°F'))
        .replace("__WINDIEST__",        metric(windiest, 'wind_mph', ' mph', 1))
        .replace("__AVGTEMP__",         f"{avg_temp:.0f}")
        .replace("__N_TEMPS__",         str(len(temps)))
        .replace("__WETTEST_MM__",      fmt_num(wettest_mm, ' mm', 1))
        .replace("__WETTEST_NAME__",    esc(wettest.get('name','—')))
        .replace("__SEVERE__",          str(severe))
        .replace("__TOTAL_ALERTS__",    str(len(noaa['alerts_tx'])))
        .replace("__TX_TABLE_ROWS__",   "".join(tx_rows))
        .replace("__ALERTS_NOAA__",     "".join(alerts_noaa)
                  or "<p class='no-alerts'>No active alerts for Texas at this time.</p>")
        .replace("__C_WETTEST__",       c_wettest_html)
        .replace("__C_DRIEST__",        c_driest_html)
        .replace("__C_FORECAST_RAIN__", c_fc24_html)
        .replace("__C_AVG_BALANCE__",   c_balance_html)
        .replace("__C_DRY_COUNT__",     str(dry_count))
        .replace("__C_WET_COUNT__",     str(wet_count))
        .replace("__COTTON_TABLE_ROWS__","".join(cotton_rows))
        .replace("__ALERTS_COTTON__",   "".join(alerts_cotton)
                  or "<p class='no-alerts'>No active alerts for Texas at this time.</p>")
        .replace("__N_BR__",              str(len(muns)))
        .replace("__BR_WETTEST__",        br_wettest_html)
        .replace("__BR_DRIEST__",         br_driest_html)
        .replace("__BR_FORECAST_RAIN__",  br_fc_html)
        .replace("__BR_AVG_BALANCE__",    br_balance_html)
        .replace("__BR_TOP_COUNT__",      str(br_top_count))
        .replace("__BR_STATE_COUNT__",    str(br_state_count))
        .replace("__BR_TABLE_ROWS__",     "".join(br_rows))
        .replace("__N_CN__",                  cn["N"])
        .replace("__CHINA_WETTEST__",         cn["WETTEST"])
        .replace("__CHINA_DRIEST__",          cn["DRIEST"])
        .replace("__CHINA_FORECAST__",        cn["FORECAST"])
        .replace("__CHINA_BALANCE__",         cn["BALANCE"])
        .replace("__CHINA_TOP_COUNT__",       cn["TOP_COUNT"])
        .replace("__CHINA_STATE_COUNT__",     cn["STATE_COUNT"])
        .replace("__CHINA_TABLE_ROWS__",      cn["TABLE_ROWS"])
        .replace("__CHINA_CONTEXT__",         cn["CONTEXT"])
        .replace("__N_IN__",                  inn["N"])
        .replace("__INDIA_WETTEST__",         inn["WETTEST"])
        .replace("__INDIA_DRIEST__",          inn["DRIEST"])
        .replace("__INDIA_FORECAST__",        inn["FORECAST"])
        .replace("__INDIA_BALANCE__",         inn["BALANCE"])
        .replace("__INDIA_TOP_COUNT__",       inn["TOP_COUNT"])
        .replace("__INDIA_STATE_COUNT__",     inn["STATE_COUNT"])
        .replace("__INDIA_TABLE_ROWS__",      inn["TABLE_ROWS"])
        .replace("__INDIA_CONTEXT__",         inn["CONTEXT"])
        .replace("__N_AU__",                  au["N"])
        .replace("__AUSTRALIA_WETTEST__",     au["WETTEST"])
        .replace("__AUSTRALIA_DRIEST__",      au["DRIEST"])
        .replace("__AUSTRALIA_FORECAST__",    au["FORECAST"])
        .replace("__AUSTRALIA_BALANCE__",     au["BALANCE"])
        .replace("__AUSTRALIA_TOP_COUNT__",   au["TOP_COUNT"])
        .replace("__AUSTRALIA_STATE_COUNT__", au["STATE_COUNT"])
        .replace("__AUSTRALIA_TABLE_ROWS__",  au["TABLE_ROWS"])
        .replace("__AUSTRALIA_CONTEXT__",     au["CONTEXT"])
        .replace("__N_TR__",                  tr["N"])
        .replace("__TURKEY_WETTEST__",        tr["WETTEST"])
        .replace("__TURKEY_DRIEST__",         tr["DRIEST"])
        .replace("__TURKEY_FORECAST__",       tr["FORECAST"])
        .replace("__TURKEY_BALANCE__",        tr["BALANCE"])
        .replace("__TURKEY_TOP_COUNT__",      tr["TOP_COUNT"])
        .replace("__TURKEY_STATE_COUNT__",    tr["STATE_COUNT"])
        .replace("__TURKEY_TABLE_ROWS__",     tr["TABLE_ROWS"])
        .replace("__TURKEY_CONTEXT__",        tr["CONTEXT"])
    )


In [33]:
def main():
    print("=" * 60)
    print("Global Cotton Weather Dashboard Generator v11")
    print("=" * 60)
    print("\nRendering HTML dashboard...")
    OUTPUT_FILE.write_text(render_html(DATA), encoding="utf-8")
    OUTPUT_FILE.with_suffix(".json").write_text(
        json.dumps(DATA, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )
    print(f"  Wrote {OUTPUT_FILE.resolve()} ({OUTPUT_FILE.stat().st_size/1024:.1f} KB)")
    return 0

main()


Global Cotton Weather Dashboard Generator v11

Rendering HTML dashboard...
  Wrote C:\Users\AMAR8\Work Folders\Downloads\global_cotton_weather.html (10592.8 KB)


0